# 4\.2\.3 Construct Weather Context and Final Forecasting Matrices

Notebooks 4.2.1 and 4.2.2 established the two origin-side forecasting layers used by this project. The first describes each forecast target using its own leakage-safe history, seasonal structure, and deterministic calendar information. The second adds compact multimodal, transportation-connected, and static geographic context available at the same forecast origin.

This notebook completes the forecasting feature handoff.

The remaining contextual family is **weather**, whose availability rules differ from mobility data. Weather already observed through the forecast origin is ordinary leakage-safe historical information. Weather associated with a future target period is different: the project does not have a historical archive of the short-range weather forecasts that would actually have been available at every forecast origin. For historical backtesting, realized target-period weather may therefore be used only as an explicitly documented **proxy for operational forecast weather**. This can make model evaluation somewhat optimistic and must remain visible in the feature contract.

Weather also behaves differently in the later no-congestion-pricing simulation. Actual post-CP mobility may never enter the hypothetical no-CP transportation state, but realized post-CP weather may remain fixed as external context because weather is exogenous to congestion pricing. The counterfactual question is therefore conditioned on the weather that actually occurred without allowing actual post-CP transportation outcomes into the prediction loop.

The notebook has four jobs:

1. inspect the authoritative Taxi Zone weather handoff and select a restrained forecasting weather family;
2. distinguish and align origin-observed weather from horizon-specific target-period weather proxies;
3. combine the completed 4.2.1 and 4.2.2 layers with weather, target timing, and the actual future mobility value as a **label rather than a predictor**; and
4. validate the complete cross-family forecasting contract before modeling begins in 4.2.4.

The final ordinary forecasting grain will be:

**Modeling Track × Forecast Sequence × Origin observation_sequence_id × Horizon**

The frozen Pre-CP Reference requires a separate runtime-aware handoff after January 4, 2025 because recursive mobility-derived features do not exist until the no-CP forecast path is actually generated. This notebook will not populate those fields with observed post-CP mobility merely to create a complete rectangular table.

No model fitting, global scaling, global statistical imputation, or holdout-informed feature selection is performed here.

## Notebook Setup

Define the shared forecasting contract, authoritative upstream paths, output locations, rebuild controls, weather candidates, analytical keys, and helper functions used throughout 4.2.3.

The completed 4.2.1 and 4.2.2 artifacts are treated as fixed upstream contracts. This notebook does not reconstruct their feature logic. Large Parquet inputs are inspected through metadata where possible, and only the columns needed for a specific transformation are materialized.

The eight weather measures carried forward from harmonization are treated as **candidates** at this point. No weather predictor is selected merely because it exists upstream; the first section will measure its real forecasting-time coverage, temporal gaps, spatial differentiation, and redundancy before the retained family is locked.

In [1]:
# 4.2.3.0A — Define imports, globals, paths, toggles, and forecasting contracts

from pathlib import Path
import shutil
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import plotly.graph_objects as go

from IPython.display import display
from project_branding import BRAND_COLORS, BRAND_COLOR_SEQUENCE


# ---------------------------------------------------------------------
# Display and chart defaults
# ---------------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

CHART_WIDTH = 920
CHART_HEIGHT = 520


# ---------------------------------------------------------------------
# Notebook behavior
#
# No expensive final-matrix build occurs before the first HITL stop, but
# downstream rebuild controls are declared here so the notebook keeps one
# consistent configuration surface.
# ---------------------------------------------------------------------

WRITE_INTERMEDIATES = True
WRITE_FINAL_OUTPUTS = True
RUN_VISUALS = True

REBUILD_WEATHER_ALIGNMENT = False
REBUILD_FINAL_MATRICES = False
REBUILD_PRE_CP_REFIT_SUPPLEMENT = False
REBUILD_COUNTERFACTUAL_TEMPLATE = False


# ---------------------------------------------------------------------
# Canonical forecasting contract
# ---------------------------------------------------------------------

STUDY_START_DATE = pd.Timestamp("2023-01-01")
STUDY_END_DATE = pd.Timestamp("2026-03-31")

CONGESTION_PRICING_START_DATE = pd.Timestamp("2025-01-05")
PRE_CP_FREEZE_DATE = pd.Timestamp("2025-01-04")

PRE_CP_FREEZE_OBSERVATION_SEQUENCE_ID = 3675
CP_FIRST_OBSERVATION_SEQUENCE_ID = 3676

FORECAST_TARGETS = [
    "taxi_trip_count",
    "taxi_avg_trip_speed",
    "fhvhv_trip_count",
    "fhvhv_avg_trip_speed",
    "subway_ridership",
]

FORECAST_HORIZONS = [1, 2, 5]

DAYPART_ORDER = [
    "overnight",
    "am_peak",
    "midday",
    "pm_peak",
    "evening",
]

MODELING_TRACKS = [
    "post_cp_primary",
    "full_history_regime_aware",
    "pre_cp_reference",
]

TRACK_LABELS = {
    "post_cp_primary": "Post-CP Primary",
    "full_history_regime_aware": "Full-History Regime-Aware",
    "pre_cp_reference": "Pre-CP Reference",
}


# ---------------------------------------------------------------------
# Authoritative weather candidates
#
# These are the eight measures retained by 1.2.5. They are CANDIDATES here,
# not yet the final 4.2.3 forecasting weather family.
# ---------------------------------------------------------------------

WEATHER_CANDIDATE_METRICS = [
    "temperature",
    "precipitation",
    "visibility",
    "wind_speed",
    "wind_gust",
    "relative_humidity",
    "ceiling_height",
    "pressure_3hr_change",
]

WEATHER_LABELS = {
    "temperature": "Temperature",
    "precipitation": "Precipitation",
    "visibility": "Visibility",
    "wind_speed": "Wind speed",
    "wind_gust": "Wind gust",
    "relative_humidity": "Relative humidity",
    "ceiling_height": "Ceiling height",
    "pressure_3hr_change": "3-hour pressure change",
}

# Units are deliberately not asserted here. The final Taxi Zone weather
# handoff stores numeric values but does not itself encode a complete unit
# contract for every retained field. 4.2.3 therefore preserves the source
# values without unit-dependent transformations.


# ---------------------------------------------------------------------
# Project directories
# ---------------------------------------------------------------------

PIPELINE_DIR = Path("pipeline_data")

INPUT_131_DIR = PIPELINE_DIR / "1.3.1.final_tables"
INPUT_411_DIR = PIPELINE_DIR / "4.1.1.final_tables"
INPUT_412_DIR = PIPELINE_DIR / "4.1.2.final_tables"
INPUT_421_DIR = PIPELINE_DIR / "4.2.1.final_tables"
INPUT_422_DIR = PIPELINE_DIR / "4.2.2.final_tables"

INTERMEDIATE_DIR = PIPELINE_DIR / "4.2.3.intermediate_tables"
FINAL_DIR = PIPELINE_DIR / "4.2.3.final_tables"

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Foundation and forecasting-contract inputs
# ---------------------------------------------------------------------

MOBILITY_PANEL_PATH = (
    INPUT_131_DIR
    / "analysis_ready_mobility_panel.parquet"
)

WEATHER_CONTEXT_PATH = (
    INPUT_131_DIR
    / "weather_taxi_zone_context.parquet"
)

FORECAST_SEQUENCE_PATH = (
    INPUT_411_DIR
    / "forecasting_target_sequence_panel.parquet"
)

ROLLING_ORIGIN_PATH = (
    INPUT_412_DIR
    / "rolling_origin_registry.parquet"
)

FORECAST_ORIGIN_HORIZON_PATH = (
    INPUT_412_DIR
    / "forecast_origin_horizon_registry.parquet"
)

TRACK_CONTRACT_PATH = (
    INPUT_412_DIR
    / "forecasting_track_contract_registry.parquet"
)

PRE_CP_REFIT_CONTRACT_PATH = (
    INPUT_412_DIR
    / "pre_cp_refit_application_contract.parquet"
)


# ---------------------------------------------------------------------
# Completed 4.2.1 inputs
# ---------------------------------------------------------------------

ORIGIN_FEATURE_PATHS_421 = {
    "post_cp_primary":
        INPUT_421_DIR / "origin_feature_surface__post_cp_primary.parquet",

    "full_history_regime_aware":
        INPUT_421_DIR / "origin_feature_surface__full_history_regime_aware.parquet",

    "pre_cp_reference":
        INPUT_421_DIR / "origin_feature_surface__pre_cp_reference.parquet",
}

FEATURE_CATALOG_421_PATH = (
    INPUT_421_DIR
    / "forecast_feature_catalog.parquet"
)

TARGET_CONTEXT_421_PATH = (
    INPUT_421_DIR
    / "forecast_target_calendar_context.parquet"
)

PRE_CP_COUNTERFACTUAL_CALENDAR_421_PATH = (
    INPUT_421_DIR
    / "pre_cp_counterfactual_calendar.parquet"
)

PRE_CP_RECURSIVE_CONTRACT_421_PATH = (
    INPUT_421_DIR
    / "pre_cp_recursive_feature_contract.parquet"
)

FINAL_QA_421_PATH = (
    INPUT_421_DIR
    / "final_feature_contract_qa.parquet"
)


# ---------------------------------------------------------------------
# Completed 4.2.2 inputs
# ---------------------------------------------------------------------

ORIGIN_CONTEXT_PATHS_422 = {
    "post_cp_primary":
        INPUT_422_DIR / "origin_context_surface__post_cp_primary.parquet",

    "full_history_regime_aware":
        INPUT_422_DIR / "origin_context_surface__full_history_regime_aware.parquet",

    "pre_cp_reference":
        INPUT_422_DIR / "origin_context_surface__pre_cp_reference.parquet",
}

CONTEXT_CATALOG_422_PATH = (
    INPUT_422_DIR
    / "forecast_context_feature_catalog.parquet"
)

PRE_CP_CONTEXT_RUNTIME_422_PATH = (
    INPUT_422_DIR
    / "pre_cp_context_runtime_contract.parquet"
)

FINAL_QA_422_PATH = (
    INPUT_422_DIR
    / "final_context_contract_qa.parquet"
)


# ---------------------------------------------------------------------
# Planned 4.2.3 outputs
#
# These paths reserve the final handoff architecture. Nothing is written to
# them before the weather feature policy is selected.
# ---------------------------------------------------------------------

WEATHER_ALIGNMENT_PATH = (
    INTERMEDIATE_DIR
    / "forecast_weather_alignment.parquet"
)

WEATHER_FEATURE_CATALOG_PATH = (
    FINAL_DIR
    / "forecast_weather_feature_catalog.parquet"
)

FINAL_MATRIX_PATHS = {
    track: FINAL_DIR / f"forecasting_matrix__{track}.parquet"
    for track in MODELING_TRACKS
}

PRE_CP_FINAL_REFIT_SUPPLEMENT_PATH = (
    FINAL_DIR
    / "pre_cp_final_refit_training_supplement.parquet"
)

PRE_CP_COUNTERFACTUAL_TEMPLATE_PATH = (
    FINAL_DIR
    / "pre_cp_counterfactual_application_template.parquet"
)

FINAL_COMBINED_FEATURE_CATALOG_PATH = (
    FINAL_DIR
    / "forecast_feature_catalog_complete.parquet"
)

FINAL_RUNTIME_CONTRACT_PATH = (
    FINAL_DIR
    / "pre_cp_final_runtime_contract.parquet"
)

FINAL_QA_423_PATH = (
    FINAL_DIR
    / "final_forecasting_matrix_qa.parquet"
)

FINAL_MANIFEST_423_PATH = (
    FINAL_DIR
    / "forecasting_matrix_manifest.parquet"
)


# ---------------------------------------------------------------------
# Analytical keys
# ---------------------------------------------------------------------

ORIGIN_KEYS = [
    "modeling_track",
    "forecast_sequence_id",
    "observation_sequence_id",
]

TARGET_CONTEXT_KEYS = [
    "modeling_track",
    "origin_observation_sequence_id",
    "horizon",
]

FINAL_MATRIX_KEYS = [
    "modeling_track",
    "forecast_sequence_id",
    "origin_observation_sequence_id",
    "horizon",
]

CANONICAL_CALENDAR_KEY = [
    "observation_sequence_id",
]

WEATHER_SEQUENCE_KEY = [
    "taxi_zone_id",
    "observation_sequence_id",
]

WEATHER_UPSTREAM_KEY = [
    "taxi_zone_id",
    "date",
    "temporal_bucket",
]

Define a small set of shared loading, schema, grain, QA, and chart helpers.

As in 4.2.2, schema checks are concentrated at notebook boundaries. Internally constructed tables are protected through explicit merge validation and declared-key uniqueness checks rather than repeated speculative column handling.

In [2]:
# 4.2.3.0B — Define shared loading, schema, QA, and chart helpers

def require_file(path, name):
    """Fail clearly when an authoritative upstream artifact is unavailable."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"{name} was not found at: {path}"
        )


def parquet_columns(path):
    """Return top-level Parquet column names without materializing the table."""
    require_file(path, "Parquet input")

    return list(
        pq.ParquetFile(path).schema_arrow.names
    )


def parquet_schema_df(path):
    """Return the exact top-level Parquet schema as a compact DataFrame."""
    require_file(path, "Parquet input")

    schema = pq.ParquetFile(path).schema_arrow

    return pd.DataFrame(
        [
            {
                "column": field.name,
                "dtype": str(field.type),
            }
            for field in schema
        ]
    )


def require_parquet_columns(path, required_columns, name):
    """Validate an authoritative Parquet schema before loading data."""
    available = set(
        parquet_columns(path)
    )

    missing = sorted(
        set(required_columns).difference(available)
    )

    if missing:
        raise ValueError(
            f"{name} is missing required columns: {missing}"
        )


def validate_unique_key(df, key_columns, name):
    """Raise when a DataFrame violates its declared analytical grain."""
    duplicate_rows = int(
        df.duplicated(key_columns).sum()
    )

    if duplicate_rows:
        raise ValueError(
            f"{name} contains {duplicate_rows:,} duplicate rows "
            f"for key {key_columns}."
        )


def qa_row(check, passed, detail):
    """Create one compact reader-facing QA record."""
    return {
        "check": check,
        "status": "PASS" if bool(passed) else "FAIL",
        "detail": detail,
    }


def parquet_inventory_row(name, path, role):
    """Summarize one Parquet artifact without materializing the full dataset."""
    require_file(path, name)

    parquet_file = pq.ParquetFile(path)

    return {
        "input": name,
        "role": role,
        "rows": int(parquet_file.metadata.num_rows),
        "columns": len(parquet_file.schema_arrow.names),
        "size_mb": round(
            path.stat().st_size / (1024 ** 2),
            3,
        ),
    }


def apply_chart_layout(
    fig,
    *,
    title,
    x_title=None,
    y_title=None,
    height=CHART_HEIGHT,
):
    """
    Apply compact project-consistent Plotly styling.

    Chart dimensions remain below the project maximum. Individual traces are
    responsible for keeping displayed numeric precision to three decimals.
    """
    fig.update_layout(
        title=title,
        width=CHART_WIDTH,
        height=height,
        template="plotly_white",
        margin=dict(l=70, r=30, t=75, b=70),
        font=dict(size=12),
        xaxis_title=x_title,
        yaxis_title=y_title,
        legend_title_text="",
    )

    return fig

## 4.2.3.1 Load and Validate the Completed Forecasting Handoff

Before making any weather decision, confirm that 4.2.3 begins from the completed forecasting experiment rather than reconstructing it.

The checks here are deliberately lightweight. The large 4.2.1 and 4.2.2 origin surfaces are inventoried through Parquet metadata, while only the small feature catalogs, target calendar, runtime contracts, and completed QA tables are loaded.

The purpose is to establish three things before touching weather:

* both completed feature layers are present at their native modeling-track populations;
* the authoritative target calendar still represents exactly h=1, h=2, and h=5 from a shared forecast origin; and
* the two upstream predictor catalogs and counterfactual contracts remain internally consistent.

No final forecasting matrix is assembled in this section.

**Inventory the authoritative 4.2.3 inputs.** Review the actual forecasting, feature-layer, weather, and runtime artifacts that this notebook will consume.

This is a metadata-only inventory. Large feature surfaces are not materialized.

In [3]:
# 4.2.3.1A — Inventory authoritative forecasting and weather inputs

input_specs = [
    (
        "Canonical mobility panel",
        MOBILITY_PANEL_PATH,
        "Canonical Taxi Zone × sequence calendar and mobility backbone",
    ),
    (
        "Taxi Zone weather context",
        WEATHER_CONTEXT_PATH,
        "Authoritative harmonized weather handoff",
    ),
    (
        "Forecast target sequence panel",
        FORECAST_SEQUENCE_PATH,
        "Authoritative forecast outcomes and sequence identities",
    ),
    (
        "Rolling-origin registry",
        ROLLING_ORIGIN_PATH,
        "Scheduled model-selection and evaluation origins",
    ),
    (
        "Complete forecast origin × horizon registry",
        FORECAST_ORIGIN_HORIZON_PATH,
        "Full canonical origin × horizon calendar, including refit-only targets",
    ),
    (
        "Forecasting track contract",
        TRACK_CONTRACT_PATH,
        "Modeling-track time and evaluation boundaries",
    ),
    (
        "Pre-CP refit/application contract",
        PRE_CP_REFIT_CONTRACT_PATH,
        "Frozen selection, final-refit, and counterfactual boundaries",
    ),
]

for track, path in ORIGIN_FEATURE_PATHS_421.items():
    input_specs.append(
        (
            f"4.2.1 origin feature surface · {TRACK_LABELS[track]}",
            path,
            "Target-history, seasonal, and calendar predictors",
        )
    )

input_specs.extend([
    (
        "4.2.1 forecast feature catalog",
        FEATURE_CATALOG_421_PATH,
        "Target-history and temporal predictor definitions",
    ),
    (
        "4.2.1 forecast target context",
        TARGET_CONTEXT_421_PATH,
        "Horizon-specific target calendar",
    ),
    (
        "4.2.1 Pre-CP recursive feature contract",
        PRE_CP_RECURSIVE_CONTRACT_421_PATH,
        "Target-history recursive requirements",
    ),
    (
        "4.2.1 final QA",
        FINAL_QA_421_PATH,
        "Completed temporal-feature contract QA",
    ),
])

for track, path in ORIGIN_CONTEXT_PATHS_422.items():
    input_specs.append(
        (
            f"4.2.2 origin context surface · {TRACK_LABELS[track]}",
            path,
            "Multimodal, connected, and static origin context",
        )
    )

input_specs.extend([
    (
        "4.2.2 context feature catalog",
        CONTEXT_CATALOG_422_PATH,
        "Multimodal, connected, and static predictor definitions",
    ),
    (
        "4.2.2 Pre-CP context runtime contract",
        PRE_CP_CONTEXT_RUNTIME_422_PATH,
        "Counterfactual context availability rules",
    ),
    (
        "4.2.2 final QA",
        FINAL_QA_422_PATH,
        "Completed context-feature contract QA",
    ),
])

input_inventory_df = pd.DataFrame(
    [
        parquet_inventory_row(
            name,
            path,
            role,
        )
        for name, path, role in input_specs
    ]
)

display(input_inventory_df)

,input,role,rows,columns,size_mb
0,Canonical mobility panel,Canonical Taxi Zone × sequence calendar and mobility backbone,1559590,42,84.225
1,Taxi Zone weather context,Authoritative harmonized weather handoff,1537120,26,65.187
2,Forecast target sequence panel,Authoritative forecast outcomes and sequence identities,7797950,38,119.555
3,Rolling-origin registry,Scheduled model-selection and evaluation origins,35496,19,0.243
4,Complete forecast origin × horizon registry,"Full canonical origin × horizon calendar, including refit-only targets",17782,17,0.155
5,Forecasting track contract,Modeling-track time and evaluation boundaries,3,15,0.013
6,Pre-CP refit/application contract,"Frozen selection, final-refit, and counterfactual boundaries",1,10,0.009
7,4.2.1 origin feature surface · Post-CP Primary,"Target-history, seasonal, and calendar predictors",2497432,39,208.802
8,4.2.1 origin feature surface · Full-History Regime-Aware,"Target-history, seasonal, and calendar predictors",6231379,39,511.289
9,4.2.1 origin feature surface · Pre-CP Reference,"Target-history, seasonal, and calendar predictors",3818325,39,304.308


**Findings.** The authoritative 4.2.3 handoff is complete at the expected scales. The canonical mobility panel contains 1,559,590 rows, the forecast target sequence panel contains 7,797,950 rows, and the rolling-origin registry contains 35,496 scheduled origin/horizon rows. Most importantly, the completed 4.2.1 and 4.2.2 surfaces align exactly within each modeling track: 2,497,432 Post-CP Primary rows, 6,231,379 Full-History Regime-Aware rows, and 3,817,286 Pre-CP Reference rows. Weather can therefore be added without reconciling competing forecast-origin populations.

**Validate the lightweight upstream contracts.** Load the completed feature catalogs, horizon-specific target calendar, counterfactual runtime contract, and final QA records.

This check does not repeat the transformations performed in 4.2.1 or 4.2.2. It verifies that their final handoffs remain usable together: upstream QA must still pass, the target calendar must remain unique and correctly horizon-aligned, predictor names must not collide across the two completed feature families, and the 4.2.2 runtime contract must continue to forbid actual post-CP mobility in the no-CP application.

In [4]:
# 4.2.3.1B — Load and validate completed upstream forecasting contracts

# ---------------------------------------------------------------------
# Validate the final-refit calendar handoff used later in Section 6E
# ---------------------------------------------------------------------

require_parquet_columns(
    FORECAST_ORIGIN_HORIZON_PATH,
    [
        "observation_sequence_id", "date", "daypart", "daypart_order",
        "temporal_bucket", "temporal_bucket_order", "pre_post_cp",
        "horizon", "horizon_label", "target_observation_sequence_id",
        "target_date", "target_daypart", "target_daypart_order",
        "target_temporal_bucket", "target_temporal_bucket_order",
        "target_pre_post_cp", "target_period",
    ],
    "4.1.2 forecast origin × horizon registry",
)

require_parquet_columns(
    PRE_CP_REFIT_CONTRACT_PATH,
    [
        "modeling_track", "model_selection_target_end_date",
        "refit_only_start_date", "final_refit_end_date",
        "final_refit_end_observation_sequence_id",
        "counterfactual_application_start_date",
        "counterfactual_application_start_observation_sequence_id",
        "native_pre_cp_sequence_count",
    ],
    "4.1.2 Pre-CP refit/application contract",
)

forecast_origin_horizon_df = pd.read_parquet(
    FORECAST_ORIGIN_HORIZON_PATH
)
pre_cp_refit_contract_df = pd.read_parquet(
    PRE_CP_REFIT_CONTRACT_PATH
)

forecast_origin_horizon_df["date"] = pd.to_datetime(
    forecast_origin_horizon_df["date"]
).dt.normalize()

forecast_origin_horizon_df["target_date"] = pd.to_datetime(
    forecast_origin_horizon_df["target_date"]
).dt.normalize()

if len(pre_cp_refit_contract_df) != 1:
    raise RuntimeError(
        "The Pre-CP refit/application contract must contain exactly one row."
    )

refit_contract = pre_cp_refit_contract_df.iloc[0]

refit_boundary_checks = {
    "modeling_track":
        str(refit_contract["modeling_track"]) == "pre_cp_reference",

    "selection_end":
        pd.Timestamp(
            refit_contract["model_selection_target_end_date"]
        ).normalize() == pd.Timestamp("2024-12-31"),

    "refit_start":
        pd.Timestamp(
            refit_contract["refit_only_start_date"]
        ).normalize() == pd.Timestamp("2025-01-01"),

    "refit_end":
        pd.Timestamp(
            refit_contract["final_refit_end_date"]
        ).normalize() == PRE_CP_FREEZE_DATE,

    "refit_end_sequence":
        int(
            refit_contract["final_refit_end_observation_sequence_id"]
        ) == PRE_CP_FREEZE_OBSERVATION_SEQUENCE_ID,

    "application_start":
        pd.Timestamp(
            refit_contract["counterfactual_application_start_date"]
        ).normalize() == CONGESTION_PRICING_START_DATE,

    "application_start_sequence":
        int(
            refit_contract[
                "counterfactual_application_start_observation_sequence_id"
            ]
        ) == CP_FIRST_OBSERVATION_SEQUENCE_ID,
}

failed_refit_boundaries = [
    name
    for name, passed in refit_boundary_checks.items()
    if not passed
]

if failed_refit_boundaries:
    raise RuntimeError(
        "The Pre-CP refit/application boundary changed: "
        f"{failed_refit_boundaries}"
    )


# ---------------------------------------------------------------------
# Validate the minimum schemas actually consumed here
# ---------------------------------------------------------------------

require_parquet_columns(
    FEATURE_CATALOG_421_PATH,
    ["feature_name"],
    "4.2.1 forecast feature catalog",
)

require_parquet_columns(
    TARGET_CONTEXT_421_PATH,
    [
        "modeling_track",
        "stage",
        "origin_observation_sequence_id",
        "horizon",
        "target_observation_sequence_id",
        "origin_date",
        "origin_daypart",
        "origin_temporal_bucket",
        "target_date",
        "target_daypart",
    ],
    "4.2.1 forecast target context",
)

require_parquet_columns(
    FINAL_QA_421_PATH,
    ["check", "status", "detail"],
    "4.2.1 final QA",
)

require_parquet_columns(
    CONTEXT_CATALOG_422_PATH,
    [
        "feature_name",
        "feature_family",
        "metric_value_class",
        "ordinary_availability_class",
        "pre_cp_post_freeze_class",
        "drop_when_source_metric_equals_target_metric",
        "requires_cross_zone_recursive_state",
        "uses_actual_post_cp_mobility",
    ],
    "4.2.2 context feature catalog",
)

require_parquet_columns(
    PRE_CP_CONTEXT_RUNTIME_422_PATH,
    [
        "feature_name",
        "feature_family",
        "pre_cp_post_freeze_class",
        "counterfactual_runtime_source",
        "requires_cross_zone_recursive_state",
        "uses_actual_post_cp_mobility",
    ],
    "4.2.2 Pre-CP context runtime contract",
)

require_parquet_columns(
    FINAL_QA_422_PATH,
    ["check", "status", "detail"],
    "4.2.2 final QA",
)


# ---------------------------------------------------------------------
# Load only lightweight completed contracts
# ---------------------------------------------------------------------

forecast_feature_catalog_421_df = pd.read_parquet(
    FEATURE_CATALOG_421_PATH
)

forecast_target_context_df = pd.read_parquet(
    TARGET_CONTEXT_421_PATH
)

final_qa_421_df = pd.read_parquet(
    FINAL_QA_421_PATH
)

forecast_context_catalog_422_df = pd.read_parquet(
    CONTEXT_CATALOG_422_PATH
)

pre_cp_context_runtime_422_df = pd.read_parquet(
    PRE_CP_CONTEXT_RUNTIME_422_PATH
)

final_qa_422_df = pd.read_parquet(
    FINAL_QA_422_PATH
)

forecast_target_context_df["origin_date"] = pd.to_datetime(
    forecast_target_context_df["origin_date"]
)

forecast_target_context_df["target_date"] = pd.to_datetime(
    forecast_target_context_df["target_date"]
)


# ---------------------------------------------------------------------
# Contract QA
# ---------------------------------------------------------------------

contract_qa_rows = []

contract_qa_rows.append(
    qa_row(
        "All completed 4.2.1 QA checks still pass",
        final_qa_421_df["status"].eq("PASS").all(),
        (
            f"{int(final_qa_421_df['status'].eq('PASS').sum()):,}/"
            f"{len(final_qa_421_df):,} checks pass"
        ),
    )
)

contract_qa_rows.append(
    qa_row(
        "All completed 4.2.2 QA checks still pass",
        final_qa_422_df["status"].eq("PASS").all(),
        (
            f"{int(final_qa_422_df['status'].eq('PASS').sum()):,}/"
            f"{len(final_qa_422_df):,} checks pass"
        ),
    )
)

duplicate_target_context_rows = int(
    forecast_target_context_df.duplicated(
        TARGET_CONTEXT_KEYS
    ).sum()
)

contract_qa_rows.append(
    qa_row(
        "Forecast target context remains unique",
        duplicate_target_context_rows == 0,
        f"duplicate target-context keys = {duplicate_target_context_rows:,}",
    )
)

actual_tracks = set(
    forecast_target_context_df[
        "modeling_track"
    ].dropna().unique()
)

contract_qa_rows.append(
    qa_row(
        "Target context contains exactly the three modeling tracks",
        actual_tracks == set(MODELING_TRACKS),
        f"tracks = {sorted(actual_tracks)}",
    )
)

actual_horizons = set(
    pd.to_numeric(
        forecast_target_context_df["horizon"],
        errors="raise",
    )
    .astype(int)
    .unique()
)

contract_qa_rows.append(
    qa_row(
        "Target context contains exactly h=1, h=2, and h=5",
        actual_horizons == set(FORECAST_HORIZONS),
        f"horizons = {sorted(actual_horizons)}",
    )
)

horizon_relation_ok = (
    pd.to_numeric(
        forecast_target_context_df[
            "target_observation_sequence_id"
        ],
        errors="raise",
    )
    - pd.to_numeric(
        forecast_target_context_df[
            "origin_observation_sequence_id"
        ],
        errors="raise",
    )
).eq(
    pd.to_numeric(
        forecast_target_context_df["horizon"],
        errors="raise",
    )
).all()

contract_qa_rows.append(
    qa_row(
        "Every target sequence ID is exactly origin ID + horizon",
        horizon_relation_ok,
        "checked across the full horizon-specific target context",
    )
)

feature_names_421 = set(
    forecast_feature_catalog_421_df[
        "feature_name"
    ].dropna()
)

feature_names_422 = set(
    forecast_context_catalog_422_df[
        "feature_name"
    ].dropna()
)

catalog_name_overlap = sorted(
    feature_names_421.intersection(
        feature_names_422
    )
)

contract_qa_rows.append(
    qa_row(
        "Completed 4.2.1 and 4.2.2 predictor names do not collide",
        len(catalog_name_overlap) == 0,
        (
            "overlap = none"
            if not catalog_name_overlap
            else f"overlap = {catalog_name_overlap}"
        ),
    )
)

runtime_feature_names = set(
    pre_cp_context_runtime_422_df[
        "feature_name"
    ].dropna()
)

contract_qa_rows.append(
    qa_row(
        "4.2.2 Pre-CP runtime contract is a subset of its feature catalog",
        runtime_feature_names.issubset(
            feature_names_422
        ),
        (
            f"runtime features = {len(runtime_feature_names):,}; "
            f"context features = {len(feature_names_422):,}"
        ),
    )
)

actual_post_cp_mobility_allowed = (
    pre_cp_context_runtime_422_df[
        "uses_actual_post_cp_mobility"
    ]
    .eq(True)
    .any()
)

contract_qa_rows.append(
    qa_row(
        "No 4.2.2 counterfactual runtime feature uses actual post-CP mobility",
        not actual_post_cp_mobility_allowed,
        (
            "zero runtime features permit actual post-CP mobility"
            if not actual_post_cp_mobility_allowed
            else "at least one illegal runtime feature was found"
        ),
    )
)

upstream_weather_feature_names = [
    feature_name
    for feature_name in (
        feature_names_421
        | feature_names_422
    )
    if "weather" in str(feature_name).lower()
]

contract_qa_rows.append(
    qa_row(
        "Weather is still absent from the completed 4.2.1 and 4.2.2 catalogs",
        len(upstream_weather_feature_names) == 0,
        (
            "weather remains isolated for 4.2.3"
            if not upstream_weather_feature_names
            else f"unexpected weather features = {upstream_weather_feature_names}"
        ),
    )
)

upstream_contract_qa_df = pd.DataFrame(
    contract_qa_rows
)

display(upstream_contract_qa_df)

if not upstream_contract_qa_df["status"].eq("PASS").all():
    raise AssertionError(
        "4.2.3 cannot proceed because the completed upstream "
        "forecasting handoff failed one or more contract checks."
    )

,check,status,detail
0,All completed 4.2.1 QA checks still pass,PASS,24/24 checks pass
1,All completed 4.2.2 QA checks still pass,PASS,18/18 checks pass
2,Forecast target context remains unique,PASS,duplicate target-context keys = 0
3,Target context contains exactly the three modeling tracks,PASS,"tracks = ['full_history_regime_aware', 'post_cp_primary', 'pre_cp_reference']"
4,"Target context contains exactly h=1, h=2, and h=5",PASS,"horizons = [1, 2, 5]"
5,Every target sequence ID is exactly origin ID + horizon,PASS,checked across the full horizon-specific target context
6,Completed 4.2.1 and 4.2.2 predictor names do not collide,PASS,overlap = none
7,4.2.2 Pre-CP runtime contract is a subset of its feature catalog,PASS,runtime features = 41; context features = 41
8,No 4.2.2 counterfactual runtime feature uses actual post-CP mobility,PASS,zero runtime features permit actual post-CP mobility
9,Weather is still absent from the completed 4.2.1 and 4.2.2 catalogs,PASS,weather remains isolated for 4.2.3


**Findings.** All ten upstream contract checks pass. The completed 4.2.1 and 4.2.2 QA records remain fully green at 24/24 and 18/18 checks, the target calendar is unique and contains exactly the three modeling tracks and h=1, h=2, and h=5, and every target sequence ID remains exactly the origin sequence ID plus its horizon. The two predictor catalogs have no name collisions, the Pre-CP context runtime contract still permits zero use of actual post-CP mobility, and weather remains isolated from both upstream feature catalogs for explicit treatment in 4.2.3.

### Section Summary

Section 4.2.3.1 confirms that the notebook begins from a stable, internally consistent forecasting handoff rather than reconstructing upstream feature logic. The completed 4.2.1 and 4.2.2 origin surfaces align exactly within each native modeling track, while the authoritative rolling schedule preserves the three modeling tracks, h=1, h=2, and h=5, and the exact target = origin + horizon relationship.

All 24 completed 4.2.1 QA checks and all 18 completed 4.2.2 QA checks remain passing. Their predictor catalogs have no naming collisions, the 4.2.2 counterfactual runtime contract still permits zero actual post-CP mobility, and weather remains absent from both upstream feature layers. The notebook can therefore add weather and construct the final cross-family handoff without reopening or reconciling the completed temporal, multimodal, or spatial feature designs.


## 4.2.3.2 Define the Weather Availability Contract

Weather can enter forecasting in two distinct information roles.

**Origin-observed weather** describes conditions that have already occurred by the time the forecast is issued. These values are ordinary leakage-safe exogenous predictors and can be shared across h=1, h=2, and h=5 from the same origin.

**Target-period weather** describes conditions during the future mobility period being predicted. Because this project does not have a historical archive of the short-range weather forecast available at every origin, realized target-period weather may later be attached only as an explicit **forecast-weather proxy**. It must be aligned independently to the actual target sequence position for h=1, h=2, and h=5 and must never be described as information observed at the forecast origin.

For the frozen no-CP simulation, realized post-CP weather may remain available as exogenous context. This does not relax the separate prohibition against using actual post-CP mobility.

The weather family will remain deliberately small. We first inspect the authoritative weather table, identify any temporal gaps, measure actual usable coverage and spatial differentiation, and check whether candidate variables are strongly redundant. Only then is the final retained metric set selected.

No global imputation or weather transformation is performed during this diagnostic step.

**Inspect the exact authoritative weather schema.** Read the Parquet schema directly rather than relying on assumptions carried forward from weather staging or harmonization.

Classify each available field as a temporal/spatial key, candidate weather measure, assignment provenance field, or other metadata. The table itself remains unloaded in this block.

In [5]:
# 4.2.3.2A — Inspect the exact Taxi Zone weather-context schema

weather_schema_df = parquet_schema_df(
    WEATHER_CONTEXT_PATH
)

weather_key_columns = {
    "taxi_zone_id",
    "date",
    "daypart",
    "temporal_bucket",
    "observation_sequence_id",
}

weather_temporal_metadata_columns = {
    "year",
    "month",
    "day_of_week",
    "daypart_order",
    "temporal_bucket_order",
    "pre_post_cp",
}

weather_assignment_metadata_columns = {
    "taxi_zone_name",
    "taxi_zone_borough",
    "dominant_weather_station_id",
    "dominant_weather_station_name",
    "dominant_weather_station_weight",
    "weather_assignment_method",
    "weather_station_blend_count",
}

weather_candidate_metric_set = set(
    WEATHER_CANDIDATE_METRICS
)


def classify_weather_column(column):
    if column in weather_candidate_metric_set:
        return "candidate_weather_measure"

    if column in weather_key_columns:
        return "spatial_temporal_key"

    if column in weather_temporal_metadata_columns:
        return "temporal_metadata"

    if column in weather_assignment_metadata_columns:
        return "assignment_provenance"

    return "other_metadata"


weather_schema_df["role"] = (
    weather_schema_df["column"]
    .map(classify_weather_column)
)

require_parquet_columns(
    WEATHER_CONTEXT_PATH,
    [
        "taxi_zone_id",
        "date",
        "daypart",
        "daypart_order",
        "temporal_bucket",
        "temporal_bucket_order",
        "observation_sequence_id",
        "pre_post_cp",
        *WEATHER_CANDIDATE_METRICS,
    ],
    "Taxi Zone weather context",
)

display(weather_schema_df)

,column,dtype,role
0,taxi_zone_id,int64,spatial_temporal_key
1,taxi_zone_name,string,assignment_provenance
2,taxi_zone_borough,string,assignment_provenance
3,date,timestamp[ns],spatial_temporal_key
4,year,int64,temporal_metadata
5,month,int64,temporal_metadata
6,day_of_week,string,temporal_metadata
7,daypart,string,spatial_temporal_key
8,daypart_order,int64,temporal_metadata
9,temporal_bucket,string,spatial_temporal_key


**Findings.** The authoritative weather handoff provides the canonical spatial and temporal keys together with eight numeric candidate measures: temperature, precipitation, visibility, wind speed, wind gust, relative humidity, ceiling height, and three-hour pressure change. Station identity, station influence, assignment method, and blend-count fields remain separate provenance metadata rather than candidate predictors, preserving a clean boundary between the weather values used for forecasting and the metadata describing how Taxi Zone weather was estimated.

**Validate the weather grain against the canonical forecasting calendar.** Load the weather handoff together with only the calendar columns needed from the canonical mobility panel.

The weather table should resolve to one row per mapped Taxi Zone × canonical sequence position wherever weather context exists. This block measures its real temporal completeness rather than assuming that a minimum and maximum date imply every intervening Daypart is present.

The calendar comparison also verifies Date, Daypart, and temporal-bucket alignment and confirms whether the nonphysical TLC Unknown placeholders 264 and 265 are absent as intended.

In [6]:
# 4.2.3.2B — Validate weather grain, temporal completeness, and calendar alignment

CANONICAL_CALENDAR_COLUMNS = [
    "observation_sequence_id",
    "date",
    "daypart",
    "daypart_order",
    "temporal_bucket",
    "temporal_bucket_order",
    "pre_post_cp",
]

require_parquet_columns(
    MOBILITY_PANEL_PATH,
    CANONICAL_CALENDAR_COLUMNS,
    "Canonical mobility panel",
)

canonical_calendar_df = (
    pd.read_parquet(
        MOBILITY_PANEL_PATH,
        columns=CANONICAL_CALENDAR_COLUMNS,
    )
    .drop_duplicates()
    .sort_values(
        "observation_sequence_id"
    )
    .reset_index(drop=True)
)

canonical_calendar_df["date"] = pd.to_datetime(
    canonical_calendar_df["date"]
)

validate_unique_key(
    canonical_calendar_df,
    ["observation_sequence_id"],
    "Canonical forecasting calendar",
)


WEATHER_LOAD_COLUMNS = [
    "taxi_zone_id",
    "date",
    "daypart",
    "daypart_order",
    "temporal_bucket",
    "temporal_bucket_order",
    "observation_sequence_id",
    "pre_post_cp",
    *WEATHER_CANDIDATE_METRICS,
]

# Assignment metadata is useful for structural QA when present but is not
# required for weather-value construction.
for optional_column in [
    "weather_assignment_method",
    "weather_station_blend_count",
]:
    if optional_column in parquet_columns(
        WEATHER_CONTEXT_PATH
    ):
        WEATHER_LOAD_COLUMNS.append(
            optional_column
        )

weather_context_df = pd.read_parquet(
    WEATHER_CONTEXT_PATH,
    columns=WEATHER_LOAD_COLUMNS,
)

weather_context_df["date"] = pd.to_datetime(
    weather_context_df["date"]
)


# ---------------------------------------------------------------------
# Declared weather grains
# ---------------------------------------------------------------------

duplicate_zone_sequence_rows = int(
    weather_context_df.duplicated(
        WEATHER_SEQUENCE_KEY
    ).sum()
)

duplicate_upstream_key_rows = int(
    weather_context_df.duplicated(
        WEATHER_UPSTREAM_KEY
    ).sum()
)

weather_sequence_zone_counts = (
    weather_context_df.groupby(
        "observation_sequence_id",
        observed=True,
    )["taxi_zone_id"]
    .nunique()
)

weather_zone_ids = set(
    pd.to_numeric(
        weather_context_df["taxi_zone_id"],
        errors="raise",
    )
    .astype(int)
    .unique()
)

unknown_weather_zone_ids = sorted(
    weather_zone_ids.intersection(
        {264, 265}
    )
)


# ---------------------------------------------------------------------
# Calendar alignment
# ---------------------------------------------------------------------

weather_calendar_alignment_df = (
    weather_context_df[
        [
            "taxi_zone_id",
            "observation_sequence_id",
            "date",
            "daypart",
            "temporal_bucket",
            "pre_post_cp",
        ]
    ]
    .merge(
        canonical_calendar_df[
            [
                "observation_sequence_id",
                "date",
                "daypart",
                "temporal_bucket",
                "pre_post_cp",
            ]
        ],
        on="observation_sequence_id",
        how="left",
        validate="many_to_one",
        suffixes=("_weather", "_canonical"),
    )
)

unmatched_calendar_rows = int(
    weather_calendar_alignment_df[
        "date_canonical"
    ].isna().sum()
)

date_mismatch_rows = int(
    (
        weather_calendar_alignment_df[
            "date_weather"
        ]
        != weather_calendar_alignment_df[
            "date_canonical"
        ]
    ).fillna(True).sum()
)

daypart_mismatch_rows = int(
    (
        weather_calendar_alignment_df[
            "daypart_weather"
        ].astype(str)
        != weather_calendar_alignment_df[
            "daypart_canonical"
        ].astype(str)
    ).fillna(True).sum()
)

bucket_mismatch_rows = int(
    (
        weather_calendar_alignment_df[
            "temporal_bucket_weather"
        ].astype(str)
        != weather_calendar_alignment_df[
            "temporal_bucket_canonical"
        ].astype(str)
    ).fillna(True).sum()
)

regime_mismatch_rows = int(
    (
        weather_calendar_alignment_df[
            "pre_post_cp_weather"
        ].astype(str)
        != weather_calendar_alignment_df[
            "pre_post_cp_canonical"
        ].astype(str)
    ).fillna(True).sum()
)


# ---------------------------------------------------------------------
# Temporal completeness
# ---------------------------------------------------------------------

canonical_sequence_ids = set(
    canonical_calendar_df[
        "observation_sequence_id"
    ].tolist()
)

weather_sequence_ids = set(
    weather_context_df[
        "observation_sequence_id"
    ].tolist()
)

missing_weather_sequence_ids = sorted(
    canonical_sequence_ids.difference(
        weather_sequence_ids
    )
)

extra_weather_sequence_ids = sorted(
    weather_sequence_ids.difference(
        canonical_sequence_ids
    )
)


# ---------------------------------------------------------------------
# Compact reader-facing structural summary
# ---------------------------------------------------------------------

weather_structure_summary_df = pd.DataFrame(
    [
        {
            "diagnostic": "Weather rows",
            "value": len(weather_context_df),
        },
        {
            "diagnostic": "Mapped Taxi Zones",
            "value": weather_context_df["taxi_zone_id"].nunique(),
        },
        {
            "diagnostic": "First weather date",
            "value": weather_context_df["date"].min(),
        },
        {
            "diagnostic": "Last weather date",
            "value": weather_context_df["date"].max(),
        },
        {
            "diagnostic": "Distinct weather dates",
            "value": weather_context_df["date"].nunique(),
        },
        {
            "diagnostic": "Canonical sequence positions",
            "value": len(canonical_sequence_ids),
        },
        {
            "diagnostic": "Weather sequence positions",
            "value": len(weather_sequence_ids),
        },
        {
            "diagnostic": "Missing canonical weather positions",
            "value": len(missing_weather_sequence_ids),
        },
        {
            "diagnostic": "Extra noncanonical weather positions",
            "value": len(extra_weather_sequence_ids),
        },
        {
            "diagnostic": "Distinct Dayparts",
            "value": weather_context_df["daypart"].nunique(),
        },
        {
            "diagnostic": "Distinct temporal buckets",
            "value": weather_context_df["temporal_bucket"].nunique(),
        },
        {
            "diagnostic": "Duplicate Taxi Zone × sequence rows",
            "value": duplicate_zone_sequence_rows,
        },
        {
            "diagnostic": "Duplicate Taxi Zone × date × temporal-bucket rows",
            "value": duplicate_upstream_key_rows,
        },
        {
            "diagnostic": "Minimum mapped zones per represented sequence",
            "value": int(weather_sequence_zone_counts.min()),
        },
        {
            "diagnostic": "Maximum mapped zones per represented sequence",
            "value": int(weather_sequence_zone_counts.max()),
        },
        {
            "diagnostic": "TLC Unknown IDs 264/265 represented",
            "value": (
                ", ".join(map(str, unknown_weather_zone_ids))
                if unknown_weather_zone_ids
                else "None"
            ),
        },
        {
            "diagnostic": "Unmatched canonical-calendar rows",
            "value": unmatched_calendar_rows,
        },
        {
            "diagnostic": "Date mismatches",
            "value": date_mismatch_rows,
        },
        {
            "diagnostic": "Daypart mismatches",
            "value": daypart_mismatch_rows,
        },
        {
            "diagnostic": "Temporal-bucket mismatches",
            "value": bucket_mismatch_rows,
        },
        {
            "diagnostic": "Pre/post-CP mismatches",
            "value": regime_mismatch_rows,
        },
    ]
)

display(weather_structure_summary_df)

,diagnostic,value
0,Weather rows,1537120
1,Mapped Taxi Zones,260
2,First weather date,2023-01-01 00:00:00
3,Last weather date,2026-03-31 00:00:00
4,Distinct weather dates,1183
5,Canonical sequence positions,5930
6,Weather sequence positions,5912
7,Missing canonical weather positions,18
8,Extra noncanonical weather positions,0
9,Distinct Dayparts,5


**Findings.** The Taxi Zone weather context contains 1,537,120 rows across 260 mapped physical Taxi Zones from January 1, 2023 through March 31, 2026. It is unique at both Taxi Zone × sequence and Taxi Zone × Date × temporal-bucket grain, all represented sequence positions contain all 260 mapped zones, and Date, Daypart, temporal bucket, and congestion-pricing regime align perfectly with the canonical calendar. The only structural incompleteness is 18 missing canonical sequence positions; there are no extra positions, and the nonphysical TLC Unknown IDs 264 and 265 remain intentionally absent.

**Locate any missing weather sequence positions in the forecasting experiment.** A weather table can span the full study start and end dates while still omitting individual Dayparts in between.

Compare missing canonical weather positions with the actual rolling-origin target calendar. This distinguishes harmless gaps outside the scored forecasting schedule from gaps that occur at forecast origins or future target periods used by validation, holdout, or Pre-CP reference evaluation.

No filling is performed. Missing realized weather remains missing until a later model-specific preprocessing step establishes a legal treatment.

In [7]:
# 4.2.3.2C — Audit missing weather positions against scheduled forecasts

missing_weather_calendar_df = (
    canonical_calendar_df.loc[
        canonical_calendar_df[
            "observation_sequence_id"
        ].isin(
            missing_weather_sequence_ids
        )
    ]
    .copy()
)


# ---------------------------------------------------------------------
# Summarize where each sequence position is used by the scheduled forecast
# registry carried through 4.2.1's target context.
# ---------------------------------------------------------------------

origin_usage_df = (
    forecast_target_context_df.groupby(
        "origin_observation_sequence_id",
        observed=True,
    )
    .agg(
        scheduled_origin_rows=(
            "horizon",
            "size",
        ),
        scheduled_origin_stages=(
            "stage",
            lambda values: ", ".join(
                sorted(
                    pd.Series(values)
                    .dropna()
                    .astype(str)
                    .unique()
                )
            ),
        ),
    )
    .reset_index()
    .rename(
        columns={
            "origin_observation_sequence_id":
                "observation_sequence_id",
        }
    )
)

target_usage_df = (
    forecast_target_context_df.groupby(
        "target_observation_sequence_id",
        observed=True,
    )
    .agg(
        scheduled_target_rows=(
            "horizon",
            "size",
        ),
        scheduled_target_stages=(
            "stage",
            lambda values: ", ".join(
                sorted(
                    pd.Series(values)
                    .dropna()
                    .astype(str)
                    .unique()
                )
            ),
        ),
    )
    .reset_index()
    .rename(
        columns={
            "target_observation_sequence_id":
                "observation_sequence_id",
        }
    )
)

weather_gap_audit_df = (
    missing_weather_calendar_df
    .merge(
        origin_usage_df,
        on="observation_sequence_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        target_usage_df,
        on="observation_sequence_id",
        how="left",
        validate="one_to_one",
    )
)

for count_column in [
    "scheduled_origin_rows",
    "scheduled_target_rows",
]:
    weather_gap_audit_df[count_column] = (
        weather_gap_audit_df[count_column]
        .fillna(0)
        .astype(int)
    )

for stage_column in [
    "scheduled_origin_stages",
    "scheduled_target_stages",
]:
    weather_gap_audit_df[stage_column] = (
        weather_gap_audit_df[stage_column]
        .fillna("Not scheduled")
    )

weather_gap_audit_df = (
    weather_gap_audit_df[
        [
            "observation_sequence_id",
            "date",
            "daypart",
            "temporal_bucket",
            "pre_post_cp",
            "scheduled_origin_rows",
            "scheduled_origin_stages",
            "scheduled_target_rows",
            "scheduled_target_stages",
        ]
    ]
    .sort_values(
        "observation_sequence_id"
    )
    .reset_index(drop=True)
)

if weather_gap_audit_df.empty:
    weather_gap_audit_display_df = pd.DataFrame(
        [
            {
                "result":
                    "No canonical sequence positions are missing from weather."
            }
        ]
    )
else:
    weather_gap_audit_display_df = (
        weather_gap_audit_df
    )

display(weather_gap_audit_display_df)

,observation_sequence_id,date,daypart,temporal_bucket,pre_post_cp,scheduled_origin_rows,scheduled_origin_stages,scheduled_target_rows,scheduled_target_stages
0,4861,2025-08-30,overnight,weekend_overnight,post_cp,6,rolling_validation,6,rolling_validation
1,4862,2025-08-30,am_peak,weekend_am_peak,post_cp,6,rolling_validation,6,rolling_validation
2,4863,2025-08-30,midday,weekend_midday,post_cp,6,rolling_validation,6,rolling_validation
3,4864,2025-08-30,pm_peak,weekend_pm_peak,post_cp,6,rolling_validation,6,rolling_validation
4,4865,2025-08-30,evening,weekend_evening,post_cp,6,rolling_validation,6,rolling_validation
5,4866,2025-08-31,overnight,weekend_overnight,post_cp,6,rolling_validation,6,rolling_validation
6,4867,2025-08-31,am_peak,weekend_am_peak,post_cp,6,rolling_validation,6,rolling_validation
7,4868,2025-08-31,midday,weekend_midday,post_cp,6,rolling_validation,6,rolling_validation
8,4869,2025-08-31,pm_peak,weekend_pm_peak,post_cp,6,rolling_validation,6,rolling_validation
9,4870,2025-08-31,evening,weekend_evening,post_cp,6,rolling_validation,6,rolling_validation


**Findings.** The 18 structurally missing weather positions form one short contiguous post-CP gap from observation_sequence_id 4861 through 4878, spanning August 30 through September 2, 2025. These positions fall inside rolling validation and are used as both scheduled forecast origins and targets, so dropping them would alter the forecasting experiment and filling them from later weather would violate information-time discipline. The final 4.2.3 treatment therefore preserves those forecasting rows and completes only the retained weather measures from strictly earlier original weather observations before horizon alignment and matrix assembly.

**Profile candidate weather coverage and spatial differentiation.** Evaluate the eight retained upstream weather measures at the final Taxi Zone context grain.

Coverage is measured against the complete mapped-zone × canonical-sequence opportunity space so an entirely absent Daypart cannot disappear from the denominator. Spatial differentiation measures whether the inverse-distance station blend actually produces different values across Taxi Zones when a weather measure is available.

The summary also reports the observed numeric range without attaching units that are not encoded in the final weather contract.

This is feature-selection evidence only. Sparse values are not imputed and no candidate is removed automatically.

In [8]:
# 4.2.3.2D — Profile weather coverage and spatial differentiation

weather_zone_count = int(
    weather_context_df[
        "taxi_zone_id"
    ].nunique()
)

canonical_position_count = int(
    len(canonical_calendar_df)
)

expected_mapped_weather_rows = (
    weather_zone_count
    * canonical_position_count
)

canonical_sequence_index = pd.Index(
    canonical_calendar_df[
        "observation_sequence_id"
    ].tolist(),
    name="observation_sequence_id",
)

weather_metric_diagnostic_records = []

for metric in WEATHER_CANDIDATE_METRICS:
    metric_series = pd.to_numeric(
        weather_context_df[metric],
        errors="coerce",
    )

    non_null_rows = int(
        metric_series.notna().sum()
    )

    metric_group = (
        weather_context_df.assign(
            _metric_value=metric_series
        )
        .groupby(
            "observation_sequence_id",
            observed=True,
        )["_metric_value"]
    )

    non_null_zone_counts = (
        metric_group.count()
        .reindex(
            canonical_sequence_index,
            fill_value=0,
        )
    )

    unique_zone_values = (
        metric_group.nunique(
            dropna=True
        )
        .reindex(
            canonical_sequence_index,
            fill_value=0,
        )
    )

    positions_with_any = int(
        non_null_zone_counts.gt(0).sum()
    )

    fully_covered_positions = int(
        non_null_zone_counts.eq(
            weather_zone_count
        ).sum()
    )

    spatially_varying_positions = int(
        unique_zone_values.gt(1).sum()
    )

    observed_unique_counts = (
        unique_zone_values.loc[
            unique_zone_values.gt(0)
        ]
    )

    weather_metric_diagnostic_records.append(
        {
            "metric": metric,
            "metric_label": WEATHER_LABELS[metric],
            "non_null_rows": non_null_rows,
            "expected_mapped_row_coverage_pct": round(
                (
                    non_null_rows
                    / expected_mapped_weather_rows
                    * 100
                ),
                3,
            ),
            "positions_with_any_value": positions_with_any,
            "canonical_position_coverage_pct": round(
                (
                    positions_with_any
                    / canonical_position_count
                    * 100
                ),
                3,
            ),
            "fully_covered_positions": fully_covered_positions,
            "missing_canonical_positions": (
                canonical_position_count
                - positions_with_any
            ),
            "spatially_varying_positions": spatially_varying_positions,
            "spatial_variation_pct_of_observed_positions": round(
                (
                    spatially_varying_positions
                    / positions_with_any
                    * 100
                )
                if positions_with_any
                else np.nan,
                3,
            ),
            "median_distinct_zone_values_when_observed": round(
                float(
                    observed_unique_counts.median()
                )
                if not observed_unique_counts.empty
                else np.nan,
                3,
            ),
            "min_value": metric_series.min(),
            "median_value": metric_series.median(),
            "max_value": metric_series.max(),
        }
    )

weather_metric_diagnostic_df = pd.DataFrame(
    weather_metric_diagnostic_records
)

display(
    weather_metric_diagnostic_df[
        [
            "metric",
            "expected_mapped_row_coverage_pct",
            "canonical_position_coverage_pct",
            "missing_canonical_positions",
            "spatial_variation_pct_of_observed_positions",
            "median_distinct_zone_values_when_observed",
            "min_value",
            "median_value",
            "max_value",
        ]
    ].round(3)
)

,metric,expected_mapped_row_coverage_pct,canonical_position_coverage_pct,missing_canonical_positions,spatial_variation_pct_of_observed_positions,median_distinct_zone_values_when_observed,min_value,median_value,max_value
0,temperature,99.680,99.680,19,100.000,260.000,-15.225,12.909,38.186
1,precipitation,99.696,99.696,18,17.608,1.000,0.000,0.000,205.051
2,visibility,99.680,99.680,19,100.000,260.000,0.295,16.081,16.093
3,wind_speed,99.680,99.680,19,100.000,260.000,0.034,3.647,14.058
4,wind_gust,60.236,60.236,2358,96.781,260.000,7.200,10.465,31.887
5,relative_humidity,99.680,99.680,19,100.000,260.000,12.565,58.423,99.203
6,ceiling_height,99.595,99.595,24,99.983,260.000,61.665,"11,614.465","22,000.000"
7,pressure_3hr_change,99.629,99.629,22,99.949,260.000,-25.904,-0.084,7.700


**Findings.** Seven of the eight candidate weather measures have essentially complete coverage, with 99.595%–99.696% of mapped canonical opportunities observed; wind gust is the clear exception at 60.236% coverage and 2,358 missing sequence positions. Temperature, visibility, wind speed, relative humidity, ceiling height, and pressure change vary across Taxi Zones at virtually every observed position. Precipitation varies spatially only 17.608% of the time because most periods are dry and therefore share zero precipitation across zones. The results support retaining a compact weather family while preserving the small amount of natural source missingness rather than globally imputing it.

**Visual checkpoint — coverage versus spatial differentiation.** Compare how consistently each weather candidate is available with how often it meaningfully differs across Taxi Zones.

A feature can be highly complete but effectively citywide, or spatially informative but too sparse to serve as a dependable default predictor. Viewing both dimensions together makes that tradeoff visible before the final weather family is selected.

In [9]:
# 4.2.3.2E — Visualize weather coverage and spatial differentiation

if RUN_VISUALS:
    weather_visual_df = (
        weather_metric_diagnostic_df[
            [
                "metric_label",
                "expected_mapped_row_coverage_pct",
                "spatial_variation_pct_of_observed_positions",
            ]
        ]
        .copy()
    )

    fig = go.Figure()

    fig.add_trace(
        go.Bar(
            name="Mapped row coverage",
            x=weather_visual_df["metric_label"],
            y=weather_visual_df[
                "expected_mapped_row_coverage_pct"
            ],
            marker_color=BRAND_COLORS["dark_teal"],
            text=weather_visual_df[
                "expected_mapped_row_coverage_pct"
            ],
            texttemplate="%{text:.3f}%",
            hovertemplate=(
                "<b>%{x}</b><br>"
                "Mapped row coverage: %{y:.3f}%"
                "<extra></extra>"
            ),
        )
    )

    fig.add_trace(
        go.Bar(
            name="Spatially varying when observed",
            x=weather_visual_df["metric_label"],
            y=weather_visual_df[
                "spatial_variation_pct_of_observed_positions"
            ],
            marker_color=BRAND_COLORS["terracotta"],
            text=weather_visual_df[
                "spatial_variation_pct_of_observed_positions"
            ],
            texttemplate="%{text:.3f}%",
            hovertemplate=(
                "<b>%{x}</b><br>"
                "Spatially varying positions: %{y:.3f}%"
                "<extra></extra>"
            ),
        )
    )

    fig.update_layout(
        barmode="group"
    )

    apply_chart_layout(
        fig,
        title="Weather Candidate Coverage and Taxi Zone Differentiation",
        x_title=None,
        y_title="Share of opportunities / observed positions (%)",
        height=560,
    )

    fig.update_xaxes(
        tickangle=-25
    )

    fig.update_yaxes(
        range=[0, 105]
    )

    fig.show()

**Findings.** The coverage-versus-differentiation view makes the weather tradeoff clear: wind gust is the only candidate with materially weak availability, while the remaining measures are nearly complete across the study period. Most well-covered measures also preserve consistent Taxi Zone differentiation. Precipitation is the expected exception because dry periods produce a common zero across zones, so its lower spatial-variation frequency reflects the episodic nature of precipitation rather than a failure of the spatial weather assignment.

**Check redundancy among the candidate weather measures.** Collapse the spatial weather surface to one citywide weather state per canonical sequence position and compare candidate measures using rank correlation.

This diagnostic is intentionally simple. Its purpose is not to optimize feature selection against the mobility target; doing that before legal model backtesting would invite outcome-informed selection. Instead, it identifies weather measures that may be carrying nearly the same exogenous information as another candidate.

Citywide sequence summaries are used only for this redundancy diagnostic. They are not forecasting predictors and will not replace Taxi Zone-level weather in the final matrices.

In [10]:
# 4.2.3.2F — Measure candidate weather redundancy without using outcomes

citywide_weather_state_df = (
    weather_context_df.groupby(
        "observation_sequence_id",
        observed=True,
    )[WEATHER_CANDIDATE_METRICS]
    .mean()
    .reindex(
        canonical_sequence_index
    )
)

weather_pair_redundancy_records = []

for left_index, left_metric in enumerate(
    WEATHER_CANDIDATE_METRICS
):
    for right_metric in WEATHER_CANDIDATE_METRICS[
        left_index + 1:
    ]:
        pair_df = (
            citywide_weather_state_df[
                [
                    left_metric,
                    right_metric,
                ]
            ]
            .dropna()
        )

        matched_positions = len(
            pair_df
        )

        if matched_positions >= 2:
            # Spearman correlation is Pearson correlation of ranks. Computing
            # it explicitly avoids introducing another package dependency.
            ranked_pair_df = (
                pair_df.rank(
                    method="average"
                )
            )

            spearman_rho = (
                ranked_pair_df[left_metric]
                .corr(
                    ranked_pair_df[right_metric]
                )
            )
        else:
            spearman_rho = np.nan

        weather_pair_redundancy_records.append(
            {
                "metric_a": left_metric,
                "metric_b": right_metric,
                "matched_sequence_positions": matched_positions,
                "matched_position_share_pct": round(
                    (
                        matched_positions
                        / canonical_position_count
                        * 100
                    ),
                    3,
                ),
                "spearman_rho": round(
                    float(spearman_rho),
                    3,
                )
                if pd.notna(spearman_rho)
                else np.nan,
                "abs_spearman_rho": round(
                    abs(
                        float(spearman_rho)
                    ),
                    3,
                )
                if pd.notna(spearman_rho)
                else np.nan,
            }
        )

weather_pair_redundancy_df = (
    pd.DataFrame(
        weather_pair_redundancy_records
    )
    .sort_values(
        [
            "abs_spearman_rho",
            "matched_sequence_positions",
        ],
        ascending=[
            False,
            False,
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)

display(weather_pair_redundancy_df)

,metric_a,metric_b,matched_sequence_positions,matched_position_share_pct,spearman_rho,abs_spearman_rho
0,wind_speed,wind_gust,3572,60.236,0.740,0.740
1,visibility,relative_humidity,5911,99.680,-0.690,0.690
2,precipitation,visibility,5911,99.680,-0.570,0.570
3,relative_humidity,ceiling_height,5906,99.595,-0.556,0.556
4,precipitation,ceiling_height,5906,99.595,-0.537,0.537
5,precipitation,relative_humidity,5911,99.680,0.536,0.536
6,visibility,ceiling_height,5906,99.595,0.420,0.420
7,wind_speed,relative_humidity,5911,99.680,-0.300,0.300
8,ceiling_height,pressure_3hr_change,5903,99.545,0.269,0.269
9,precipitation,pressure_3hr_change,5908,99.629,-0.231,0.231


**Findings.** Redundancy is moderate rather than severe across the weather candidates. Wind speed and wind gust form the strongest pair at Spearman ρ = 0.740, but gust is available for only 60.236% of sequence positions, making wind speed the more dependable representation of that weather dimension. Visibility and relative humidity are also related at ρ = −0.690, while precipitation has moderate associations with visibility, ceiling height, and humidity. No well-covered pair is close enough to duplicate one another outright, so the final reduction can emphasize interpretability, transportation relevance, and parsimony rather than imposing an arbitrary correlation threshold.

> ✅ Decision — Weather Feature and Missingness Contract RESOLVED
The forecasting weather family is locked to four compact, interpretable source measures:

Temperature · Precipitation · Visibility · Wind Speed

These measures provide distinct representations of thermal conditions, wet\-weather disruption, atmospheric impairment, and wind exposure while avoiding unnecessary feature proliferation\. Wind gust is excluded because of materially weaker coverage and substantial overlap with wind speed; relative humidity, ceiling height, and three\-hour pressure change are excluded because the retained family already captures the principal weather dimensions most directly interpretable for mobility\.

Each retained measure will enter forecasting in two explicitly different timing roles:

1\. Origin\-observed weather— weather already observed at the forecast origin and shared across h=1, h=2, and h=5 from that origin\.
2\. Realized target\-period weather proxy— weather aligned to the actual future target sequence position for each horizon\. Because archived historical weather forecasts are unavailable, this remains an explicitly labeled proxy for the weather forecast that would have been available operationally\.

The small amount of source weather missingness will be resolved once in 4\.2\.3 rather than independently inside each modeling pipeline\. Missing retained weather values will use a deterministic causal climatology based only on weather observations occurring strictly earlier than the missing sequence position:

Taxi Zone × calendar month × Daypart historical median → Taxi Zone × Daypart historical median → citywide calendar month × Daypart historical median → overall historical metric median

No future weather observations, mobility outcomes, interpolation across future periods, or model\-track outcomes may contribute to imputation\. The completed source weather surface is therefore reusable across all three modeling tracks without introducing model\-specific preprocessing differences\.

Imputation indicators and methods will be retained for QA and provenance only; they will \*\*not\*\* become forecasting predictors\.

For the frozen Pre\-CP no\-CP application, completed weather remains valid exogenous context, including realized post\-CP weather\. The separate prohibition against using actual post\-CP mobility remains unchanged\.


### Section Summary

Section 4.2.3.2 established the forecasting-time weather contract from the authoritative Taxi Zone weather handoff.

The source is structurally clean: represented weather rows are unique at Taxi Zone × canonical sequence position, all represented positions map consistently to the forecasting calendar, all 260 physical weather-mapped Taxi Zones are present, and the nonphysical TLC Unknown placeholders remain appropriately absent. The only calendar-level limitation is a short contiguous source outage of **18 canonical Daypart positions** from August 30 through September 2, 2025. Because that interval occurs during rolling validation and can appear at both forecast origins and future targets, it cannot simply be removed from the experiment.

Coverage profiling also separated ordinary source missingness from a genuinely sparse candidate. Temperature, precipitation, visibility, wind speed, relative humidity, ceiling height, and three-hour pressure change are essentially complete across the mapped weather surface, whereas wind gust is available for only about 60% of canonical positions. Candidate redundancy is moderate rather than extreme, with wind speed and wind gust forming the strongest pair and visibility and relative humidity also carrying overlapping atmospheric information.

The resulting production weather contract therefore keeps **temperature, precipitation, visibility, and wind speed**. Each will enter forecasting twice: once as weather already observed at the forecast origin and once as realized weather at the future target period, explicitly labeled as a proxy for unavailable archived operational weather forecasts.

Source missingness will also be resolved here rather than independently inside each later modeling pipeline. Missing retained weather values will be completed once using a causal historical climatology based exclusively on weather observations from earlier canonical sequence positions. This keeps the later ARIMA/SARIMAX, VARIMA, panel-tree, and neural-sequence workflows on one common weather surface while preventing future observations or mobility outcomes from entering the completion rule.

## 4.2.3.3 Construct and Align Weather Features

The weather policy is now fixed. This section turns it into a reusable forecasting layer before any model-specific work begins.

There are two separate tasks.

First, construct the complete physical Taxi Zone weather surface. The authoritative source omits a small number of entire canonical positions and contains a few additional metric-level gaps. Because these are properties of the external weather source rather than properties of a forecasting algorithm, they are completed once here. Original observed weather values are never overwritten, and imputed values never become donors for later imputations.

Second, attach the resulting weather state to the forecasting schedule in two distinct information-time roles. Origin weather is joined from the actual forecast-origin sequence position and therefore remains identical across h=1, h=2, and h=5 issued from that origin. Target-period weather is joined independently from each horizon's target sequence position and remains explicitly labeled as a realized-weather proxy.

No mobility outcome is used to construct or complete weather.

**Lock the production weather feature and source-completion contract.** Record the four retained weather measures, their eight final forecasting feature names, the approved causal climatology hierarchy, and the additional 4.2.3 artifacts used to audit that process.

In [11]:
# 4.2.3.3A — Lock the production weather feature and completion contract

WEATHER_RETAINED_METRICS = ["temperature", "precipitation", "visibility", "wind_speed"]

# Missing source values are filled from the most local strictly-prior climatology
# available. Broader fallbacks exist only so every downstream model receives the
# same completed exogenous source rather than reimplementing its own imputer.
WEATHER_IMPUTATION_METHOD_ORDER = [
    "taxi_zone_month_daypart_prior_median",
    "taxi_zone_daypart_prior_median",
    "citywide_month_daypart_prior_median",
    "overall_prior_median",
]

WEATHER_ORIGIN_FEATURE_COLUMNS = [f"weather_origin__{m}" for m in WEATHER_RETAINED_METRICS]
WEATHER_TARGET_PROXY_FEATURE_COLUMNS = [f"weather_target_proxy__{m}" for m in WEATHER_RETAINED_METRICS]
WEATHER_FORECAST_FEATURE_COLUMNS = WEATHER_ORIGIN_FEATURE_COLUMNS + WEATHER_TARGET_PROXY_FEATURE_COLUMNS

COMPLETED_WEATHER_CONTEXT_PATH = INTERMEDIATE_DIR / "weather_taxi_zone_context_completed.parquet"
WEATHER_IMPUTATION_AUDIT_PATH = FINAL_DIR / "weather_imputation_audit.parquet"
WEATHER_RUNTIME_CONTRACT_PATH = FINAL_DIR / "pre_cp_weather_runtime_contract.parquet"
FEATURE_PACKAGE_REVIEW_PATH = INTERMEDIATE_DIR / "forecast_feature_package_review.parquet"

# These rebuild switches are deliberately local to the weather artifacts whose
# contents changed when the retained family and completion policy were locked.
REBUILD_COMPLETED_WEATHER = False

weather_decision_contract_df = pd.DataFrame({
    "source_metric": WEATHER_RETAINED_METRICS,
    "origin_feature": [f"weather_origin__{m}" for m in WEATHER_RETAINED_METRICS],
    "target_proxy_feature": [f"weather_target_proxy__{m}" for m in WEATHER_RETAINED_METRICS],
    "source_completion": "Strictly-prior causal historical climatology",
})

display(weather_decision_contract_df)

,source_metric,origin_feature,target_proxy_feature,source_completion
0,temperature,weather_origin__temperature,weather_target_proxy__temperature,Strictly-prior causal historical climatology
1,precipitation,weather_origin__precipitation,weather_target_proxy__precipitation,Strictly-prior causal historical climatology
2,visibility,weather_origin__visibility,weather_target_proxy__visibility,Strictly-prior causal historical climatology
3,wind_speed,weather_origin__wind_speed,weather_target_proxy__wind_speed,Strictly-prior causal historical climatology


**Findings.** The production weather contract is now fixed at four source measures—temperature, precipitation, visibility, and wind speed—each represented in both origin-observed and horizon-specific target-period proxy form. This creates eight final weather predictor definitions while keeping source completion centralized in 4.2.3. The same completed exogenous weather surface can therefore be reused by every downstream forecasting approach without independently reimplementing missing-value logic.

**Define the causal source-completion helper.** Each missing metric value is filled from the original observed weather history available strictly before that sequence position.

The helper deliberately keeps the donor DataFrame separate from the completed DataFrame. An imputed value may therefore solve the current missing cell, but it can never propagate forward and influence the climatological median used to fill another one.

In [12]:
# 4.2.3.3B — Define causal weather completion

def fill_weather_metric_causally(completed_df, donor_df, metric):
    """
    Fill one retained weather metric from strictly earlier observed weather.

    Hierarchy
    ---------
    1. Same Taxi Zone × calendar month × Daypart median.
    2. Same Taxi Zone × Daypart median.
    3. Citywide calendar month × Daypart median.
    4. Overall prior metric median.

    `donor_df` must contain original observed source values only; completed or
    imputed weather is intentionally never recycled into later donor history.
    """
    flag_col = f"{metric}__imputed"
    completed_df[flag_col] = completed_df[metric].isna()

    missing_ids = sorted(
        completed_df.loc[completed_df[metric].isna(), "observation_sequence_id"]
        .astype(int).unique().tolist()
    )
    audit_frames = []

    for sequence_id in missing_ids:
        target_idx = completed_df.index[
            completed_df["observation_sequence_id"].eq(sequence_id)
            & completed_df[metric].isna()
        ]
        if not len(target_idx):
            continue

        target = completed_df.loc[
            target_idx, ["taxi_zone_id", "observation_sequence_id", "date", "month", "daypart"]
        ].copy()

        # One canonical sequence position must resolve to one calendar month and
        # Daypart before climatology can be applied consistently across zones.
        months = target["month"].dropna().astype(int).unique()
        dayparts = target["daypart"].dropna().astype(str).unique()
        if len(months) != 1 or len(dayparts) != 1:
            raise ValueError(f"Sequence {sequence_id} has inconsistent calendar metadata.")

        month, daypart = int(months[0]), str(dayparts[0])
        history = donor_df.loc[
            donor_df["observation_sequence_id"].lt(sequence_id) & donor_df[metric].notna(),
            ["taxi_zone_id", "observation_sequence_id", "month", "daypart", metric],
        ]

        if history.empty:
            raise ValueError(
                f"No strictly earlier observed {metric} history exists for sequence {sequence_id}."
            )

        latest_donor_id = int(history["observation_sequence_id"].max())
        if latest_donor_id >= sequence_id:
            raise AssertionError("Weather completion attempted to use non-prior donor history.")

        values = pd.Series(np.nan, index=target_idx, dtype=float)
        methods = pd.Series(pd.NA, index=target_idx, dtype="object")
        zone_ids = target["taxi_zone_id"]

        # Build all four candidates from the same legal history cutoff. Later
        # candidates fill only cells unresolved by the more local comparison.
        zone_month_daypart = (
            history.loc[history["month"].eq(month) & history["daypart"].astype(str).eq(daypart)]
            .groupby("taxi_zone_id", observed=True)[metric].median()
        )
        zone_daypart = (
            history.loc[history["daypart"].astype(str).eq(daypart)]
            .groupby("taxi_zone_id", observed=True)[metric].median()
        )
        city_month_daypart = history.loc[
            history["month"].eq(month) & history["daypart"].astype(str).eq(daypart), metric
        ].median()
        overall_prior = history[metric].median()

        candidates = [
            ("taxi_zone_month_daypart_prior_median", zone_ids.map(zone_month_daypart)),
            ("taxi_zone_daypart_prior_median", zone_ids.map(zone_daypart)),
            ("citywide_month_daypart_prior_median", pd.Series(city_month_daypart, index=target_idx)),
            ("overall_prior_median", pd.Series(overall_prior, index=target_idx)),
        ]

        for method, candidate in candidates:
            use = values.isna() & candidate.notna()
            values.loc[use] = candidate.loc[use]
            methods.loc[use] = method

        if values.isna().any():
            raise ValueError(
                f"Causal weather completion left {int(values.isna().sum()):,} "
                f"{metric} values unresolved at sequence {sequence_id}."
            )

        completed_df.loc[target_idx, metric] = values

        # Keep detailed provenance outside the predictor surface so later model
        # code can audit completion without accidentally learning the outage flag.
        audit = target.assign(
            source_metric=metric,
            imputed_value=values,
            imputation_method=methods,
            history_cutoff_observation_sequence_id=sequence_id - 1,
            latest_observed_donor_sequence_id=latest_donor_id,
        )
        audit_frames.append(audit)

    audit_df = pd.concat(audit_frames, ignore_index=True) if audit_frames else pd.DataFrame()
    return completed_df, audit_df

**Construct the completed physical weather surface.** Expand the source onto the expected 260 physical Taxi Zones × 5,930 canonical sequence positions before filling anything.

This makes entire missing Dayparts explicit rather than allowing them to disappear from the source table. Completion is then applied independently to the four retained metrics from the immutable original observed source.

In [13]:
# 4.2.3.3C — Construct and export the completed source-weather surface

if COMPLETED_WEATHER_CONTEXT_PATH.exists() and not REBUILD_COMPLETED_WEATHER:
    completed_weather_context_df = pd.read_parquet(COMPLETED_WEATHER_CONTEXT_PATH)
    weather_imputation_audit_df = pd.read_parquet(WEATHER_IMPUTATION_AUDIT_PATH)

else:
    # Recreate the complete physical weather opportunity space first so the
    # 18 fully absent source positions become ordinary missing cells.
    physical_zone_df = pd.DataFrame({"taxi_zone_id": sorted(int(z) for z in weather_zone_ids)})
    weather_calendar_df = canonical_calendar_df[
        [
            "observation_sequence_id", "date", "daypart", "daypart_order",
            "temporal_bucket", "temporal_bucket_order", "pre_post_cp",
        ]
    ].copy()

    weather_grid_df = physical_zone_df.merge(weather_calendar_df, how="cross")
    weather_grid_df["month"] = pd.to_datetime(weather_grid_df["date"]).dt.month.astype(int)

    # Only original weather measurements become donors. The completed copy is
    # mutated separately so synthetic values never enter future climatologies.
    weather_source_df = weather_grid_df.merge(
        weather_context_df[
            ["taxi_zone_id", "observation_sequence_id", *WEATHER_RETAINED_METRICS]
        ],
        on=["taxi_zone_id", "observation_sequence_id"],
        how="left",
        validate="one_to_one",
    )
    weather_original_observed_df = weather_source_df.copy()
    completed_weather_context_df = weather_source_df.copy()

    audit_frames = []
    for metric in WEATHER_RETAINED_METRICS:
        completed_weather_context_df, metric_audit_df = fill_weather_metric_causally(
            completed_weather_context_df, weather_original_observed_df, metric
        )
        if not metric_audit_df.empty:
            audit_frames.append(metric_audit_df)

    weather_imputation_audit_df = pd.concat(audit_frames, ignore_index=True)

    imputation_flags = [f"{m}__imputed" for m in WEATHER_RETAINED_METRICS]
    completed_weather_context_df["weather_imputed_metric_count"] = (
        completed_weather_context_df[imputation_flags].sum(axis=1).astype(int)
    )
    completed_weather_context_df["weather_any_imputed"] = (
        completed_weather_context_df["weather_imputed_metric_count"].gt(0)
    )

    if WRITE_INTERMEDIATES:
        completed_weather_context_df.to_parquet(COMPLETED_WEATHER_CONTEXT_PATH, index=False)
    if WRITE_FINAL_OUTPUTS:
        weather_imputation_audit_df.to_parquet(WEATHER_IMPUTATION_AUDIT_PATH, index=False)

# Reader-facing output focuses on how much source repair was required and which
# fallback actually solved it; detailed cell-level provenance remains exported.
completion_records = []
for metric in WEATHER_RETAINED_METRICS:
    metric_audit = weather_imputation_audit_df.loc[
        weather_imputation_audit_df["source_metric"].eq(metric)
    ]
    method_counts = metric_audit["imputation_method"].value_counts().to_dict()

    completion_records.append({
        "source_metric": metric,
        "imputed_cells": len(metric_audit),
        "imputed_share_pct": round(
            len(metric_audit) / len(completed_weather_context_df) * 100, 3
        ),
        "methods_used": ", ".join(f"{k}={v:,}" for k, v in method_counts.items()),
        "remaining_missing_cells": int(completed_weather_context_df[metric].isna().sum()),
    })

weather_completion_summary_df = pd.DataFrame(completion_records)
display(weather_completion_summary_df)

,source_metric,imputed_cells,imputed_share_pct,methods_used,remaining_missing_cells
0,temperature,4940,0.320,"taxi_zone_month_daypart_prior_median=4,940",0
1,precipitation,4680,0.304,"taxi_zone_month_daypart_prior_median=4,680",0
2,visibility,4940,0.320,"taxi_zone_month_daypart_prior_median=4,940",0
3,wind_speed,4940,0.320,"taxi_zone_month_daypart_prior_median=4,940",0


**Findings.** Causal source completion fills 19,500 retained-weather cells across the 1,541,800-row physical Taxi Zone weather surface, with only 0.304%–0.320% of rows requiring completion for any individual retained measure. All missing values are resolved by the most local approved fallback—the same Taxi Zone × calendar month × Daypart median calculated from strictly prior observations—so the broader zone-Daypart, citywide month-Daypart, and overall historical fallbacks are never needed. No retained weather value remains missing after completion.

**Validate causal weather completion.** The completed surface must preserve the exact physical Taxi Zone × canonical-position grain, fill every retained metric, use only approved historical fallbacks, and keep the nonphysical Unknown placeholders outside the weather system.

The known August 30–September 2, 2025 outage is checked separately because it is the structural gap that intersects scored rolling validation.

In [14]:
# 4.2.3.3D — Validate causal source completion

KNOWN_WEATHER_OUTAGE_SEQUENCE_IDS = list(range(4861, 4879))
expected_weather_rows = weather_zone_count * canonical_position_count
expected_outage_cells = (
    len(KNOWN_WEATHER_OUTAGE_SEQUENCE_IDS)
    * weather_zone_count
    * len(WEATHER_RETAINED_METRICS)
)

qa_rows = [
    qa_row(
        "Completed weather has the full physical-zone × canonical-position grain",
        len(completed_weather_context_df) == expected_weather_rows,
        f"rows = {len(completed_weather_context_df):,}; expected = {expected_weather_rows:,}",
    ),
    qa_row(
        "Completed weather remains unique by Taxi Zone × sequence",
        not completed_weather_context_df.duplicated(WEATHER_SEQUENCE_KEY).any(),
        f"duplicate keys = {int(completed_weather_context_df.duplicated(WEATHER_SEQUENCE_KEY).sum()):,}",
    ),
    qa_row(
        "All four retained weather measures are complete",
        completed_weather_context_df[WEATHER_RETAINED_METRICS].notna().all().all(),
        f"remaining NaNs = {int(completed_weather_context_df[WEATHER_RETAINED_METRICS].isna().sum().sum()):,}",
    ),
    qa_row(
        "Every imputation donor is strictly earlier than the missing sequence",
        weather_imputation_audit_df["latest_observed_donor_sequence_id"]
        .lt(weather_imputation_audit_df["observation_sequence_id"]).all(),
        f"audited cells = {len(weather_imputation_audit_df):,}",
    ),
    qa_row(
        "Only approved causal completion methods are used",
        set(weather_imputation_audit_df["imputation_method"].dropna())
        .issubset(set(WEATHER_IMPUTATION_METHOD_ORDER)),
        f"methods = {sorted(weather_imputation_audit_df['imputation_method'].dropna().unique())}",
    ),
]

outage_audit_cells = int(
    weather_imputation_audit_df["observation_sequence_id"]
    .isin(KNOWN_WEATHER_OUTAGE_SEQUENCE_IDS).sum()
)
qa_rows.extend([
    qa_row(
        "The 18-position 2025 source outage is complete for all retained measures",
        outage_audit_cells == expected_outage_cells,
        f"completed outage cells = {outage_audit_cells:,}; expected = {expected_outage_cells:,}",
    ),
    qa_row(
        "TLC Unknown IDs 264/265 remain outside the completed weather surface",
        not set(completed_weather_context_df["taxi_zone_id"].astype(int)).intersection({264, 265}),
        "IDs 264/265 absent",
    ),
])

weather_completion_qa_df = pd.DataFrame(qa_rows)
display(weather_completion_qa_df)

if not weather_completion_qa_df["status"].eq("PASS").all():
    raise AssertionError("Causal weather source completion failed QA.")

,check,status,detail
0,Completed weather has the full physical-zone × canonical-position grain,PASS,"rows = 1,541,800; expected = 1,541,800"
1,Completed weather remains unique by Taxi Zone × sequence,PASS,duplicate keys = 0
2,All four retained weather measures are complete,PASS,remaining NaNs = 0
3,Every imputation donor is strictly earlier than the missing sequence,PASS,"audited cells = 19,500"
4,Only approved causal completion methods are used,PASS,methods = ['taxi_zone_month_daypart_prior_median']
5,The 18-position 2025 source outage is complete for all retained measures,PASS,"completed outage cells = 18,720; expected = 18,720"
6,TLC Unknown IDs 264/265 remain outside the completed weather surface,PASS,IDs 264/265 absent


**Findings.** All seven source-completion checks pass. The completed weather surface contains exactly 1,541,800 rows with zero duplicate Taxi Zone × sequence keys and zero remaining missing values across the four retained measures. All 19,500 completed cells use observed donor history strictly earlier than the missing sequence position, and the 18-position August 30–September 2, 2025 source outage is fully repaired across all 260 mapped zones and all four retained measures. TLC Unknown IDs 264 and 265 remain outside the weather surface rather than receiving fabricated geographic weather.

**Inspect the repaired outage directly.** Show a compact long-form sample from three Taxi Zones during the known source outage.

The purpose is not to judge the weather values individually. It is to make the completion rule auditable by showing the completed value, the fallback used, and the latest legal observed sequence available to the donor history.

In [15]:
# 4.2.3.3E — Review a compact sample of the completed outage

review_zone_ids = sorted(completed_weather_context_df["taxi_zone_id"].unique())[:3]

outage_review_df = (
    weather_imputation_audit_df.loc[
        weather_imputation_audit_df["observation_sequence_id"]
        .isin(KNOWN_WEATHER_OUTAGE_SEQUENCE_IDS)
        & weather_imputation_audit_df["taxi_zone_id"].isin(review_zone_ids),
        [
            "taxi_zone_id", "observation_sequence_id", "date", "daypart",
            "source_metric", "imputed_value", "imputation_method",
            "latest_observed_donor_sequence_id",
        ],
    ]
    .sort_values(["observation_sequence_id", "taxi_zone_id", "source_metric"])
    .head(24)
    .reset_index(drop=True)
)

display(outage_review_df.round(3))

,taxi_zone_id,observation_sequence_id,date,daypart,source_metric,imputed_value,imputation_method,latest_observed_donor_sequence_id
0,1,4861,2025-08-30,overnight,precipitation,0.000,taxi_zone_month_daypart_prior_median,4860
1,1,4861,2025-08-30,overnight,temperature,23.312,taxi_zone_month_daypart_prior_median,4860
2,1,4861,2025-08-30,overnight,visibility,16.078,taxi_zone_month_daypart_prior_median,4860
3,1,4861,2025-08-30,overnight,wind_speed,3.065,taxi_zone_month_daypart_prior_median,4860
4,2,4861,2025-08-30,overnight,precipitation,0.000,taxi_zone_month_daypart_prior_median,4860
5,2,4861,2025-08-30,overnight,temperature,22.569,taxi_zone_month_daypart_prior_median,4860
6,2,4861,2025-08-30,overnight,visibility,16.080,taxi_zone_month_daypart_prior_median,4860
7,2,4861,2025-08-30,overnight,wind_speed,3.353,taxi_zone_month_daypart_prior_median,4860
8,3,4861,2025-08-30,overnight,precipitation,0.000,taxi_zone_month_daypart_prior_median,4860
9,3,4861,2025-08-30,overnight,temperature,23.010,taxi_zone_month_daypart_prior_median,4860


**Findings.** The sampled outage rows confirm the intended completion behavior at the individual-cell level. Missing temperature, precipitation, visibility, and wind-speed values are filled from each Taxi Zone's own strictly prior month × Daypart climatology, while the recorded donor cutoff remains before the missing sequence position. The review therefore supports the aggregate QA result: the short source outage is repaired locally without interpolation, future weather, or propagation of previously imputed values.

**Attach completed weather to the forecasting schedule.** Build a normalized Track × Taxi Zone × Origin × Horizon weather layer before duplicating it across target Forecast Sequences.

Origin weather is attached by `origin_observation_sequence_id`. Target-period proxy weather is attached separately by `target_observation_sequence_id`. This preserves the project's core horizon rule: h=1, h=2, and h=5 reuse the same origin state while pointing to different future target positions.

In [16]:
# 4.2.3.3F — Construct the horizon-aligned weather layer

weather_schedule_cols = [
    "modeling_track", "stage", "origin_observation_sequence_id", "horizon",
    "target_observation_sequence_id", "origin_date", "origin_daypart",
    "origin_temporal_bucket", "target_date", "target_daypart",
]
weather_schedule_df = forecast_target_context_df[weather_schedule_cols].copy()
physical_zone_df = pd.DataFrame({"taxi_zone_id": sorted(int(z) for z in weather_zone_ids)})

# Keep only predictor values plus one provenance flag in the large alignment.
# Metric-level imputation detail already lives in the normalized audit artifact.
origin_rename = {
    m: f"weather_origin__{m}" for m in WEATHER_RETAINED_METRICS
}
origin_rename["observation_sequence_id"] = "origin_observation_sequence_id"
origin_rename["weather_any_imputed"] = "origin_weather_any_imputed"

target_rename = {
    m: f"weather_target_proxy__{m}" for m in WEATHER_RETAINED_METRICS
}
target_rename["observation_sequence_id"] = "target_observation_sequence_id"
target_rename["weather_any_imputed"] = "target_weather_any_imputed"

join_cols = [
    "taxi_zone_id", "observation_sequence_id",
    *WEATHER_RETAINED_METRICS, "weather_any_imputed",
]
origin_weather_df = completed_weather_context_df[join_cols].rename(columns=origin_rename)
target_weather_df = completed_weather_context_df[join_cols].rename(columns=target_rename)

if WEATHER_ALIGNMENT_PATH.exists() and not REBUILD_WEATHER_ALIGNMENT:
    forecast_weather_alignment_df = pd.read_parquet(WEATHER_ALIGNMENT_PATH)
else:
    forecast_weather_alignment_df = (
        weather_schedule_df
        .merge(physical_zone_df, how="cross")
        .merge(
            origin_weather_df,
            on=["taxi_zone_id", "origin_observation_sequence_id"],
            how="left",
            validate="many_to_one",
        )
        .merge(
            target_weather_df,
            on=["taxi_zone_id", "target_observation_sequence_id"],
            how="left",
            validate="many_to_one",
        )
        .sort_values(
            ["modeling_track", "origin_observation_sequence_id", "horizon", "taxi_zone_id"]
        )
        .reset_index(drop=True)
    )

    if WRITE_INTERMEDIATES:
        forecast_weather_alignment_df.to_parquet(WEATHER_ALIGNMENT_PATH, index=False)

weather_alignment_summary_df = (
    forecast_weather_alignment_df
    .groupby(["modeling_track", "horizon"], observed=True)
    .agg(
        rows=("taxi_zone_id", "size"),
        taxi_zones=("taxi_zone_id", "nunique"),
        origins=("origin_observation_sequence_id", "nunique"),
        targets=("target_observation_sequence_id", "nunique"),
        origin_rows_with_imputation=("origin_weather_any_imputed", "sum"),
        target_rows_with_imputation=("target_weather_any_imputed", "sum"),
    )
    .reset_index()
)

display(weather_alignment_summary_df)

,modeling_track,horizon,rows,taxi_zones,origins,targets,origin_rows_with_imputation,target_rows_with_imputation
0,full_history_regime_aware,1,1541540,260,5929,5929,4940,4940
1,full_history_regime_aware,2,1541280,260,5928,5928,4940,4940
2,full_history_regime_aware,5,1540500,260,5925,5925,4940,4940
3,post_cp_primary,1,586040,260,2254,2254,4940,4940
4,post_cp_primary,2,585780,260,2253,2253,4940,4940
5,post_cp_primary,5,585000,260,2250,2250,4940,4940
6,pre_cp_reference,1,950040,260,3654,3654,0,0
7,pre_cp_reference,2,949780,260,3653,3653,0,0
8,pre_cp_reference,5,949000,260,3650,3650,0,0


**Findings.** The completed weather surface expands cleanly across the authoritative forecasting schedule, producing a normalized Track × Taxi Zone × Origin × Horizon layer for all three modeling tracks and all three forecast horizons. Origin-side completion appears only where an imputed source position coincides with a scheduled origin, while target-side completion follows the corresponding horizon-specific target position. This preserves the intended information structure: one common origin weather state can support h=1, h=2, and h=5, while each horizon receives weather from its own future target sequence.

**Validate horizon alignment.** Every scheduled Track × Origin × Horizon row must be preserved for all 260 physical weather zones, every target sequence must remain exactly `origin + horizon`, and the eight retained weather predictors must be complete.

The test verifies alignment by sequence identity rather than demanding different numeric weather at every horizon; neighboring future periods can legitimately have similar or identical weather.

In [17]:
# 4.2.3.3G — Validate the horizon-aligned weather layer

expected_alignment_rows = len(weather_schedule_df) * weather_zone_count
alignment_key = [
    "modeling_track", "taxi_zone_id", "origin_observation_sequence_id", "horizon"
]

weather_alignment_qa_df = pd.DataFrame([
    qa_row(
        "Every scheduled Track × Origin × Horizon row is preserved for every physical weather zone",
        len(forecast_weather_alignment_df) == expected_alignment_rows,
        f"rows = {len(forecast_weather_alignment_df):,}; expected = {expected_alignment_rows:,}",
    ),
    qa_row(
        "Weather alignment is unique at Track × Zone × Origin × Horizon",
        not forecast_weather_alignment_df.duplicated(alignment_key).any(),
        f"duplicate keys = {int(forecast_weather_alignment_df.duplicated(alignment_key).sum()):,}",
    ),
    qa_row(
        "Every target weather position equals origin position + horizon",
        (
            forecast_weather_alignment_df["target_observation_sequence_id"]
            - forecast_weather_alignment_df["origin_observation_sequence_id"]
        ).eq(forecast_weather_alignment_df["horizon"]).all(),
        "checked across the complete weather alignment",
    ),
    qa_row(
        "All eight retained weather predictors are complete",
        forecast_weather_alignment_df[WEATHER_FORECAST_FEATURE_COLUMNS].notna().all().all(),
        f"remaining predictor NaNs = {int(forecast_weather_alignment_df[WEATHER_FORECAST_FEATURE_COLUMNS].isna().sum().sum()):,}",
    ),
    qa_row(
        "Weather alignment contains exactly h=1, h=2, and h=5",
        sorted(forecast_weather_alignment_df["horizon"].astype(int).unique().tolist())
        == FORECAST_HORIZONS,
        f"horizons = {sorted(forecast_weather_alignment_df['horizon'].astype(int).unique())}",
    ),
    qa_row(
        "Weather alignment remains restricted to mapped physical zones",
        len(set(forecast_weather_alignment_df["taxi_zone_id"].astype(int))) == weather_zone_count
        and not set(forecast_weather_alignment_df["taxi_zone_id"].astype(int)).intersection({264, 265}),
        f"mapped zones = {forecast_weather_alignment_df['taxi_zone_id'].nunique():,}; IDs 264/265 absent",
    ),
])

display(weather_alignment_qa_df)

if not weather_alignment_qa_df["status"].eq("PASS").all():
    raise AssertionError("Horizon-aligned weather layer failed QA.")

,check,status,detail
0,Every scheduled Track × Origin × Horizon row is preserved for every physical weather zone,PASS,"rows = 9,228,960; expected = 9,228,960"
1,Weather alignment is unique at Track × Zone × Origin × Horizon,PASS,duplicate keys = 0
2,Every target weather position equals origin position + horizon,PASS,checked across the complete weather alignment
3,All eight retained weather predictors are complete,PASS,remaining predictor NaNs = 0
4,"Weather alignment contains exactly h=1, h=2, and h=5",PASS,"horizons = [1, 2, 5]"
5,Weather alignment remains restricted to mapped physical zones,PASS,mapped zones = 260; IDs 264/265 absent


**Findings.** All six horizon-alignment checks pass. The weather layer contains exactly 9,228,960 rows—the 35,496 scheduled Track × Origin × Horizon records crossed with 260 mapped physical Taxi Zones—with zero duplicate alignment keys. Every target sequence remains exactly origin + horizon, h=1, h=2, and h=5 are all preserved, and all eight production weather predictors are complete. The layer remains restricted to the intended 260 mapped zones, with TLC Unknown IDs 264 and 265 absent.

**Publish the weather feature catalog.** Encode the eight production weather predictors with their timing, ordinary availability, counterfactual availability, completion policy, and target-weather proxy caveat.

The catalog—not individual model code—becomes the authoritative statement of what a weather feature means and when it is legal to use.

In [18]:
# 4.2.3.3H — Build and export the production weather feature catalog

weather_catalog_records = []

for metric in WEATHER_RETAINED_METRICS:
    common = {
        "feature_family": "weather",
        "context_subfamily": "exogenous_weather",
        "source_metric": metric,
        "source_metric_label": WEATHER_LABELS[metric],
        "metric_value_class": "continuous",
        "feature_form": "source_weather_value",
        "drop_when_source_metric_equals_target_metric": False,
        "requires_cross_zone_recursive_state": False,
        "uses_actual_post_cp_mobility": False,
        "uses_realized_post_cp_weather": True,
        "source_completion_policy": (
            "Strictly-prior causal climatology: Taxi Zone × month × Daypart → "
            "Taxi Zone × Daypart → citywide month × Daypart → overall prior median."
        ),
    }

    weather_catalog_records.extend([
        {
            **common,
            "feature_name": f"weather_origin__{metric}",
            "weather_timing_role": "origin_observed",
            "ordinary_availability_class": "origin_observed_exogenous",
            "pre_cp_post_freeze_class": "exogenous_available",
            "counterfactual_runtime_source": "completed realized weather at forecast origin",
            "proxy_caveat": None,
        },
        {
            **common,
            "feature_name": f"weather_target_proxy__{metric}",
            "weather_timing_role": "target_period_realized_proxy",
            "ordinary_availability_class": "target_period_exogenous_proxy",
            "pre_cp_post_freeze_class": "exogenous_available",
            "counterfactual_runtime_source": "completed realized weather at exact target position",
            "proxy_caveat": (
                "Realized target-period weather substitutes for unavailable archived "
                "origin-time weather forecasts and may make historical evaluation optimistic."
            ),
        },
    ])

forecast_weather_feature_catalog_df = (
    pd.DataFrame(weather_catalog_records)
    .sort_values(["weather_timing_role", "source_metric"])
    .reset_index(drop=True)
)

if len(forecast_weather_feature_catalog_df) != 8:
    raise AssertionError("The production weather catalog must contain exactly eight predictors.")
if forecast_weather_feature_catalog_df["feature_name"].duplicated().any():
    raise ValueError("The production weather catalog contains duplicate feature names.")

if WRITE_FINAL_OUTPUTS:
    forecast_weather_feature_catalog_df.to_parquet(WEATHER_FEATURE_CATALOG_PATH, index=False)

display(
    forecast_weather_feature_catalog_df[
        [
            "feature_name", "source_metric", "weather_timing_role",
            "ordinary_availability_class", "pre_cp_post_freeze_class", "proxy_caveat",
        ]
    ]
)

,feature_name,source_metric,weather_timing_role,ordinary_availability_class,pre_cp_post_freeze_class,proxy_caveat
0,weather_origin__precipitation,precipitation,origin_observed,origin_observed_exogenous,exogenous_available,None
1,weather_origin__temperature,temperature,origin_observed,origin_observed_exogenous,exogenous_available,None
2,weather_origin__visibility,visibility,origin_observed,origin_observed_exogenous,exogenous_available,None
3,weather_origin__wind_speed,wind_speed,origin_observed,origin_observed_exogenous,exogenous_available,None
4,weather_target_proxy__precipitation,precipitation,target_period_realized_proxy,target_period_exogenous_proxy,exogenous_available,Realized target-period weather substitutes for unavailable archived origin-time weather forecasts and may make historical evaluation optimistic.
5,weather_target_proxy__temperature,temperature,target_period_realized_proxy,target_period_exogenous_proxy,exogenous_available,Realized target-period weather substitutes for unavailable archived origin-time weather forecasts and may make historical evaluation optimistic.
6,weather_target_proxy__visibility,visibility,target_period_realized_proxy,target_period_exogenous_proxy,exogenous_available,Realized target-period weather substitutes for unavailable archived origin-time weather forecasts and may make historical evaluation optimistic.
7,weather_target_proxy__wind_speed,wind_speed,target_period_realized_proxy,target_period_exogenous_proxy,exogenous_available,Realized target-period weather substitutes for unavailable archived origin-time weather forecasts and may make historical evaluation optimistic.


**Findings.** The production catalog contains exactly eight unique weather predictors: four origin-observed values and four horizon-specific realized target-period weather proxies. Origin weather is explicitly classified as observed exogenous information, while the four target-period fields retain a separate proxy classification and the caveat that realized weather may make historical forecast evaluation optimistic relative to a true archived weather forecast. All eight features remain exogenous and independent of post-CP mobility.

### Section Summary

Section 4\.2\.3\.3 converts the selected weather family into a complete forecasting\-ready source and timing contract\. The physical weather surface now spans all 260 mapped Taxi Zones and all 5,930 canonical sequence positions, with 19,500 sparse source cells completed using only each Taxi Zone's own strictly prior month × Daypart climatology\. No broader fallback was required, no observed value was replaced, and the nonphysical TLC Unknown zones remain outside the weather system\.

That completed source is then attached separately at the forecast origin and the horizon\-specific target period\. The resulting weather alignment contains all 9,228,960 scheduled Track × Taxi Zone × Origin × Horizon rows with exact h=1, h=2, and h=5 target arithmetic and no remaining weather missingness\. The final catalog therefore contributes eight production predictors—four genuinely origin\-observed weather fields and four explicitly labeled realized target\-period weather proxies—under one common source\-completion policy for every later forecasting approach\.

## 4.2.3.4 Define Weather Treatment for the No-CP Counterfactual

Weather does not follow the recursive mobility rules used by the frozen Pre-CP Reference.

After January 4, 2025, actual transportation outcomes belong to the treated congestion-pricing world and cannot populate the hypothetical no-CP state. Weather remains external to that treatment. The counterfactual can therefore ask what mobility would have looked like without congestion pricing **under the weather that actually occurred**.

Both production weather timing forms remain available after the freeze. Origin weather comes from the completed exogenous weather state at the current synthetic forecast origin, while target-period proxy weather comes from the completed realized weather state at the exact future target position. Neither requires actual post-CP mobility or cross-zone recursive transportation state.

**Formalize the frozen Pre-CP weather runtime contract.** Carry the production feature definitions forward and state exactly how each weather timing role is sourced after the mobility history freezes.

In [19]:
# 4.2.3.4A — Build the frozen Pre-CP weather runtime contract

runtime_cols = [
    "feature_name", "feature_family", "context_subfamily", "source_metric",
    "weather_timing_role", "ordinary_availability_class", "pre_cp_post_freeze_class",
    "counterfactual_runtime_source", "uses_realized_post_cp_weather",
    "uses_actual_post_cp_mobility", "source_completion_policy", "proxy_caveat",
]
pre_cp_weather_runtime_contract_df = forecast_weather_feature_catalog_df[runtime_cols].copy()

# Weather reads from the completed external source; it never participates in
# the recursive mobility-state generator that supplies post-freeze target history.
pre_cp_weather_runtime_contract_df["post_freeze_action"] = (
    pre_cp_weather_runtime_contract_df["weather_timing_role"].map({
        "origin_observed": "Read completed exogenous weather at the current counterfactual origin.",
        "target_period_realized_proxy": "Read completed realized weather at the exact horizon-specific target.",
    })
)
pre_cp_weather_runtime_contract_df["counterfactual_interpretation"] = (
    "No-CP mobility conditioned on the weather that actually occurred."
)

if WRITE_FINAL_OUTPUTS:
    pre_cp_weather_runtime_contract_df.to_parquet(WEATHER_RUNTIME_CONTRACT_PATH, index=False)

display(
    pre_cp_weather_runtime_contract_df[
        [
            "feature_name", "weather_timing_role", "pre_cp_post_freeze_class",
            "uses_realized_post_cp_weather", "uses_actual_post_cp_mobility",
            "post_freeze_action",
        ]
    ]
)

,feature_name,weather_timing_role,pre_cp_post_freeze_class,uses_realized_post_cp_weather,uses_actual_post_cp_mobility,post_freeze_action
0,weather_origin__precipitation,origin_observed,exogenous_available,True,False,Read completed exogenous weather at the current counterfactual origin.
1,weather_origin__temperature,origin_observed,exogenous_available,True,False,Read completed exogenous weather at the current counterfactual origin.
2,weather_origin__visibility,origin_observed,exogenous_available,True,False,Read completed exogenous weather at the current counterfactual origin.
3,weather_origin__wind_speed,origin_observed,exogenous_available,True,False,Read completed exogenous weather at the current counterfactual origin.
4,weather_target_proxy__precipitation,target_period_realized_proxy,exogenous_available,True,False,Read completed realized weather at the exact horizon-specific target.
5,weather_target_proxy__temperature,target_period_realized_proxy,exogenous_available,True,False,Read completed realized weather at the exact horizon-specific target.
6,weather_target_proxy__visibility,target_period_realized_proxy,exogenous_available,True,False,Read completed realized weather at the exact horizon-specific target.
7,weather_target_proxy__wind_speed,target_period_realized_proxy,exogenous_available,True,False,Read completed realized weather at the exact horizon-specific target.


**Findings.** The frozen Pre-CP runtime contract carries all eight production weather predictors forward as exogenous context. Origin weather reads the completed realized weather state at the current counterfactual origin, while the four target-period proxy fields read completed realized weather at the exact h=1, h=2, or h=5 target position. All eight features explicitly permit realized post-CP weather while permitting no actual post-CP mobility, keeping weather outside the recursive transportation-state generator.

**Validate the no-CP weather boundary.** Every retained weather predictor must remain exogenous after the freeze, use zero actual post-CP mobility, and preserve the realized-target-weather caveat where applicable.

In [20]:
# 4.2.3.4B — Validate frozen no-CP weather availability

target_proxy_rows = pre_cp_weather_runtime_contract_df[
    pre_cp_weather_runtime_contract_df["weather_timing_role"].eq("target_period_realized_proxy")
]

weather_runtime_qa_df = pd.DataFrame([
    qa_row(
        "All eight weather predictors are represented in the runtime contract",
        len(pre_cp_weather_runtime_contract_df) == 8,
        f"runtime rows = {len(pre_cp_weather_runtime_contract_df):,}",
    ),
    qa_row(
        "Every weather predictor remains exogenous after the Pre-CP freeze",
        pre_cp_weather_runtime_contract_df["pre_cp_post_freeze_class"]
        .eq("exogenous_available").all(),
        "all eight features = exogenous_available",
    ),
    qa_row(
        "No counterfactual weather predictor uses actual post-CP mobility",
        not pre_cp_weather_runtime_contract_df["uses_actual_post_cp_mobility"].eq(True).any(),
        "actual post-CP mobility usage = 0",
    ),
    qa_row(
        "Realized post-CP weather remains legal exogenous context",
        pre_cp_weather_runtime_contract_df["uses_realized_post_cp_weather"].eq(True).all(),
        "all eight weather features may condition on realized weather",
    ),
    qa_row(
        "Every target-period weather feature retains its explicit proxy caveat",
        target_proxy_rows["proxy_caveat"].notna().all(),
        f"target-proxy features = {len(target_proxy_rows):,}",
    ),
    qa_row(
        "Weather never requires recursive cross-zone mobility state",
        not forecast_weather_feature_catalog_df["requires_cross_zone_recursive_state"]
        .eq(True).any(),
        "recursive cross-zone weather features = 0",
    ),
])

display(weather_runtime_qa_df)

if not weather_runtime_qa_df["status"].eq("PASS").all():
    raise AssertionError("Frozen no-CP weather runtime contract failed QA.")

,check,status,detail
0,All eight weather predictors are represented in the runtime contract,PASS,runtime rows = 8
1,Every weather predictor remains exogenous after the Pre-CP freeze,PASS,all eight features = exogenous_available
2,No counterfactual weather predictor uses actual post-CP mobility,PASS,actual post-CP mobility usage = 0
3,Realized post-CP weather remains legal exogenous context,PASS,all eight weather features may condition on realized weather
4,Every target-period weather feature retains its explicit proxy caveat,PASS,target-proxy features = 4
5,Weather never requires recursive cross-zone mobility state,PASS,recursive cross-zone weather features = 0


**Findings.** All six counterfactual weather checks pass. The runtime contract contains all eight weather predictors, every one remains classified as exogenous after the January 4 freeze, and none uses actual post-CP mobility or requires recursive cross-zone transportation state. All four target-period weather features retain their explicit realized-weather proxy caveat. The frozen no-CP path can therefore condition predicted mobility on the weather that actually occurred without weakening the project's separate prohibition against treated post-CP mobility observations.

### Section Summary

Section 4\.2\.3\.4 makes the distinction between transportation state and external weather explicit for the no\-congestion\-pricing application\. After the Pre\-CP Reference freezes, target\-derived mobility history must be generated recursively, but weather does not become part of that recursion\. The completed observed weather state remains available as external context at both the current origin and each future target period\.

All eight weather predictors therefore survive the freeze without reading actual post\-CP mobility or requiring predicted cross\-zone transportation state\. This preserves the intended counterfactual interpretation: later forecasts estimate mobility without congestion pricing while conditioning on the weather that actually occurred\.

## 4.2.3.5 Assemble the Complete Forecasting Feature Universe

The three forecasting feature layers are now complete enough to describe the full predictor universe before the large horizon-specific model matrices are physically materialized.

This is the visual and architectural checkpoint intentionally deferred from 4.2.1 and 4.2.2. The combined catalog records **what every feature is, when it exists, and how its availability changes in the frozen no-CP application**.

A separate package-review layer then records how those definitions may be used by the later forecasting approaches:

* **ARIMA/SARIMAX** — univariate statistical forecasting in which target dependence is primarily represented through the model's own autoregressive structure and time-varying exogenous context may enter SARIMAX;
* **VARIMA** — multivariate statistical forecasting in which the joint history of the forecast targets is represented as the endogenous vector rather than by feeding the full tabular feature matrix into the model;
* **panel tree models** — tabular panel forecasting capable of using the broad leakage-safe feature universe and nonlinear interactions;
* **neural sequence models** — ordered target history plus optional dynamic, static, and exogenous context.

This package review does **not** select predictors using mobility outcomes, validation performance, or holdout results. It records structural suitability before modeling begins.

**Normalize the completed 4.2.1, 4.2.2, and weather catalogs.** Preserve the upstream contracts while projecting all three feature layers into one common schema.

The shared catalog contains feature definitions rather than a target-specific physical matrix. The two local 4.2.2 fields derived from a Forecast Sequence's own target metric therefore remain present in the shared catalog with their exclusion rule explicitly preserved; they will be removed during target-specific assembly.

In [21]:
# 4.2.3.5A — Assemble the complete cross-family feature catalog

require_421 = [
    "feature_name", "feature_family", "feature_side", "model_role",
    "ordinary_availability_class", "pre_cp_post_freeze_class",
    "ordinary_scope", "plain_english_meaning",
]

require_422 = [
    "feature_name", "feature_family", "context_subfamily", "source_metric",
    "source_metric_label", "metric_value_class", "feature_form",
    "ordinary_availability_class", "pre_cp_post_freeze_class",
    "counterfactual_runtime_source",
    "drop_when_source_metric_equals_target_metric", "missing_value_policy",
    "requires_cross_zone_recursive_state", "uses_actual_post_cp_mobility",
]

missing_421 = sorted(set(require_421) - set(forecast_feature_catalog_421_df.columns))
missing_422 = sorted(set(require_422) - set(forecast_context_catalog_422_df.columns))

if missing_421:
    raise ValueError(f"4.2.1 feature catalog schema changed: {missing_421}")

if missing_422:
    raise ValueError(f"4.2.2 context catalog schema changed: {missing_422}")


runtime_source_421 = {
    "recursive_required": "model-generated no-CP prediction history",
    "deterministic_future_known": "deterministic future-known calendar",
    "frozen_training_fitted": "state frozen from legal Pre-CP fitting",
    "not_used": "not used after the Pre-CP freeze",
}


# ---------------------------------------------------------------------
# Normalize 4.2.1 target-history and calendar features
# ---------------------------------------------------------------------
# 4.2.1 does not define the mobility value-class taxonomy introduced by
# 4.2.2. Preserve that distinction as missing metadata rather than assigning
# an invented class to target-history, seasonal, or calendar definitions.

catalog_421_df = pd.DataFrame({
    "source_notebook": "4.2.1",
    "feature_name": forecast_feature_catalog_421_df["feature_name"],
    "feature_family": forecast_feature_catalog_421_df["feature_family"],
    "feature_subfamily": pd.NA,
    "source_metric": pd.NA,
    "metric_value_class": pd.NA,
    "timing_role": forecast_feature_catalog_421_df["feature_side"],
    "model_role": forecast_feature_catalog_421_df["model_role"],
    "ordinary_availability_class": forecast_feature_catalog_421_df[
        "ordinary_availability_class"
    ],
    "pre_cp_post_freeze_class": forecast_feature_catalog_421_df[
        "pre_cp_post_freeze_class"
    ],
    "counterfactual_runtime_source": forecast_feature_catalog_421_df[
        "pre_cp_post_freeze_class"
    ].map(runtime_source_421),
    "ordinary_scope": forecast_feature_catalog_421_df["ordinary_scope"],
    "drop_when_source_metric_equals_target_metric": False,
    "requires_cross_zone_recursive_state": False,
    "uses_actual_post_cp_mobility": False,
    "missing_value_policy": "Preserve completed 4.2.1 feature contract.",
    "plain_english_meaning": forecast_feature_catalog_421_df["plain_english_meaning"],
})


# ---------------------------------------------------------------------
# Normalize 4.2.2 multimodal, connected, and static context
# ---------------------------------------------------------------------
# `metric_value_class` is part of the upstream semantic contract: it
# distinguishes count/activity from performance measures and therefore
# determines whether missing mobility means zero activity or remains genuinely
# missing for model-stage treatment. Carry it forward without reconstruction.

catalog_422_df = pd.DataFrame({
    "source_notebook": "4.2.2",
    "feature_name": forecast_context_catalog_422_df["feature_name"],
    "feature_family": forecast_context_catalog_422_df["feature_family"],
    "feature_subfamily": forecast_context_catalog_422_df["context_subfamily"],
    "source_metric": forecast_context_catalog_422_df["source_metric"],
    "metric_value_class": forecast_context_catalog_422_df["metric_value_class"],
    "timing_role": np.where(
        forecast_context_catalog_422_df["context_subfamily"].eq("static"),
        "static",
        "origin_context",
    ),
    "model_role": forecast_context_catalog_422_df["feature_form"],
    "ordinary_availability_class": forecast_context_catalog_422_df[
        "ordinary_availability_class"
    ],
    "pre_cp_post_freeze_class": forecast_context_catalog_422_df[
        "pre_cp_post_freeze_class"
    ],
    "counterfactual_runtime_source": forecast_context_catalog_422_df[
        "counterfactual_runtime_source"
    ],
    "ordinary_scope": "all tracks",
    "drop_when_source_metric_equals_target_metric": forecast_context_catalog_422_df[
        "drop_when_source_metric_equals_target_metric"
    ].astype(bool),
    "requires_cross_zone_recursive_state": forecast_context_catalog_422_df[
        "requires_cross_zone_recursive_state"
    ].astype(bool),
    "uses_actual_post_cp_mobility": forecast_context_catalog_422_df[
        "uses_actual_post_cp_mobility"
    ].astype(bool),
    "missing_value_policy": forecast_context_catalog_422_df["missing_value_policy"],
    "plain_english_meaning": forecast_context_catalog_422_df[
        "source_metric_label"
    ].fillna(forecast_context_catalog_422_df["feature_name"]),
})


# ---------------------------------------------------------------------
# Normalize 4.2.3 weather context
# ---------------------------------------------------------------------
# The weather catalog already defines its own value class as continuous, so
# preserve that source-native metadata alongside the 4.2.2 classifications.

catalog_weather_df = pd.DataFrame({
    "source_notebook": "4.2.3",
    "feature_name": forecast_weather_feature_catalog_df["feature_name"],
    "feature_family": "weather",
    "feature_subfamily": forecast_weather_feature_catalog_df["weather_timing_role"],
    "source_metric": forecast_weather_feature_catalog_df["source_metric"],
    "metric_value_class": forecast_weather_feature_catalog_df["metric_value_class"],
    "timing_role": forecast_weather_feature_catalog_df["weather_timing_role"],
    "model_role": "numeric_predictor",
    "ordinary_availability_class": forecast_weather_feature_catalog_df[
        "ordinary_availability_class"
    ],
    "pre_cp_post_freeze_class": forecast_weather_feature_catalog_df[
        "pre_cp_post_freeze_class"
    ],
    "counterfactual_runtime_source": forecast_weather_feature_catalog_df[
        "counterfactual_runtime_source"
    ],
    "ordinary_scope": "all tracks",
    "drop_when_source_metric_equals_target_metric": False,
    "requires_cross_zone_recursive_state": False,
    "uses_actual_post_cp_mobility": False,
    "missing_value_policy": forecast_weather_feature_catalog_df[
        "source_completion_policy"
    ],
    "plain_english_meaning": (
        forecast_weather_feature_catalog_df["source_metric_label"]
        + " · "
        + forecast_weather_feature_catalog_df["weather_timing_role"]
    ),
})


# ---------------------------------------------------------------------
# Assemble and validate the shared 86-definition catalog
# ---------------------------------------------------------------------

forecast_feature_catalog_complete_df = (
    pd.concat([catalog_421_df, catalog_422_df, catalog_weather_df], ignore_index=True)
    .sort_values(["source_notebook", "feature_family", "feature_name"])
    .reset_index(drop=True)
)

if forecast_feature_catalog_complete_df["feature_name"].duplicated().any():
    duplicates = (
        forecast_feature_catalog_complete_df.loc[
            forecast_feature_catalog_complete_df["feature_name"].duplicated(False),
            "feature_name",
        ]
        .unique()
        .tolist()
    )
    raise ValueError(f"Complete feature catalog contains duplicate names: {duplicates}")

EXPECTED_COMPLETE_FEATURE_DEFINITIONS = 37 + 41 + 8

if len(forecast_feature_catalog_complete_df) != EXPECTED_COMPLETE_FEATURE_DEFINITIONS:
    raise AssertionError(
        f"Complete feature universe has {len(forecast_feature_catalog_complete_df):,} "
        f"definitions; expected {EXPECTED_COMPLETE_FEATURE_DEFINITIONS:,}."
    )


# Prove that normalization preserved every 4.2.2 value-class assignment
# exactly. Downstream models may depend on this distinction for missingness
# handling, so a future catalog edit must not silently drop or rewrite it.
source_value_class = (
    forecast_context_catalog_422_df
    .set_index("feature_name")["metric_value_class"]
    .astype("string")
    .sort_index()
)

normalized_value_class = (
    forecast_feature_catalog_complete_df.loc[
        forecast_feature_catalog_complete_df["source_notebook"].eq("4.2.2")
    ]
    .set_index("feature_name")["metric_value_class"]
    .astype("string")
    .sort_index()
)

if not source_value_class.equals(normalized_value_class):
    raise AssertionError(
        "4.2.2 metric_value_class metadata changed during 4.2.3 normalization."
    )


# These booleans summarize the factual availability contract. They do not say
# whether a later model family should ultimately retain the predictor.
forecast_feature_catalog_complete_df["ordinary_forecasting_available"] = True

forecast_feature_catalog_complete_df["frozen_no_cp_structurally_available"] = (
    ~forecast_feature_catalog_complete_df["pre_cp_post_freeze_class"]
    .isin(["ordinary_only", "not_used"])
)

forecast_feature_catalog_complete_df["frozen_no_cp_requires_recursive_state"] = (
    forecast_feature_catalog_complete_df["pre_cp_post_freeze_class"].eq(
        "recursive_required"
    )
)

if WRITE_FINAL_OUTPUTS:
    forecast_feature_catalog_complete_df.to_parquet(
        FINAL_COMBINED_FEATURE_CATALOG_PATH,
        index=False,
    )


feature_family_summary_df = (
    forecast_feature_catalog_complete_df
    .groupby(
        ["source_notebook", "feature_family", "feature_subfamily"],
        observed=True,
        dropna=False,
    )
    .agg(
        feature_definitions=("feature_name", "size"),
        target_conditional_exclusions=(
            "drop_when_source_metric_equals_target_metric",
            "sum",
        ),
        frozen_no_cp_available=("frozen_no_cp_structurally_available", "sum"),
        recursive_required=("frozen_no_cp_requires_recursive_state", "sum"),
    )
    .reset_index()
)

display(feature_family_summary_df)

print(
    "Preserved metric_value_class for all "
    f"{len(source_value_class):,} 4.2.2 feature definitions."
)

,source_notebook,feature_family,feature_subfamily,feature_definitions,target_conditional_exclusions,frozen_no_cp_available,recursive_required
0,4.2.1,Annual calendar cycle,NaN,2,0,2,0
1,4.2.1,Comparable Daypart history,NaN,11,0,11,11
2,4.2.1,Current mobility state,NaN,1,0,1,1
3,4.2.1,Origin calendar,NaN,7,0,7,0
4,4.2.1,Policy-regime context,NaN,2,0,0,0
5,4.2.1,Recent sequence history,NaN,5,0,5,5
6,4.2.1,Target calendar,NaN,7,0,7,0
7,4.2.1,Weekly seasonal state,NaN,2,0,2,0
8,4.2.2,multimodal_origin,auxiliary_source,10,0,0,0
9,4.2.2,multimodal_origin,forecast_target_source,10,10,10,10


Preserved metric_value_class for all 41 4.2.2 feature definitions.


**Findings.** The combined forecasting catalog reconciles exactly to 86 unique shared feature definitions: 37 from the target-history, seasonal, policy, and calendar layer; 41 from multimodal, transportation-connected, and static context; and 8 from weather. For ordinary forecasting, all 86 definitions belong to the shared universe, with the existing target-conditional rule removing the two local multimodal fields derived from the active target and leaving 84 ordinarily eligible definitions for that Forecast Sequence. In the frozen no-CP application, 69 definitions remain structurally available: 32 require recursive mobility state, while deterministic, training-fitted, static, and weather context remain available without recursion. The 15 auxiliary dynamic context definitions and two policy-regime fields are unavailable after the freeze under the current contract.

**Map every feature definition to the later forecasting packages.** The table below distinguishes four downstream modeling uses rather than treating all advanced models as one common tabular consumer.

For **ARIMA/SARIMAX**, target-history features mainly describe information that the autoregressive model may represent natively; engineered history can still be reviewed as optional exogenous structure. Deterministic, dynamic contextual, and weather features are candidates for SARIMAX-style exogenous regressors. Static Taxi Zone attributes are not useful as ordinary regressors inside an independently estimated single-zone time series because they do not vary through time.

For **VARIMA**, the principal input is the joint ordered history of multiple forecast targets as an endogenous vector. The engineered univariate history library therefore describes relevant historical information but is not intended to be copied wholesale into the model as exogenous columns. Other contextual predictors are not part of the default plain-VARIMA package; adding them later would be a separate multivariate-with-exogenous design choice.

Panel-tree and neural-sequence models can structurally consume the broader forecasting context, subject to the same target-equality, information-time, and frozen no-CP restrictions already encoded in the catalogs.

In [22]:
# 4.2.3.5B — Build the proposed forecasting-package feature map

def arima_sarimax_role(row):
    """Classify how one feature definition can participate in univariate statistical forecasting."""
    if row["source_notebook"] == "4.2.2" and row["feature_subfamily"] == "static":
        return "not_default_time_invariant"

    if (
        row["source_notebook"] == "4.2.1"
        and row["ordinary_availability_class"] == "observed_history"
    ):
        return "native_ar_structure_or_optional_engineered_history"

    return "candidate_sarimax_exogenous"


def varima_role(row):
    """Classify feature relevance to the plain multivariate endogenous VARIMA design."""
    if (
        row["source_notebook"] == "4.2.1"
        and row["ordinary_availability_class"] == "observed_history"
    ):
        return "joint_endogenous_history_reference"

    return "not_default_plain_varima"


def neural_role(row):
    """Separate ordered target-history state from contextual neural inputs."""
    if (
        row["source_notebook"] == "4.2.1"
        and row["ordinary_availability_class"] == "observed_history"
    ):
        return "sequence_state_or_optional_engineered_history"

    if row["source_notebook"] == "4.2.2" and row["feature_subfamily"] == "static":
        return "candidate_static_context"

    return "candidate_dynamic_context"


feature_package_review_df = forecast_feature_catalog_complete_df.copy()

feature_package_review_df["arima_sarimax_role"] = feature_package_review_df.apply(
    arima_sarimax_role, axis=1
)
feature_package_review_df["varima_role"] = feature_package_review_df.apply(
    varima_role, axis=1
)

# Panel trees can consume the entire ordinary leakage-safe tabular universe;
# the existing own-target exclusion remains a separate per-target constraint.
feature_package_review_df["panel_tree_role"] = "candidate_predictor"
feature_package_review_df["neural_sequence_role"] = feature_package_review_df.apply(
    neural_role, axis=1
)

# "Applicable" includes information handled natively by a model family as well
# as direct candidate regressors. The detailed role columns explain the distinction.
feature_package_review_df["arima_sarimax_applicable"] = (
    ~feature_package_review_df["arima_sarimax_role"].eq("not_default_time_invariant")
)
feature_package_review_df["varima_applicable"] = (
    feature_package_review_df["varima_role"].eq("joint_endogenous_history_reference")
)
feature_package_review_df["panel_tree_applicable"] = True
feature_package_review_df["neural_sequence_applicable"] = True

if WRITE_INTERMEDIATES:
    feature_package_review_df.to_parquet(FEATURE_PACKAGE_REVIEW_PATH, index=False)

display(
    feature_package_review_df[
        [
            "feature_name", "source_notebook", "feature_family", "feature_subfamily",
            "ordinary_availability_class", "pre_cp_post_freeze_class",
            "drop_when_source_metric_equals_target_metric",
            "arima_sarimax_role", "varima_role",
            "panel_tree_role", "neural_sequence_role",
        ]
    ]
)

,feature_name,source_notebook,feature_family,feature_subfamily,ordinary_availability_class,pre_cp_post_freeze_class,drop_when_source_metric_equals_target_metric,arima_sarimax_role,varima_role,panel_tree_role,neural_sequence_role
0,annual_fourier_cos,4.2.1,Annual calendar cycle,<NA>,deterministic_future_known,deterministic_future_known,False,candidate_sarimax_exogenous,not_default_plain_varima,candidate_predictor,candidate_dynamic_context
1,annual_fourier_sin,4.2.1,Annual calendar cycle,<NA>,deterministic_future_known,deterministic_future_known,False,candidate_sarimax_exogenous,not_default_plain_varima,candidate_predictor,candidate_dynamic_context
2,same_daypart_delta_1d,4.2.1,Comparable Daypart history,<NA>,observed_history,recursive_required,False,native_ar_structure_or_optional_engineered_history,joint_endogenous_history_reference,candidate_predictor,sequence_state_or_optional_engineered_history
3,same_daypart_ewm_mean_7d,4.2.1,Comparable Daypart history,<NA>,observed_history,recursive_required,False,native_ar_structure_or_optional_engineered_history,joint_endogenous_history_reference,candidate_predictor,sequence_state_or_optional_engineered_history
4,same_daypart_lag_1d,4.2.1,Comparable Daypart history,<NA>,observed_history,recursive_required,False,native_ar_structure_or_optional_engineered_history,joint_endogenous_history_reference,candidate_predictor,sequence_state_or_optional_engineered_history
...,...,...,...,...,...,...,...,...,...,...,...
81,weather_origin__wind_speed,4.2.3,weather,origin_observed,origin_observed_exogenous,exogenous_available,False,candidate_sarimax_exogenous,not_default_plain_varima,candidate_predictor,candidate_dynamic_context
82,weather_target_proxy__precipitation,4.2.3,weather,target_period_realized_proxy,target_period_exogenous_proxy,exogenous_available,False,candidate_sarimax_exogenous,not_default_plain_varima,candidate_predictor,candidate_dynamic_context
83,weather_target_proxy__temperature,4.2.3,weather,target_period_realized_proxy,target_period_exogenous_proxy,exogenous_available,False,candidate_sarimax_exogenous,not_default_plain_varima,candidate_predictor,candidate_dynamic_context
84,weather_target_proxy__visibility,4.2.3,weather,target_period_realized_proxy,target_period_exogenous_proxy,exogenous_available,False,candidate_sarimax_exogenous,not_default_plain_varima,candidate_predictor,candidate_dynamic_context


**Findings.** The 86-row package review exposes a deliberately different feature architecture for each downstream forecasting approach. At the shared-catalog level, ARIMA/SARIMAX can draw on 75 definitions, excluding the 11 time-invariant static-geography fields while treating observed target history primarily as autoregressive structure or optional engineered support. Plain VARIMA is intentionally narrower: 17 observed-history definitions describe information represented through the joint endogenous mobility history, while the broader tabular context is not part of the default specification. Panel-tree and neural-sequence models can structurally access the full shared feature universe before target-specific exclusions, with the neural package distinguishing ordered target-history state from dynamic and static contextual inputs.

**Visual checkpoint — complete feature availability and forecasting-package relevance.** Summarize the assembled feature universe by feature family so the reader can see both the legal forecasting contract and the intended downstream use without reading all 86 catalog rows.

Each cell shows the number of feature definitions available or relevant out of the total definitions in that family. The first two columns are factual availability rules. The final four columns summarize the proposed ARIMA/SARIMAX, VARIMA, panel-tree, and neural-sequence package roles.

A high count means the information is structurally usable by that package; it does **not** mean the feature has demonstrated predictive importance or must enter the final tuned model.

In [23]:
# 4.2.3.5C — Render the deferred complete feature-universe matrix

def display_feature_group(row):
    """Keep weather timing roles visible while leaving established upstream family names intact."""
    if row["source_notebook"] != "4.2.3":
        return str(row["feature_family"])

    timing = {
        "origin_observed": "Origin observed",
        "target_period_realized_proxy": "Target weather proxy",
    }.get(row["feature_subfamily"], str(row["feature_subfamily"]))

    return f"Weather · {timing}"


visual_df = feature_package_review_df.copy()
visual_df["display_feature_group"] = visual_df.apply(display_feature_group, axis=1)

dimensions = [
    ("Ordinary forecast", "ordinary_forecasting_available"),
    ("Frozen no-CP", "frozen_no_cp_structurally_available"),
    ("ARIMA / SARIMAX", "arima_sarimax_applicable"),
    ("VARIMA", "varima_applicable"),
    ("Panel tree", "panel_tree_applicable"),
    ("Neural sequence", "neural_sequence_applicable"),
]

matrix_records = []
for group_name, group_df in visual_df.groupby("display_feature_group", observed=True):
    total = len(group_df)

    for dimension_label, flag_col in dimensions:
        count = int(group_df[flag_col].sum())
        matrix_records.append({
            "feature_group": group_name,
            "dimension": dimension_label,
            "count": count,
            "total": total,
            "share": count / total,
            "cell_text": f"{count}/{total}",
        })

matrix_long_df = pd.DataFrame(matrix_records)

family_order = (
    visual_df.groupby("display_feature_group", observed=True)
    .size().sort_values(ascending=False).index.tolist()
)
dimension_order = [label for label, _ in dimensions]

share_matrix = (
    matrix_long_df.pivot(index="feature_group", columns="dimension", values="share")
    .reindex(index=family_order, columns=dimension_order)
)
text_matrix = (
    matrix_long_df.pivot(index="feature_group", columns="dimension", values="cell_text")
    .reindex(index=family_order, columns=dimension_order)
)
count_matrix = (
    matrix_long_df.pivot(index="feature_group", columns="dimension", values="count")
    .reindex(index=family_order, columns=dimension_order)
)
total_matrix = (
    matrix_long_df.pivot(index="feature_group", columns="dimension", values="total")
    .reindex(index=family_order, columns=dimension_order)
)

if RUN_VISUALS:
    # Hover data exposes the exact numerator/denominator while the heatmap shade
    # communicates how much of each family survives each availability/package rule.
    customdata = np.dstack([count_matrix.to_numpy(), total_matrix.to_numpy()])

    fig = go.Figure(go.Heatmap(
        z=share_matrix.to_numpy(),
        x=dimension_order,
        y=family_order,
        zmin=0,
        zmax=1,
        colorscale=[
            [0.00, BRAND_COLORS["pale_peach"]],
            [0.50, BRAND_COLORS["seafoam"]],
            [1.00, BRAND_COLORS["dark_teal"]],
        ],
        text=text_matrix.to_numpy(),
        texttemplate="%{text}",
        customdata=customdata,
        hovertemplate=(
            "<b>%{y}</b><br>%{x}<br>"
            "Relevant / available: %{customdata[0]:.0f}/%{customdata[1]:.0f}<br>"
            "Share: %{z:.3f}<extra></extra>"
        ),
        colorbar=dict(title="Share", tickformat=".0%"),
    ))

    chart_height = min(660, max(430, 90 + 36 * len(family_order)))
    apply_chart_layout(
        fig,
        title="Complete Forecast Feature Availability and Model-Package Relevance",
        height=chart_height,
    )
    fig.update_layout(margin=dict(l=210, r=40, t=80, b=80))
    fig.show()

**Findings.** The feature-universe matrix makes the downstream architecture visible in one view. Ordinary forecasting retains the complete shared catalog, while the frozen no-CP column clearly removes auxiliary post-freeze mobility context and policy-regime fields but preserves recursively reconstructable target history, static geography, deterministic context, and weather. The model-package columns show the intended contrast between the narrower statistical designs and the broader panel-tree and neural designs: ARIMA/SARIMAX excludes time-invariant geography, plain VARIMA centers on joint mobility history, and the panel-tree and neural packages can use the broader leakage-safe context. These cells represent structural availability or relevance, not predictive importance; the two own-target local context fields will still be removed during target-specific matrix assembly.

## 4.2.3.6 Materialize the Historical Forecasting Matrices

The ordinary historical forecasting handoff can now be assembled at the final modeling grain:

> **Modeling Track × Forecast Sequence × Origin observation_sequence_id × Horizon**

Each row combines the legal origin-side state from 4.2.1 and 4.2.2 with the exact horizon-specific target calendar, the resolved weekly seasonal target state, the eight completed weather predictors, and the actual future mobility value from the authoritative 4.1.1 sequence panel.

The future mobility value is attached strictly as a **label**. It never enters the predictor catalog.

The three native modeling-track populations remain independent. Nothing in this section filters Post-CP Primary, Full-History Regime-Aware, or Pre-CP Reference down to the smaller 1,029-sequence cross-regime comparison cohort.

Because the final matrices are large, they are assembled one forecast target at a time and streamed into one atomic Parquet file per modeling track. This keeps memory use bounded without changing the logical matrix grain or schema.

**Load the final assembly contracts and seasonal runtime artifacts.** Bring in only the small supporting tables that are still needed to convert the completed origin surfaces into horizon-specific forecasting rows.

The target label comes from the authoritative 4.1.1 Forecast Sequence panel rather than from any predictor surface. Weekly seasonal target state is resolved from the already-fitted 4.2.1 registries so this notebook does not refit or reinterpret the seasonal model.

In [24]:
# 4.2.3.6A — Load final assembly contracts and normalize target timing

import gc
import pyarrow as pa

WEEKLY_STL_SELECTION_421_PATH = INPUT_421_DIR / "weekly_stl_selection_registry.parquet"
WEEKLY_STL_INTERVAL_421_PATH = INPUT_421_DIR / "weekly_stl_template_interval_registry.parquet"
PRE_CP_WEEKLY_STATE_421_PATH = INPUT_421_DIR / "pre_cp_counterfactual_weekly_state.parquet"
ANNUAL_FOURIER_421_PATH = INPUT_421_DIR / "annual_fourier_calendar.parquet"

for name, path in [
    ("4.2.1 weekly-STL selection registry", WEEKLY_STL_SELECTION_421_PATH),
    ("4.2.1 weekly-STL interval registry", WEEKLY_STL_INTERVAL_421_PATH),
    ("4.2.1 frozen Pre-CP weekly state", PRE_CP_WEEKLY_STATE_421_PATH),
    ("4.2.1 annual Fourier calendar", ANNUAL_FOURIER_421_PATH),
    ("4.2.1 Pre-CP counterfactual calendar", PRE_CP_COUNTERFACTUAL_CALENDAR_421_PATH),
    ("4.2.1 recursive feature contract", PRE_CP_RECURSIVE_CONTRACT_421_PATH),
]:
    require_file(path, name)

weekly_stl_selection_registry_df = pd.read_parquet(WEEKLY_STL_SELECTION_421_PATH)
weekly_stl_template_interval_registry_df = pd.read_parquet(WEEKLY_STL_INTERVAL_421_PATH)
pre_cp_counterfactual_weekly_state_df = pd.read_parquet(PRE_CP_WEEKLY_STATE_421_PATH)
pre_cp_counterfactual_calendar_df = pd.read_parquet(PRE_CP_COUNTERFACTUAL_CALENDAR_421_PATH)
pre_cp_recursive_feature_contract_421_df = pd.read_parquet(PRE_CP_RECURSIVE_CONTRACT_421_PATH)

# The 4.1.1 sequence panel is the authoritative source for the future outcome.
# Only sequence identity and the observed target value are needed here.
forecast_label_lookup_df = pd.read_parquet(
    FORECAST_SEQUENCE_PATH,
    columns=[
        "forecast_sequence_id", "observation_sequence_id", "taxi_zone_id",
        "metric", "observed_value",
    ],
).rename(columns={
    "observation_sequence_id": "target_observation_sequence_id",
    "observed_value": "target_observed_value",
})

label_key = ["forecast_sequence_id", "target_observation_sequence_id"]
if forecast_label_lookup_df.duplicated(label_key).any():
    raise ValueError("The authoritative 4.1.1 target-label lookup is not unique.")

# 4.2.1 intentionally carries a few duplicated x/y calendar fields from its
# alignment audit. Resolve them once to canonical target-side names.
matrix_target_context_df = forecast_target_context_df.copy()
target_alias_candidates = {
    "target_daypart_order": ["target_daypart_order_y", "target_daypart_order_x"],
    "target_temporal_bucket": ["target_temporal_bucket_y", "target_temporal_bucket_x"],
    "target_temporal_bucket_order": [
        "target_temporal_bucket_order_y", "target_temporal_bucket_order_x"
    ],
    "target_pre_post_cp": ["target_pre_post_cp_y", "target_pre_post_cp_x"],
}

for canonical_name, candidates in target_alias_candidates.items():
    if canonical_name in matrix_target_context_df.columns:
        continue
    source_name = next((c for c in candidates if c in matrix_target_context_df.columns), None)
    if source_name is None:
        raise ValueError(f"Could not resolve canonical target field: {canonical_name}")
    matrix_target_context_df[canonical_name] = matrix_target_context_df[source_name]

# Feature names are read from the completed catalogs rather than recreated from
# memory. This keeps the physical matrix schema tied to the persisted contracts.
feature_names_421 = forecast_feature_catalog_421_df["feature_name"].tolist()
feature_names_422 = forecast_context_catalog_422_df["feature_name"].tolist()
all_predictor_names = forecast_feature_catalog_complete_df["feature_name"].tolist()

origin_feature_names_421 = forecast_feature_catalog_421_df.loc[
    forecast_feature_catalog_421_df["feature_side"].eq("origin"), "feature_name"
].tolist()
target_feature_names_421 = forecast_feature_catalog_421_df.loc[
    forecast_feature_catalog_421_df["feature_side"].eq("target"), "feature_name"
].tolist()

weekly_target_features = {"weekly_stl_enabled", "weekly_stl_seasonal_offset"}
target_context_feature_names = [
    f for f in target_feature_names_421 if f not in weekly_target_features
]

missing_target_features = sorted(
    set(target_context_feature_names) - set(matrix_target_context_df.columns)
)
if missing_target_features:
    raise ValueError(
        "4.2.1 target context is missing expected target predictors: "
        f"{missing_target_features}"
    )

if len(all_predictor_names) != 86 or len(set(all_predictor_names)) != 86:
    raise AssertionError("The complete feature catalog must contain 86 unique definitions.")

# Each forecast target must exclude exactly its own local level and Δ1 context.
own_target_context_catalog_df = forecast_context_catalog_422_df.loc[
    forecast_context_catalog_422_df["drop_when_source_metric_equals_target_metric"].astype(bool)
].copy()

own_target_feature_map = {
    metric: group["feature_name"].tolist()
    for metric, group in own_target_context_catalog_df.groupby("source_metric", observed=True)
}
for metric in FORECAST_TARGETS:
    if len(own_target_feature_map.get(metric, [])) != 2:
        raise AssertionError(f"{metric} does not resolve to exactly two own-target exclusions.")

# Preserve the exact Monday→Sunday template convention established in 4.2.1.
WEEKLY_STL_TEMPLATE_OFFSET_COLUMNS = [
    "weekly_stl_offset_mon", "weekly_stl_offset_tue", "weekly_stl_offset_wed",
    "weekly_stl_offset_thu", "weekly_stl_offset_fri", "weekly_stl_offset_sat",
    "weekly_stl_offset_sun",
]

support_contract_df = pd.DataFrame([
    {"component": "4.2.1 predictors", "definitions": len(feature_names_421), "role": "Origin history + deterministic target state"},
    {"component": "4.2.2 predictors", "definitions": len(feature_names_422), "role": "Origin multimodal + connected + static context"},
    {"component": "4.2.3 weather", "definitions": len(WEATHER_FORECAST_FEATURE_COLUMNS), "role": "Origin weather + target-period proxy"},
    {"component": "Complete shared catalog", "definitions": len(all_predictor_names), "role": "Authoritative predictor universe"},
    {"component": "Target label", "definitions": 1, "role": "Future observed mobility outcome; label only"},
])
display(support_contract_df)

,component,definitions,role
0,4.2.1 predictors,37,Origin history + deterministic target state
1,4.2.2 predictors,41,Origin multimodal + connected + static context
2,4.2.3 weather,8,Origin weather + target-period proxy
3,Complete shared catalog,86,Authoritative predictor universe
4,Target label,1,Future observed mobility outcome; label only


**Findings.** All of the pieces needed to build the final forecasting datasets line up correctly. We now have 37 predictors from 4.2.1, 41 from 4.2.2, and 8 weather predictors from this notebook, for 86 shared feature definitions in total. The future mobility value we are trying to predict is kept separate as the target label rather than being mixed into the predictor list. This gives us one clean feature contract that can now be used to build the three final historical forecasting datasets.

**Resolve the fitted weekly seasonal state at each historical forecast target.** Weekly STL is a training-fitted feature, so it does not exist before a modeling track reaches the first information state at which its initial seasonal fit is legally available.

Initial-training rows are still retained in the historical modeling matrix, but enabled single_7 series carry a missing weekly seasonal offset until that first fitted state exists. Once available, the forecast origin selects the latest legal fitted template and the target weekday selects the corresponding Monday–Sunday offset. Series assigned none retain their intentional zero seasonal adjustment throughout.

This preserves the full training population without back-projecting a fitted seasonal state into earlier rows that could not yet have known it.

In [25]:
# 4.2.3.6B — Reuse legally available sequence-specific weekly-STL target state

WEEKLY_TARGET_SELECTION_KEYS = [
    "modeling_track", "forecast_sequence_id", "target_daypart"
]

weekly_selection_lookup_df = (
    weekly_stl_selection_registry_df[
        [
            "modeling_track", "forecast_sequence_id", "daypart",
            "daypart_series_id", "weekly_stl_enabled",
        ]
    ]
    .rename(columns={"daypart": "target_daypart"})
    .copy()
)
weekly_selection_lookup_df["_weekly_selection_present"] = True

# The persisted validity contract is Track × Forecast Sequence × Daypart.
# `daypart_series_id` is descriptive identity, not the interval lookup key.
weekly_interval_groups = {
    key: group.sort_values(
        "weekly_stl_valid_from_observation_sequence_id"
    ).reset_index(drop=True)
    for key, group in weekly_stl_template_interval_registry_df.groupby(
        ["modeling_track", "forecast_sequence_id", "daypart"],
        observed=True,
        sort=False,
    )
}


def attach_weekly_stl_target_state(rows):
    """
    Attach the latest weekly-STL state that legally exists at each forecast origin.

    `single_7` series legitimately have no fitted seasonal state during their
    initial-training history. Those pre-fit rows remain in the modeling matrix
    with a missing offset rather than receiving a back-projected future fit.

    `none` series need no fitted state and always receive their intentional
    0.0 seasonal adjustment.
    """
    result = (
        rows.copy()
        .reset_index(drop=True)
        .merge(
            weekly_selection_lookup_df,
            on=WEEKLY_TARGET_SELECTION_KEYS,
            how="left",
            validate="many_to_one",
        )
    )

    if result["_weekly_selection_present"].isna().any():
        missing = (
            result.loc[
                result["_weekly_selection_present"].isna(),
                WEEKLY_TARGET_SELECTION_KEYS,
            ]
            .drop_duplicates()
            .head(10)
            .to_dict("records")
        )
        raise ValueError(
            "At least one target row did not match the frozen weekly-STL "
            f"selection registry. Examples: {missing}"
        )

    # Provenance stays separate from the two production predictors. It tells
    # downstream code whether a fitted weekly state actually existed at this row.
    result["weekly_stl_refresh_observation_sequence_id"] = pd.Series(
        pd.NA, index=result.index, dtype="Int64"
    )
    result["weekly_stl_refresh_role"] = pd.NA
    result["weekly_stl_valid_from_observation_sequence_id"] = pd.Series(
        pd.NA, index=result.index, dtype="Int64"
    )
    result["weekly_stl_valid_through_observation_sequence_id"] = pd.Series(
        pd.NA, index=result.index, dtype="Int64"
    )
    result["weekly_stl_seasonal_offset"] = np.nan
    result["weekly_stl_state_available"] = False
    result["weekly_stl_state_source"] = "not_yet_fit"

    enabled = result["weekly_stl_enabled"].fillna(False).astype(bool)

    # `none` is itself the fitted seasonal decision, so zero is a real available
    # feature value rather than an imputation.
    result.loc[~enabled, "weekly_stl_seasonal_offset"] = 0.0
    result.loc[~enabled, "weekly_stl_state_available"] = True
    result.loc[~enabled, "weekly_stl_state_source"] = "disabled_zero"

    enabled_groups = result.loc[enabled].groupby(
        ["modeling_track", "forecast_sequence_id", "target_daypart"],
        observed=True,
        sort=False,
    ).groups

    for key, row_index in enabled_groups.items():
        row_index = pd.Index(sorted(row_index))
        intervals = weekly_interval_groups.get(key)

        # A selected `single_7` series should have fitted states somewhere in
        # its legal forecasting period. No intervals at all is a real defect.
        if intervals is None or intervals.empty:
            raise ValueError(
                "An enabled weekly-STL series has no fitted validity intervals "
                f"under Track × Forecast Sequence × Daypart: {key}."
            )

        origin_ids = pd.to_numeric(
            result.loc[row_index, "origin_observation_sequence_id"],
            errors="raise",
        ).to_numpy(dtype=np.int64)

        valid_from = pd.to_numeric(
            intervals["weekly_stl_valid_from_observation_sequence_id"],
            errors="raise",
        ).to_numpy(dtype=np.int64)

        # Negative positions are expected initial-training rows: the first
        # fitted weekly state did not yet legally exist at those origins.
        positions = np.searchsorted(valid_from, origin_ids, side="right") - 1
        legal_mask = positions >= 0

        if not legal_mask.any():
            continue

        legal_index = row_index[legal_mask]
        legal_positions = positions[legal_mask]
        legal_origins = origin_ids[legal_mask]
        chosen = intervals.iloc[legal_positions].reset_index(drop=True)

        valid_through = pd.to_numeric(
            chosen["weekly_stl_valid_through_observation_sequence_id"],
            errors="raise",
        ).to_numpy(dtype=np.int64)

        # Missing a fitted state BEFORE the first interval is expected.
        # Falling beyond the chosen interval is not.
        outside_interval = legal_origins > valid_through
        if outside_interval.any():
            first_bad = int(legal_origins[outside_interval][0])
            raise ValueError(
                f"Weekly-STL origin {first_bad} falls beyond its latest legal "
                f"interval for {key}."
            )

        target_weekdays = pd.to_numeric(
            result.loc[legal_index, "target_weekday_index"],
            errors="raise",
        ).to_numpy(dtype=np.int64)

        offset_matrix = chosen[
            WEEKLY_STL_TEMPLATE_OFFSET_COLUMNS
        ].to_numpy(dtype=float)

        selected_offsets = offset_matrix[
            np.arange(len(chosen)), target_weekdays
        ]

        result.loc[legal_index, "weekly_stl_seasonal_offset"] = selected_offsets
        result.loc[
            legal_index, "weekly_stl_refresh_observation_sequence_id"
        ] = chosen["refresh_observation_sequence_id"].to_numpy()
        result.loc[
            legal_index, "weekly_stl_refresh_role"
        ] = chosen["refresh_role"].to_numpy()
        result.loc[
            legal_index, "weekly_stl_valid_from_observation_sequence_id"
        ] = chosen["weekly_stl_valid_from_observation_sequence_id"].to_numpy()
        result.loc[
            legal_index, "weekly_stl_valid_through_observation_sequence_id"
        ] = chosen["weekly_stl_valid_through_observation_sequence_id"].to_numpy()

        result.loc[legal_index, "weekly_stl_state_available"] = True
        result.loc[legal_index, "weekly_stl_state_source"] = "fitted_interval"

    # Fitted intervals must always produce finite offsets. Missing offsets are
    # legal only for enabled rows whose first fitted state has not yet occurred.
    fitted_rows = enabled & result["weekly_stl_state_available"]
    fitted_offsets = pd.to_numeric(
        result.loc[fitted_rows, "weekly_stl_seasonal_offset"],
        errors="coerce",
    )
    if not np.isfinite(fitted_offsets).all():
        raise ValueError(
            "At least one legally available weekly-STL state did not resolve "
            "to a finite seasonal offset."
        )

    return result.drop(columns="_weekly_selection_present")


# ---------------------------------------------------------------------
# Reader-facing availability audit
# ---------------------------------------------------------------------

# The origin surfaces contain initial-training history as well as later scored
# origins, so beginning before the first fitted STL state is expected. The real
# contract failure would be an origin AFTER the final legal interval.
origin_bounds_df = (
    matrix_target_context_df[
        ["modeling_track", "origin_observation_sequence_id"]
    ]
    .drop_duplicates()
)

interval_bounds_df = (
    weekly_stl_template_interval_registry_df.groupby(
        "modeling_track", as_index=False
    )
    .agg(
        first_legal_weekly_origin=(
            "weekly_stl_valid_from_observation_sequence_id", "min"
        ),
        last_legal_weekly_origin=(
            "weekly_stl_valid_through_observation_sequence_id", "max"
        ),
    )
)

origin_availability_df = origin_bounds_df.merge(
    interval_bounds_df,
    on="modeling_track",
    how="left",
    validate="many_to_one",
)
origin_availability_df["before_first_weekly_state"] = (
    origin_availability_df["origin_observation_sequence_id"]
    < origin_availability_df["first_legal_weekly_origin"]
)
origin_availability_df["after_final_weekly_state"] = (
    origin_availability_df["origin_observation_sequence_id"]
    > origin_availability_df["last_legal_weekly_origin"]
)

weekly_stl_assembly_preflight_df = (
    origin_availability_df.groupby("modeling_track", as_index=False)
    .agg(
        first_matrix_origin=("origin_observation_sequence_id", "min"),
        first_legal_weekly_origin=("first_legal_weekly_origin", "first"),
        pre_fit_origin_count=("before_first_weekly_state", "sum"),
        legal_or_later_origin_count=("before_first_weekly_state", lambda s: int((~s).sum())),
        last_matrix_origin=("origin_observation_sequence_id", "max"),
        last_legal_weekly_origin=("last_legal_weekly_origin", "first"),
        origins_after_final_weekly_state=("after_final_weekly_state", "sum"),
    )
)

# Initial-training rows before the lower bound are expected; origins beyond the
# upper bound would mean the fitted-state registry does not cover the experiment.
if weekly_stl_assembly_preflight_df[
    "origins_after_final_weekly_state"
].gt(0).any():
    raise AssertionError(
        "At least one modeling track extends beyond its final legal weekly-STL state."
    )

weekly_stl_assembly_preflight_df["modeling_track"] = (
    weekly_stl_assembly_preflight_df["modeling_track"].map(TRACK_LABELS)
)

display(weekly_stl_assembly_preflight_df)

,modeling_track,first_matrix_origin,first_legal_weekly_origin,pre_fit_origin_count,legal_or_later_origin_count,last_matrix_origin,last_legal_weekly_origin,origins_after_final_weekly_state
0,Full-History Regime-Aware,1,4121,4120,1809,5929,5930,0
1,Post-CP Primary,3676,4121,445,1809,5929,5930,0
2,Pre-CP Reference,1,1821,1820,1834,3654,5930,0


**Findings.** Some of the earliest training rows occur before a weekly seasonal pattern had been fitted for that modeling track. This affects 4,120 early Full-History origins, 445 Post-CP origins, and 1,820 Pre-CP origins. Those rows are still valid training history, so we keep them and simply leave the weekly seasonal adjustment unavailable until the first legal fit exists. Most importantly, there are no forecast origins after the last valid weekly seasonal fit in any of the three tracks. This means we can use the full historical training data without filling early rows with information learned later and without using a seasonal fit beyond the period where it is valid

**Assemble and stream the historical forecasting matrices.** Process one target metric at a time inside each modeling track, combine the completed feature layers, enforce the target-specific local-context exclusion, attach the future outcome label, and stream the partitions into one atomic Parquet file per track.

The partitioning is an execution strategy only. The exported logical grain remains Modeling Track × Forecast Sequence × Origin × Horizon, and every final matrix contains the same shared 86-definition schema before row-specific structural exclusions and natural source missingness are considered.

In [26]:
# 4.2.3.6C — Build the three native historical forecasting matrices

def append_arrow_partition(writer, schema, df, temp_path):
    """Append one bounded Pandas partition to a stable single-file Parquet schema."""
    table = pa.Table.from_pandas(df, preserve_index=False).replace_schema_metadata(None)

    if writer is None:
        schema = table.schema
        writer = pq.ParquetWriter(temp_path, schema, compression="snappy")
    elif not table.schema.equals(schema):
        try:
            table = table.cast(schema)
        except Exception as exc:
            raise TypeError("A target partition changed the final matrix Arrow schema.") from exc

    writer.write_table(table)
    return writer, schema


def build_historical_matrix_partition(track, target_metric):
    """
    Assemble one Modeling Track × target-metric partition at final horizon grain.

    The function never derives predictors from the future target outcome.
    """
    origin_path = ORIGIN_FEATURE_PATHS_421[track]
    context_path = ORIGIN_CONTEXT_PATHS_422[track]

    # Filter the large surfaces by target before materialization. Both upstream
    # surfaces preserve Forecast Sequence metric identity.
    if "metric" not in parquet_columns(origin_path) or "metric" not in parquet_columns(context_path):
        raise ValueError("Final matrix partitioning requires metric on both origin surfaces.")

    origin = pd.read_parquet(origin_path, filters=[("metric", "==", target_metric)])
    context_add_features = [f for f in feature_names_422 if f not in origin.columns]
    context = pd.read_parquet(
        context_path,
        columns=list(dict.fromkeys([*ORIGIN_KEYS, *context_add_features])),
        filters=[("metric", "==", target_metric)],
    )

    # 4.2.1 and 4.2.2 were built at the same sequence-origin grain; this must
    # remain one-to-one before horizons are introduced.
    base = origin.merge(context, on=ORIGIN_KEYS, how="left", validate="one_to_one")
    base = base.rename(columns={"observation_sequence_id": "origin_observation_sequence_id"})

    schedule_cols = list(dict.fromkeys([
        "modeling_track", "stage", "origin_observation_sequence_id", "horizon",
        "horizon_label", "target_period", "target_observation_sequence_id",
        "target_date", "target_daypart", "target_daypart_order",
        "target_temporal_bucket", "target_temporal_bucket_order",
        "target_pre_post_cp", "target_weekday_index", "target_is_weekend",
        "target_month", "target_day_of_year",
        *target_context_feature_names,
    ]))
    schedule_cols = [c for c in schedule_cols if c in matrix_target_context_df.columns]

    schedule = matrix_target_context_df.loc[
        matrix_target_context_df["modeling_track"].eq(track), schedule_cols
    ].copy()

    # One sequence-origin legitimately expands to h=1, h=2, and h=5 rows.
    matrix = base.merge(
        schedule,
        on=["modeling_track", "origin_observation_sequence_id"],
        how="inner",
        validate="many_to_many",
    )

    matrix = attach_weekly_stl_target_state(matrix)

    # Future observed mobility is attached only after the predictor surface is
    # complete, making its label-only role explicit in both code and schema.
    matrix = matrix.merge(
        forecast_label_lookup_df[[
            "forecast_sequence_id", "target_observation_sequence_id", "target_observed_value"
        ]],
        on=["forecast_sequence_id", "target_observation_sequence_id"],
        how="left",
        validate="many_to_one",
    )
    matrix["target_label_available"] = matrix["target_observed_value"].notna()

    # Weather is normalized at Track × Zone × Origin × Horizon, so every target
    # sequence in one Taxi Zone reuses the same physical weather alignment.
    weather_cols = [
        "modeling_track", "taxi_zone_id", "origin_observation_sequence_id", "horizon",
        *WEATHER_FORECAST_FEATURE_COLUMNS,
        "origin_weather_any_imputed", "target_weather_any_imputed",
    ]
    matrix = matrix.merge(
        forecast_weather_alignment_df[weather_cols],
        on=["modeling_track", "taxi_zone_id", "origin_observation_sequence_id", "horizon"],
        how="left",
        validate="many_to_one",
    )
    matrix["weather_zone_mapped"] = matrix["taxi_zone_id"].isin(weather_zone_ids)

    # The shared 4.2.2 schema is target-neutral. Once the target is known, blank
    # its own local level and Δ1 so they cannot duplicate the richer 4.2.1 history.
    own_features = own_target_feature_map[target_metric]
    matrix.loc[:, own_features] = np.nan
    matrix["own_target_context_exclusion_count"] = len(own_features)
    matrix["shared_feature_definitions_after_target_exclusion"] = len(all_predictor_names) - len(own_features)

    missing_predictors = sorted(set(all_predictor_names) - set(matrix.columns))
    if missing_predictors:
        raise ValueError(
            f"{track} · {target_metric} is missing predictor columns: {missing_predictors}"
        )

    # Fail before writing if the partition itself violates the final row contract.
    if matrix.duplicated(FINAL_MATRIX_KEYS).any():
        raise ValueError(f"{track} · {target_metric} contains duplicate final matrix keys.")

    horizon_ok = (
        matrix["target_observation_sequence_id"]
        .sub(matrix["origin_observation_sequence_id"])
        .eq(matrix["horizon"])
        .all()
    )
    if not horizon_ok:
        raise ValueError(f"{track} · {target_metric} violates target = origin + horizon.")

    if matrix.loc[:, own_features].notna().any().any():
        raise ValueError(f"{track} · {target_metric} did not preserve own-target exclusions.")

    mapped_weather = matrix["weather_zone_mapped"]
    if matrix.loc[mapped_weather, WEATHER_FORECAST_FEATURE_COLUMNS].isna().any().any():
        raise ValueError(f"{track} · {target_metric} lost weather on mapped physical zones.")

    # Keep identity/timing/label fields readable at the front while preserving
    # one physical copy of columns such as borough that also act as predictors.
    metadata_order = [
        "modeling_track", "stage", "forecast_sequence_id", "taxi_zone_id",
        "zone", "borough", "metric",
        "origin_observation_sequence_id", "target_observation_sequence_id",
        "horizon", "horizon_label", "target_period",
        "origin_date", "origin_daypart", "origin_temporal_bucket", "origin_pre_post_cp",
        "target_date", "target_daypart", "target_temporal_bucket", "target_pre_post_cp",
        "target_observed_value", "target_label_available",
    ]
    provenance_order = [
        "origin_role", "native_track_eligible",
        "post_cp_common_comparison_eligible", "cross_regime_comparison_eligible",
        "weekly_stl_refresh_observation_sequence_id", "weekly_stl_refresh_role",
        "weather_zone_mapped", "origin_weather_any_imputed", "target_weather_any_imputed",
        "own_target_context_exclusion_count",
        "shared_feature_definitions_after_target_exclusion",
    ]

    ordered_cols = []
    for col in [*metadata_order, *all_predictor_names, *provenance_order]:
        if col in matrix.columns and col not in ordered_cols:
            ordered_cols.append(col)

    extras = [c for c in matrix.columns if c not in ordered_cols]
    return matrix[ordered_cols + extras]


def summarize_matrix_file(track, path):
    """Read only compact audit columns when a completed matrix is reused."""
    audit = pd.read_parquet(
        path,
        columns=[
            "metric", "forecast_sequence_id", "target_label_available",
            "weather_zone_mapped", "horizon", "stage",
        ],
    )

    return (
        audit.groupby("metric", observed=True)
        .agg(
            rows=("forecast_sequence_id", "size"),
            forecast_sequences=("forecast_sequence_id", "nunique"),
            target_labels_available=("target_label_available", "sum"),
            weather_mapped_rows=("weather_zone_mapped", "sum"),
            horizons=("horizon", "nunique"),
            stages=("stage", "nunique"),
        )
        .reset_index()
        .assign(modeling_track=track)
    )

def finalize_large_parquet(temp_path, output_path):
    """
    Publish a completed large Parquet file on Deepnote-backed storage.

    Deepnote can reject a large-file rename with Errno 5 even after the file
    has been written successfully. Copy the completed temp artifact instead,
    verify its Parquet metadata, and treat temp cleanup as best-effort only.
    """
    shutil.copyfile(temp_path, output_path)

    temp_pf = pq.ParquetFile(temp_path)
    final_pf = pq.ParquetFile(output_path)

    integrity_ok = (
        temp_path.stat().st_size == output_path.stat().st_size
        and temp_pf.metadata.num_rows == final_pf.metadata.num_rows
        and temp_pf.metadata.num_columns == final_pf.metadata.num_columns
        and temp_pf.metadata.num_row_groups == final_pf.metadata.num_row_groups
    )

    if not integrity_ok:
        raise IOError(
            f"Published copy failed integrity validation for {output_path.name}."
        )

    # Cleanup failure leaves only an unnecessary duplicate; it does not
    # invalidate the verified production artifact.
    try:
        temp_path.unlink()
        cleanup_status = "temp_removed"
    except OSError as exc:
        cleanup_status = f"temp_retained_errno_{exc.errno}"
        print(
            f"Warning: {output_path.name} was verified successfully, "
            f"but temp cleanup failed: {exc}"
        )

    return f"verified_copy__{cleanup_status}"

if not WRITE_FINAL_OUTPUTS:
    raise ValueError("Final matrix assembly requires WRITE_FINAL_OUTPUTS=True.")

matrix_build_summary_parts = []

for track in MODELING_TRACKS:
    output_path = FINAL_MATRIX_PATHS[track]
    print(f"\n=== TRACK: {track} ===")

    if output_path.exists() and not REBUILD_FINAL_MATRICES:
        print(f"Reusing existing final matrix: {output_path.name}")
        matrix_build_summary_parts.append(summarize_matrix_file(track, output_path))
        continue

    # Use a unique temp path so an undeletable stale Deepnote artifact can never
    # block a later build.
    temp_path = output_path.with_name(
        f"{output_path.stem}__build_{pd.Timestamp.now():%Y%m%d_%H%M%S}.tmp.parquet"
    )

    writer = None
    writer_schema = None
    track_records = []

    try:
        for i, target_metric in enumerate(FORECAST_TARGETS, start=1):
            print(f"[{track}] target {i}/{len(FORECAST_TARGETS)}: {target_metric} — building...")
            t0 = time.time()

            partition = build_historical_matrix_partition(track, target_metric)

            track_records.append({
                "modeling_track": track,
                "metric": target_metric,
                "rows": len(partition),
                "forecast_sequences": partition["forecast_sequence_id"].nunique(),
                "target_labels_available": int(partition["target_label_available"].sum()),
                "weather_mapped_rows": int(partition["weather_zone_mapped"].sum()),
                "horizons": partition["horizon"].nunique(),
                "stages": partition["stage"].nunique(),
            })

            writer, writer_schema = append_arrow_partition(
                writer, writer_schema, partition, temp_path
            )
            elapsed = time.time() - t0
            print(
                f"[{track}] target {i}/{len(FORECAST_TARGETS)}: {target_metric} — "
                f"done ({len(partition):,} rows, {elapsed/60:.2f} min)"
            )

            del partition
            gc.collect()

    finally:
        if writer is not None:
            writer.close()

    if not temp_path.exists() or temp_path.stat().st_size == 0:
        raise IOError(f"Failed to create final matrix for {track}.")

    print(f"[{track}] finished partition writes; finalizing {output_path.name} ...")
    finalization_method = finalize_large_parquet(temp_path, output_path)
    print(f"[{track}] finalization method: {finalization_method}")

    for record in track_records:
        record["finalization_method"] = finalization_method

    matrix_build_summary_parts.append(pd.DataFrame(track_records))

matrix_build_summary_df = pd.concat(matrix_build_summary_parts, ignore_index=True)
matrix_build_summary_df["target_label_coverage_pct"] = (
    matrix_build_summary_df["target_labels_available"] / matrix_build_summary_df["rows"] * 100
).round(3)
matrix_build_summary_df["weather_mapped_share_pct"] = (
    matrix_build_summary_df["weather_mapped_rows"] / matrix_build_summary_df["rows"] * 100
).round(3)

display(
    matrix_build_summary_df[
        [
            "modeling_track", "metric", "rows", "forecast_sequences",
            "horizons", "stages", "target_label_coverage_pct",
            "weather_mapped_share_pct",
        ]
    ]
)


=== TRACK: post_cp_primary ===
Reusing existing final matrix: forecasting_matrix__post_cp_primary.parquet

=== TRACK: full_history_regime_aware ===
Reusing existing final matrix: forecasting_matrix__full_history_regime_aware.parquet

=== TRACK: pre_cp_reference ===
Reusing existing final matrix: forecasting_matrix__pre_cp_reference.parquet


,modeling_track,metric,rows,forecast_sequences,horizons,stages,target_label_coverage_pct,weather_mapped_share_pct
0,post_cp_primary,fhvhv_avg_trip_speed,1729792,256,3,3,99.845,99.219
1,post_cp_primary,fhvhv_trip_count,1736549,257,3,3,99.871,98.833
2,post_cp_primary,subway_ridership,1006793,149,3,3,99.535,100.000
3,post_cp_primary,taxi_avg_trip_speed,1277073,189,3,3,97.585,98.942
4,post_cp_primary,taxi_trip_count,1736549,257,3,3,99.871,98.833
5,full_history_regime_aware,fhvhv_avg_trip_speed,4569974,257,3,3,99.770,99.222
6,full_history_regime_aware,fhvhv_trip_count,4587756,258,3,3,99.791,98.837
7,full_history_regime_aware,subway_ridership,2756210,155,3,3,98.889,100.000
8,full_history_regime_aware,taxi_avg_trip_speed,2187186,123,3,3,97.196,98.374
9,full_history_regime_aware,taxi_trip_count,4587756,258,3,3,99.791,98.837


**Findings.** The three final historical forecasting datasets were built successfully. Post-CP Primary contains 7,486,756 rows, Full-History Regime-Aware contains 18,688,882 rows, and Pre-CP Reference contains 11,384,323 rows. All five mobility targets are represented at the 1-, 2-, and 5-step forecast horizons. Most rows have an observed future value available for model training and evaluation, with coverage ranging from about 97.2% to 99.9% depending on the target and track; rows without an observed target are kept and explicitly marked rather than silently dropped. The Post-CP file was reused from the already validated recovery, while the other two were built successfully. The temporary-file cleanup warnings are only a Deepnote storage issue and do not affect the completed forecasting files.

**Validate the three historical forecasting matrices.** Re-open only the columns needed for final contract QA and independently confirm the exported files rather than trusting the in-memory build.

The checks protect the native sequence populations, exact horizon arithmetic, rolling schedule membership, target-specific local-context exclusion, mapped-zone weather completeness, and the separation between predictor columns and the future outcome label.

Natural missingness in performance context is allowed to remain. Unmapped weather zones also remain missing rather than receiving fabricated spatial weather.

In [27]:
# 4.2.3.6D — Validate the exported historical forecasting matrices

ordinary_matrix_qa_rows = []
schedule_compare_cols = [
    "modeling_track", "stage", "origin_observation_sequence_id", "horizon",
    "target_observation_sequence_id", "target_date", "target_daypart",
]
own_target_context_features = own_target_context_catalog_df["feature_name"].tolist()

for track in MODELING_TRACKS:
    path = FINAL_MATRIX_PATHS[track]
    require_file(path, f"{TRACK_LABELS[track]} final forecasting matrix")

    schema_names = set(parquet_columns(path))
    required_schema = set(FINAL_MATRIX_KEYS + [
        "taxi_zone_id", "metric", "stage", "target_observation_sequence_id",
        "target_date", "target_daypart", "target_observed_value",
        "target_label_available", "weather_zone_mapped",
        *all_predictor_names,
    ])
    missing_schema = sorted(required_schema - schema_names)

    qa_cols = list(dict.fromkeys([
        *FINAL_MATRIX_KEYS, "taxi_zone_id", "metric", "stage",
        "target_observation_sequence_id", "target_date", "target_daypart",
        "weather_zone_mapped", *own_target_context_features,
        *WEATHER_FORECAST_FEATURE_COLUMNS,
    ]))
    audit = pd.read_parquet(path, columns=qa_cols)

    native_sequence_count = pd.read_parquet(
        ORIGIN_FEATURE_PATHS_421[track], columns=["forecast_sequence_id"]
    )["forecast_sequence_id"].nunique()

    actual_sequence_count = audit["forecast_sequence_id"].nunique()
    duplicate_rows = int(audit.duplicated(FINAL_MATRIX_KEYS).sum())
    horizon_ok = (
        audit["target_observation_sequence_id"]
        .sub(audit["origin_observation_sequence_id"])
        .eq(audit["horizon"])
        .all()
    )

    expected_schedule = (
        matrix_target_context_df.loc[
            matrix_target_context_df["modeling_track"].eq(track), schedule_compare_cols
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    actual_schedule = audit[schedule_compare_cols].drop_duplicates().reset_index(drop=True)

    schedule_compare = expected_schedule.merge(
        actual_schedule, on=schedule_compare_cols, how="outer", indicator=True
    )
    schedule_ok = schedule_compare["_merge"].eq("both").all()

    own_exclusion_ok = True
    for metric, own_features in own_target_feature_map.items():
        metric_rows = audit["metric"].eq(metric)
        if metric_rows.any() and audit.loc[metric_rows, own_features].notna().any().any():
            own_exclusion_ok = False
            break

    mapped_rows = audit["weather_zone_mapped"].fillna(False).astype(bool)
    mapped_weather_ok = (
        not mapped_rows.any()
        or audit.loc[mapped_rows, WEATHER_FORECAST_FEATURE_COLUMNS].notna().all().all()
    )

    track_label = TRACK_LABELS[track]
    ordinary_matrix_qa_rows.extend([
        qa_row(
            f"{track_label}: final matrix schema contains the complete predictor contract",
            not missing_schema,
            "all 86 shared feature definitions present" if not missing_schema
            else f"missing columns = {missing_schema}",
        ),
        qa_row(
            f"{track_label}: final row key is unique",
            duplicate_rows == 0,
            f"duplicate final keys = {duplicate_rows:,}",
        ),
        qa_row(
            f"{track_label}: target sequence equals origin + horizon",
            horizon_ok,
            f"rows checked = {len(audit):,}",
        ),
        qa_row(
            f"{track_label}: rolling-origin schedule is preserved exactly",
            schedule_ok,
            f"scheduled rows = {len(expected_schedule):,}; represented = {len(actual_schedule):,}",
        ),
        qa_row(
            f"{track_label}: native Forecast Sequence population is preserved",
            actual_sequence_count == native_sequence_count,
            f"matrix = {actual_sequence_count:,}; native = {native_sequence_count:,}",
        ),
        qa_row(
            f"{track_label}: matrix was not restricted to the 1,029 cross-regime cohort",
            actual_sequence_count > 1_029,
            f"native sequences retained = {actual_sequence_count:,}",
        ),
        qa_row(
            f"{track_label}: own-target local context remains structurally excluded",
            own_exclusion_ok,
            "two local 4.2.2 definitions blanked for each active target",
        ),
        qa_row(
            f"{track_label}: mapped-zone weather remains complete",
            mapped_weather_ok,
            f"mapped matrix rows = {int(mapped_rows.sum()):,}",
        ),
    ])

    del audit, expected_schedule, actual_schedule, schedule_compare
    gc.collect()

ordinary_matrix_qa_rows.extend([
    qa_row(
        "Future target outcome is not part of the predictor catalog",
        "target_observed_value" not in all_predictor_names,
        "target_observed_value is retained only as the modeling label",
    ),
    qa_row(
        "Complete predictor catalog remains unique",
        len(all_predictor_names) == len(set(all_predictor_names)) == 86,
        "86 unique shared feature definitions",
    ),
])

ordinary_matrix_qa_df = pd.DataFrame(ordinary_matrix_qa_rows)
display(ordinary_matrix_qa_df)

if not ordinary_matrix_qa_df["status"].eq("PASS").all():
    raise AssertionError("Historical forecasting matrix QA failed.")

,check,status,detail
0,Post-CP Primary: final matrix schema contains the complete predictor contract,PASS,all 86 shared feature definitions present
1,Post-CP Primary: final row key is unique,PASS,duplicate final keys = 0
2,Post-CP Primary: target sequence equals origin + horizon,PASS,"rows checked = 7,486,756"
3,Post-CP Primary: rolling-origin schedule is preserved exactly,PASS,"scheduled rows = 6,757; represented = 6,757"
4,Post-CP Primary: native Forecast Sequence population is preserved,PASS,"matrix = 1,108; native = 1,108"
5,"Post-CP Primary: matrix was not restricted to the 1,029 cross-regime cohort",PASS,"native sequences retained = 1,108"
6,Post-CP Primary: own-target local context remains structurally excluded,PASS,two local 4.2.2 definitions blanked for each active target
7,Post-CP Primary: mapped-zone weather remains complete,PASS,"mapped matrix rows = 7,419,186"
8,Full-History Regime-Aware: final matrix schema contains the complete predictor contract,PASS,all 86 shared feature definitions present
9,Full-History Regime-Aware: final row key is unique,PASS,duplicate final keys = 0


**Findings.** All 26 checks on the final historical forecasting datasets pass. There are no duplicate forecast rows, every 1-, 2-, and 5-step target points to the correct future time period, and each modeling track keeps its full intended population: 1,108 Post-CP sequences, 1,051 Full-History sequences, and 1,039 Pre-CP sequences. The two redundant same-target context features from 4.2.2 are correctly left out, weather is complete for zones with weather coverage, and the future mobility value remains only the outcome being predicted—not a predictor. The three historical datasets are therefore ready for downstream forecasting work.

**4.2.3.6E — Materialize the final-refit-only Pre-CP training supplement.**

The Pre-CP Reference uses two supervised training surfaces with different roles. The historical matrix contains the **2023 initial-training and 2024 model-selection experiment**. A separate final-refit supplement contains the additional observed Pre-CP targets from **January 1–4, 2025**, which become available only after family selection is complete.

The supplement follows the same physical forecasting-matrix contract as the historical Pre-CP matrix: the same Forecast Sequences, h=1/h=2/h=5 timing, 4.2.1 historical and seasonal state, 4.2.2 multimodal and spatial context, target-specific exclusions, weather treatment, and observed future value as a training label.

This separation keeps the model-selection population fixed through December 31 while providing the complete observed Pre-CP history required for the final production refit through January 4.

In [28]:
# 4.2.3.6E — Build the final-refit-only Pre-CP training supplement

refit_contract = pre_cp_refit_contract_df.iloc[0]

refit_start_date = pd.Timestamp(
    refit_contract["refit_only_start_date"]
).normalize()

refit_end_date = pd.Timestamp(
    refit_contract["final_refit_end_date"]
).normalize()

refit_end_sequence_id = int(
    refit_contract["final_refit_end_observation_sequence_id"]
)

expected_native_sequences = int(
    refit_contract["native_pre_cp_sequence_count"]
)


# ---------------------------------------------------------------------
# Build only the Jan. 1–4 Target × Horizon schedule
# ---------------------------------------------------------------------
# Membership follows TARGET date. Each horizon therefore receives the same
# 20 observed Pre-CP target positions, but starts from the appropriate origin.

refit_schedule_df = forecast_origin_horizon_df.loc[
    forecast_origin_horizon_df["target_date"].between(
        refit_start_date,
        refit_end_date,
        inclusive="both",
    )
].copy()

refit_schedule_df = refit_schedule_df.rename(columns={
    "observation_sequence_id": "origin_observation_sequence_id",
    "date": "origin_date",
    "daypart": "origin_daypart",
    "daypart_order": "origin_daypart_order",
    "temporal_bucket": "origin_temporal_bucket",
    "temporal_bucket_order": "origin_temporal_bucket_order",
    "pre_post_cp": "origin_pre_post_cp",
})

refit_schedule_df.insert(0, "modeling_track", "pre_cp_reference")
refit_schedule_df.insert(1, "stage", "final_refit_only")


# ---------------------------------------------------------------------
# Recreate the deterministic target-side fields normally supplied by 4.2.1
# ---------------------------------------------------------------------
# These depend only on the target calendar. No future mobility outcome enters
# this feature construction.

refit_schedule_df["target_weekday_index"] = (
    pd.to_datetime(refit_schedule_df["target_date"])
    .dt.weekday
    .astype("int8")
)

refit_schedule_df["target_is_weekend"] = (
    refit_schedule_df["target_weekday_index"]
    .ge(5)
    .astype("int8")
)

refit_schedule_df["target_month"] = (
    pd.to_datetime(refit_schedule_df["target_date"])
    .dt.month
    .astype("int8")
)

refit_schedule_df["target_day_of_year"] = (
    pd.to_datetime(refit_schedule_df["target_date"])
    .dt.dayofyear
    .astype("int16")
)

annual_fourier_calendar_df = pd.read_parquet(
    ANNUAL_FOURIER_421_PATH,
    columns=[
        "observation_sequence_id",
        "annual_fourier_sin",
        "annual_fourier_cos",
    ],
)

refit_schedule_df = refit_schedule_df.merge(
    annual_fourier_calendar_df.rename(columns={
        "observation_sequence_id": "target_observation_sequence_id"
    }),
    on="target_observation_sequence_id",
    how="left",
    validate="many_to_one",
)

refit_schedule_cols = list(dict.fromkeys([
    "modeling_track", "stage",
    "origin_observation_sequence_id",
    "horizon", "horizon_label", "target_period",
    "target_observation_sequence_id",
    "target_date", "target_daypart", "target_daypart_order",
    "target_temporal_bucket", "target_temporal_bucket_order",
    "target_pre_post_cp",
    "target_weekday_index", "target_is_weekend",
    "target_month", "target_day_of_year",
    *target_context_feature_names,
]))

refit_schedule_cols = [
    column
    for column in refit_schedule_cols
    if column in refit_schedule_df.columns
]

refit_schedule_df = refit_schedule_df[
    refit_schedule_cols
].copy()

expected_schedule_rows = 20 * len(FORECAST_HORIZONS)

if len(refit_schedule_df) != expected_schedule_rows:
    raise RuntimeError(
        "Final-refit schedule must contain 20 targets for each of "
        "h=1, h=2, and h=5: "
        f"{len(refit_schedule_df):,} vs {expected_schedule_rows:,}."
    )

if set(refit_schedule_df["horizon"].astype(int)) != set(FORECAST_HORIZONS):
    raise RuntimeError(
        "Final-refit schedule does not contain exactly h=1, h=2, and h=5."
    )

if (
    int(refit_schedule_df["target_observation_sequence_id"].max())
    != refit_end_sequence_id
):
    raise RuntimeError(
        "Final-refit schedule does not reach the January 4 freeze sequence."
    )


# ---------------------------------------------------------------------
# Build weather only for the 60 new Origin × Horizon positions
# ---------------------------------------------------------------------
# Reuse the completed source-weather table. Do not rebuild the existing
# 9.2-million-row ordinary weather alignment just for this four-day supplement.

physical_zone_df = pd.DataFrame({
    "taxi_zone_id": sorted(
        int(zone_id)
        for zone_id in weather_zone_ids
    )
})

weather_join_cols = [
    "taxi_zone_id",
    "observation_sequence_id",
    *WEATHER_RETAINED_METRICS,
    "weather_any_imputed",
]

origin_weather_df = completed_weather_context_df[
    weather_join_cols
].rename(columns={
    **{
        metric: f"weather_origin__{metric}"
        for metric in WEATHER_RETAINED_METRICS
    },
    "observation_sequence_id": "origin_observation_sequence_id",
    "weather_any_imputed": "origin_weather_any_imputed",
})

target_weather_df = completed_weather_context_df[
    weather_join_cols
].rename(columns={
    **{
        metric: f"weather_target_proxy__{metric}"
        for metric in WEATHER_RETAINED_METRICS
    },
    "observation_sequence_id": "target_observation_sequence_id",
    "weather_any_imputed": "target_weather_any_imputed",
})

refit_weather_alignment_df = (
    refit_schedule_df[
        [
            "modeling_track", "stage",
            "origin_observation_sequence_id",
            "horizon",
            "target_observation_sequence_id",
            "target_date", "target_daypart",
        ]
    ]
    .merge(physical_zone_df, how="cross")
    .merge(
        origin_weather_df,
        on=["taxi_zone_id", "origin_observation_sequence_id"],
        how="left",
        validate="many_to_one",
    )
    .merge(
        target_weather_df,
        on=["taxi_zone_id", "target_observation_sequence_id"],
        how="left",
        validate="many_to_one",
    )
)

if refit_weather_alignment_df[
    WEATHER_FORECAST_FEATURE_COLUMNS
].isna().any().any():
    raise RuntimeError(
        "Final-refit weather alignment contains missing retained predictors."
    )


# ---------------------------------------------------------------------
# Freeze the historical physical schema before building the supplement
# ---------------------------------------------------------------------

historical_pre_cp_path = FINAL_MATRIX_PATHS["pre_cp_reference"]

historical_pre_cp_schema = pq.ParquetFile(
    historical_pre_cp_path
).schema_arrow

historical_pre_cp_columns = historical_pre_cp_schema.names


def build_pre_cp_refit_partition(target_metric):
    """Build one target's Jan. 1–4 rows using the historical matrix contract."""
    origin_path = ORIGIN_FEATURE_PATHS_421["pre_cp_reference"]
    context_path = ORIGIN_CONTEXT_PATHS_422["pre_cp_reference"]

    min_origin = int(
        refit_schedule_df["origin_observation_sequence_id"].min()
    )
    max_origin = int(
        refit_schedule_df["origin_observation_sequence_id"].max()
    )

    filters = [
        ("metric", "==", target_metric),
        ("observation_sequence_id", ">=", min_origin),
        ("observation_sequence_id", "<=", max_origin),
    ]

    origin = pd.read_parquet(
        origin_path,
        filters=filters,
    )

    context_add_features = [
        name
        for name in feature_names_422
        if name not in origin.columns
    ]

    context = pd.read_parquet(
        context_path,
        columns=list(dict.fromkeys([
            *ORIGIN_KEYS,
            *context_add_features,
        ])),
        filters=filters,
    )

    base = origin.merge(
        context,
        on=ORIGIN_KEYS,
        how="left",
        validate="one_to_one",
    )

    base = base.rename(columns={
        "observation_sequence_id": "origin_observation_sequence_id"
    })

    matrix = base.merge(
        refit_schedule_df,
        on=["modeling_track", "origin_observation_sequence_id"],
        how="inner",
        validate="many_to_many",
    )

    # Use the latest seasonal state legally available at each origin.
    matrix = attach_weekly_stl_target_state(matrix)

    # Attach the observed Jan. 1–4 outcome only after predictors are complete.
    matrix = matrix.merge(
        forecast_label_lookup_df[
            [
                "forecast_sequence_id",
                "target_observation_sequence_id",
                "target_observed_value",
            ]
        ],
        on=[
            "forecast_sequence_id",
            "target_observation_sequence_id",
        ],
        how="left",
        validate="many_to_one",
    )

    matrix["target_label_available"] = (
        matrix["target_observed_value"].notna()
    )

    weather_cols = [
        "modeling_track",
        "taxi_zone_id",
        "origin_observation_sequence_id",
        "horizon",
        *WEATHER_FORECAST_FEATURE_COLUMNS,
        "origin_weather_any_imputed",
        "target_weather_any_imputed",
    ]

    matrix = matrix.merge(
        refit_weather_alignment_df[weather_cols],
        on=[
            "modeling_track",
            "taxi_zone_id",
            "origin_observation_sequence_id",
            "horizon",
        ],
        how="left",
        validate="many_to_one",
    )

    matrix["weather_zone_mapped"] = (
        matrix["taxi_zone_id"].isin(weather_zone_ids)
    )

    # Preserve the same target-specific exclusion used in the historical matrix.
    own_features = own_target_feature_map[target_metric]
    matrix.loc[:, own_features] = np.nan

    matrix["own_target_context_exclusion_count"] = len(
        own_features
    )

    matrix["shared_feature_definitions_after_target_exclusion"] = (
        len(all_predictor_names) - len(own_features)
    )

    missing_columns = sorted(
        set(historical_pre_cp_columns) - set(matrix.columns)
    )

    if missing_columns:
        raise RuntimeError(
            f"{target_metric}: final-refit partition is missing "
            f"historical-schema columns: {missing_columns}"
        )

    partition = matrix[
        historical_pre_cp_columns
    ].copy()

    if partition.duplicated(FINAL_MATRIX_KEYS).any():
        raise RuntimeError(
            f"{target_metric}: final-refit partition contains duplicate keys."
        )

    # Label availability must faithfully describe the source observation.
    # Genuine missing mobility outcomes remain missing, exactly as they do in
    # the historical forecasting matrices.
    label_contract_ok = (
        partition["target_label_available"]
        .astype(bool)
        .eq(partition["target_observed_value"].notna())
        .all()
    )

    if not label_contract_ok:
        raise RuntimeError(
            f"{target_metric}: target-label availability does not match "
            "the observed target values."
        )

    return partition


# ---------------------------------------------------------------------
# Build once; reuse on ordinary notebook reruns
# ---------------------------------------------------------------------

if (
    PRE_CP_FINAL_REFIT_SUPPLEMENT_PATH.exists()
    and not REBUILD_PRE_CP_REFIT_SUPPLEMENT
):
    pre_cp_final_refit_supplement_df = pd.read_parquet(
        PRE_CP_FINAL_REFIT_SUPPLEMENT_PATH
    )
    supplement_mode = "REUSE"

else:
    supplement_parts = []

    for index, target_metric in enumerate(
        FORECAST_TARGETS,
        start=1,
    ):
        print(
            f"Final-refit supplement "
            f"{index}/{len(FORECAST_TARGETS)} · {target_metric}",
            flush=True,
        )

        supplement_parts.append(
            build_pre_cp_refit_partition(target_metric)
        )

    pre_cp_final_refit_supplement_df = pd.concat(
        supplement_parts,
        ignore_index=True,
    )

    # Cast to the already-persisted historical schema so Chapter 5 can append
    # the files without target-specific dtype surprises.
    expected_refit_rows = (
        expected_native_sequences
        * len(refit_schedule_df)
    )

    # Validate the complete in-memory surface BEFORE publishing it. A failed
    # build must never leave behind an artifact that a later rerun can reuse.
    if len(pre_cp_final_refit_supplement_df) != expected_refit_rows:
        rows_per_sequence = (
            len(pre_cp_final_refit_supplement_df)
            / expected_native_sequences
        )

        raise RuntimeError(
            "Final-refit supplement row count changed before publication: "
            f"{len(pre_cp_final_refit_supplement_df):,} "
            f"vs {expected_refit_rows:,} expected "
            f"({rows_per_sequence:.3f} rows per Forecast Sequence)."
        )

    supplement_table = pa.Table.from_pandas(
        pre_cp_final_refit_supplement_df,
        preserve_index=False,
    ).replace_schema_metadata(None)

    supplement_table = supplement_table.cast(
        historical_pre_cp_schema
    )

    temp_path = PRE_CP_FINAL_REFIT_SUPPLEMENT_PATH.with_name(
        PRE_CP_FINAL_REFIT_SUPPLEMENT_PATH.stem
        + "__build.tmp.parquet"
    )

    temp_path.unlink(missing_ok=True)

    pq.write_table(
        supplement_table,
        temp_path,
        compression="snappy",
    )

    finalize_large_parquet(
        temp_path,
        PRE_CP_FINAL_REFIT_SUPPLEMENT_PATH,
    )

    del supplement_parts, supplement_table
    gc.collect()

    supplement_mode = "BUILD"


expected_refit_rows = (
    expected_native_sequences
    * len(refit_schedule_df)
)

if len(pre_cp_final_refit_supplement_df) != expected_refit_rows:
    raise RuntimeError(
        "Persisted final-refit supplement does not match the current contract: "
        f"{len(pre_cp_final_refit_supplement_df):,} "
        f"vs {expected_refit_rows:,} expected."
    )


supplement_summary_df = pd.DataFrame([{
    "Mode": supplement_mode,
    "Rows": len(pre_cp_final_refit_supplement_df),
    "Forecast Sequences":
        pre_cp_final_refit_supplement_df["forecast_sequence_id"].nunique(),
    "Observed labels":
        int(pre_cp_final_refit_supplement_df["target_label_available"].sum()),
    "Label coverage (%)":
        100.0
        * float(pre_cp_final_refit_supplement_df["target_label_available"].mean()),
    "Targets":
        pre_cp_final_refit_supplement_df[
            "target_observation_sequence_id"
        ].nunique(),
    "First target":
        pd.to_datetime(
            pre_cp_final_refit_supplement_df["target_date"]
        ).min(),
    "Last target":
        pd.to_datetime(
            pre_cp_final_refit_supplement_df["target_date"]
        ).max(),
}])

display(supplement_summary_df)

Final-refit supplement 1/5 · taxi_trip_count
Final-refit supplement 2/5 · taxi_avg_trip_speed
Final-refit supplement 3/5 · fhvhv_trip_count
Final-refit supplement 4/5 · fhvhv_avg_trip_speed
Final-refit supplement 5/5 · subway_ridership


,Mode,Rows,Forecast Sequences,Observed labels,Label coverage (%),Targets,First target,Last target
0,BUILD,62340,1039,61866,99.240,20,2025-01-01,2025-01-04


**Findings.** The final-refit supplement contains **62,340 forecast rows** covering all **1,039 native Pre-CP Forecast Sequences**, all three horizons, and exactly the **20 canonical target positions from January 1–4, 2025**. It uses the same forecasting-matrix schema and feature semantics as the historical Pre-CP training surface, including preservation of genuine source missingness through `target_label_available`.

Rows with an observed target can contribute to the final production refit; rows whose source mobility outcome is unavailable remain in the forecasting surface but are not treated as training examples. Together, the historical matrix and supplement provide the complete legal observed-history window through January 4.

**4.2.3.6F — Validate the final-refit-only Pre-CP training contract.**

The final-refit supplement must satisfy the same structural and leakage-safe rules as the historical Pre-CP matrix while occupying a distinct target window.

This validation checks the physical schema, native Forecast Sequence population, h=1/h=2/h=5 target arithmetic, January 1–4 date boundary, observed training labels, target-specific context exclusions, and weather completeness. It also verifies that the historical model-selection matrix and final-refit supplement have no overlapping forecast rows.

A passing supplement therefore represents the complete supervised Pre-CP extension from the end of model selection through the final legal observed state at **January 4, 2025 / observation sequence 3675**.

In [29]:
# 4.2.3.6F — Validate the final-refit-only Pre-CP supplement

supplement_schema = pq.ParquetFile(
    PRE_CP_FINAL_REFIT_SUPPLEMENT_PATH
).schema_arrow

schema_matches_history = supplement_schema.equals(
    historical_pre_cp_schema,
    check_metadata=False,
)

refit_target_ids = set(
    refit_schedule_df[
        "target_observation_sequence_id"
    ].astype(int)
)

refit_horizons = set(
    pre_cp_final_refit_supplement_df[
        "horizon"
    ].astype(int)
)


# The existing selection matrix should have no rows at or beyond the first
# refit-only target. Filter at the boundary rather than loading all 11.4M keys.
historical_overlap_df = pd.read_parquet(
    historical_pre_cp_path,
    columns=[
        *FINAL_MATRIX_KEYS,
        "target_observation_sequence_id",
    ],
    filters=[
        (
            "target_observation_sequence_id",
            ">=",
            min(refit_target_ids),
        ),
    ],
)


own_target_exclusions_ok = True

for metric, own_features in own_target_feature_map.items():
    metric_rows = (
        pre_cp_final_refit_supplement_df["metric"].eq(metric)
    )

    if (
        metric_rows.any()
        and pre_cp_final_refit_supplement_df.loc[
            metric_rows,
            own_features,
        ].notna().any().any()
    ):
        own_target_exclusions_ok = False
        break


mapped_rows = (
    pre_cp_final_refit_supplement_df[
        "weather_zone_mapped"
    ]
    .fillna(False)
    .astype(bool)
)

mapped_weather_complete = (
    not mapped_rows.any()
    or pre_cp_final_refit_supplement_df.loc[
        mapped_rows,
        WEATHER_FORECAST_FEATURE_COLUMNS,
    ].notna().all().all()
)


pre_cp_refit_supplement_qa_df = pd.DataFrame([
    qa_row(
        "Final-refit supplement matches the historical Pre-CP matrix schema",
        schema_matches_history,
        f"columns = {len(supplement_schema.names)}",
    ),
    qa_row(
        "Final-refit supplement contains the expected derived row count",
        len(pre_cp_final_refit_supplement_df) == expected_refit_rows,
        (
            f"rows = {len(pre_cp_final_refit_supplement_df):,}; "
            f"expected = {expected_refit_rows:,}"
        ),
    ),
    qa_row(
        "Final-refit supplement preserves all native Pre-CP Forecast Sequences",
        (
            pre_cp_final_refit_supplement_df[
                "forecast_sequence_id"
            ].nunique()
            == expected_native_sequences
        ),
        (
            "sequences = "
            f"{pre_cp_final_refit_supplement_df['forecast_sequence_id'].nunique():,}; "
            f"expected = {expected_native_sequences:,}"
        ),
    ),
    qa_row(
        "Final-refit supplement is isolated to the Pre-CP final_refit_only stage",
        (
            pre_cp_final_refit_supplement_df[
                "modeling_track"
            ].eq("pre_cp_reference").all()
            and pre_cp_final_refit_supplement_df[
                "stage"
            ].eq("final_refit_only").all()
        ),
        "track = pre_cp_reference; stage = final_refit_only",
    ),
    qa_row(
        "Final-refit supplement contains exactly h=1, h=2, and h=5",
        refit_horizons == set(FORECAST_HORIZONS),
        f"horizons = {sorted(refit_horizons)}",
    ),
    qa_row(
        "Every final-refit target sequence equals origin + horizon",
        (
            pre_cp_final_refit_supplement_df[
                "target_observation_sequence_id"
            ]
            .sub(
                pre_cp_final_refit_supplement_df[
                    "origin_observation_sequence_id"
                ]
            )
            .eq(
                pre_cp_final_refit_supplement_df["horizon"]
            )
            .all()
        ),
        f"rows checked = {len(pre_cp_final_refit_supplement_df):,}",
    ),
    qa_row(
        "Final-refit targets are exactly the January 1–4 target positions",
        (
            set(
                pre_cp_final_refit_supplement_df[
                    "target_observation_sequence_id"
                ].astype(int)
            )
            == refit_target_ids
        ),
        (
            f"target IDs = "
            f"{min(refit_target_ids)}–{max(refit_target_ids)}"
        ),
    ),
    qa_row(
        "Final-refit target dates are confined to January 1–4, 2025",
        (
            pd.to_datetime(
                pre_cp_final_refit_supplement_df["target_date"]
            )
            .between(
                refit_start_date,
                refit_end_date,
                inclusive="both",
            )
            .all()
        ),
        (
            f"dates = {refit_start_date.date()} "
            f"through {refit_end_date.date()}"
        ),
    ),
    qa_row(
        "Final-refit rows remain entirely pre-policy",
        (
            pre_cp_final_refit_supplement_df[
                "target_pre_post_cp"
            ].eq("pre_cp").all()
            and pre_cp_final_refit_supplement_df[
                "origin_pre_post_cp"
            ].eq("pre_cp").all()
        ),
        "origin and target policy labels = pre_cp",
    ),
    qa_row(
        "Final-refit label availability matches the observed target values",
        (
            pre_cp_final_refit_supplement_df["target_label_available"]
            .astype(bool)
            .eq(
                pre_cp_final_refit_supplement_df[
                    "target_observed_value"
                ].notna()
            )
            .all()
        ),
        (
            "observed labels = "
            f"{int(pre_cp_final_refit_supplement_df['target_label_available'].sum()):,} "
            f"of {len(pre_cp_final_refit_supplement_df):,} rows"
        ),
    ),
    qa_row(
        "Final-refit supplement does not overlap the model-selection matrix",
        historical_overlap_df.empty,
        (
            "historical rows at/after first refit target = "
            f"{len(historical_overlap_df):,}"
        ),
    ),
    qa_row(
        "Target-specific local context exclusions remain intact",
        own_target_exclusions_ok,
        "two local 4.2.2 definitions remain blank for each active target",
    ),
    qa_row(
        "Mapped-zone weather remains complete in final-refit rows",
        mapped_weather_complete,
        f"mapped supplement rows = {int(mapped_rows.sum()):,}",
    ),
])

display(pre_cp_refit_supplement_qa_df)

if not pre_cp_refit_supplement_qa_df["status"].eq("PASS").all():
    failures = pre_cp_refit_supplement_qa_df.loc[
        pre_cp_refit_supplement_qa_df["status"].ne("PASS")
    ]

    raise AssertionError(
        "Final-refit-only Pre-CP supplement QA failed:\n"
        + failures[
            ["check", "detail"]
        ].to_string(index=False)
    )

,check,status,detail
0,Final-refit supplement matches the historical Pre-CP matrix schema,PASS,columns = 120
1,Final-refit supplement contains the expected derived row count,PASS,"rows = 62,340; expected = 62,340"
2,Final-refit supplement preserves all native Pre-CP Forecast Sequences,PASS,"sequences = 1,039; expected = 1,039"
3,Final-refit supplement is isolated to the Pre-CP final_refit_only stage,PASS,track = pre_cp_reference; stage = final_refit_only
4,"Final-refit supplement contains exactly h=1, h=2, and h=5",PASS,"horizons = [1, 2, 5]"
5,Every final-refit target sequence equals origin + horizon,PASS,"rows checked = 62,340"
6,Final-refit targets are exactly the January 1–4 target positions,PASS,target IDs = 3656–3675
7,"Final-refit target dates are confined to January 1–4, 2025",PASS,dates = 2025-01-01 through 2025-01-04
8,Final-refit rows remain entirely pre-policy,PASS,origin and target policy labels = pre_cp
9,Final-refit label availability matches the observed target values,PASS,"observed labels = 61,866 of 62,340 rows"


**Findings.** All **13 final-refit supplement checks pass**. The supplement matches the physical schema of the historical Pre-CP matrix, preserves all **1,039 native Forecast Sequences**, contains exactly h=1/h=2/h=5, and preserves observed-target availability without manufacturing values for genuinely missing mobility observations. Target-specific context exclusions and mapped-zone weather remain valid, the historical and final-refit surfaces do not overlap, and the supplement reaches the final legal Pre-CP state at observation sequence **3675**.

### Section Summary

Section 4.2.3.6 converts the completed forecasting feature contracts into the three ordinary historical modeling matrices without changing the experiment established upstream.

Each output is keyed by **Modeling Track × Forecast Sequence × Origin × Horizon** and combines the legal 4.2.1 target-history/seasonal layer, the 4.2.2 multimodal/connected/static layer, horizon-specific weather, target timing, and the actual future mobility value as a label. Target-specific assembly blanks the two local multimodal fields derived from the active target while preserving the rest of the shared schema and any natural source missingness.

The matrices retain the native Post-CP Primary, Full-History Regime-Aware, and Pre-CP Reference Forecast Sequence populations. The 1,029-sequence cross-regime set remains a comparison population only. The final files also preserve the exact rolling-origin schedule and h=1, h=2, and h=5 target arithmetic established in 4.1.2.

For the Pre-CP Reference track, this section also publishes a separate **final-refit-only training supplement** for January 1–4, 2025. Those rows use the same physical forecasting-matrix contract and the same legal feature-construction rules, but remain separate from the historical matrix so the four final Pre-CP days can be used only after family selection is frozen. Together, the historical Pre-CP matrix and this supplement provide the complete supervised history needed for the eventual final production refit through January 4.

The historical matrices and final-refit supplement are now suitable for ordinary model training and final refitting. The remaining task is different: create the **runtime-aware post-freeze Pre-CP application handoff** without filling recursively generated no-CP features with actual post-CP mobility.

## 4.2.3.7 Build the Frozen Pre-CP Counterfactual Application Handoff

The historical Pre-CP Reference matrix ends with legal observed mobility on January 4, 2025. It cannot simply be extended into the post-CP period by joining the real transportation panel.

Instead, the post-freeze no-CP application requires two complementary artifacts:

1. a **feature runtime contract** describing what happens to every one of the 86 shared definitions after the freeze; and
2. a **counterfactual application template** containing the legal deterministic, frozen, static, and exogenous values that can be materialized in advance while leaving recursively generated mobility state as explicit runtime placeholders.

The template therefore remains intentionally incomplete in a very specific way. Once origin sequence 3675 has been passed, target-derived mobility history and reconstructable multimodal state must be generated from the evolving synthetic no-CP forecast path. Auxiliary ordinary-only mobility context remains unavailable unless separately modeled.

This is a stronger leakage boundary than merely dropping the final target label: **actual post-CP mobility must not appear anywhere in the recursive feature state.**

**Publish the complete frozen-runtime contract.** Convert the combined feature catalog into an explicit post-freeze action for all 86 definitions.

The contract distinguishes features that are recursively recomputed, features that become unavailable, deterministic future-known fields, frozen training-fitted state, static geography, exogenous weather, and policy-regime fields that are deliberately not used in the no-CP path.

In [30]:
# 4.2.3.7A — Build the complete frozen Pre-CP runtime contract

runtime_action_map = {
    "recursive_required": "recompute_from_recursive_no_cp_state",
    "ordinary_only": "unavailable_after_freeze",
    "not_used": "not_used_in_frozen_counterfactual",
    "deterministic_future_known": "materialize_from_future_known_calendar",
    "frozen_training_fitted": "materialize_from_frozen_pre_cp_fit",
    "training_fitted": "materialize_from_frozen_pre_cp_fit",
    "static_available": "materialize_from_static_geography",
    "exogenous_available": "materialize_from_exogenous_weather",
}

pre_cp_final_runtime_contract_df = forecast_feature_catalog_complete_df.copy()
unknown_runtime_classes = sorted(
    set(pre_cp_final_runtime_contract_df["pre_cp_post_freeze_class"].dropna())
    - set(runtime_action_map)
)
if unknown_runtime_classes:
    raise ValueError(f"Unrecognized post-freeze feature classes: {unknown_runtime_classes}")

pre_cp_final_runtime_contract_df["post_freeze_action"] = (
    pre_cp_final_runtime_contract_df["pre_cp_post_freeze_class"].map(runtime_action_map)
)
pre_cp_final_runtime_contract_df["runtime_placeholder_after_seed"] = (
    pre_cp_final_runtime_contract_df["pre_cp_post_freeze_class"].eq("recursive_required")
)
pre_cp_final_runtime_contract_df["structurally_unavailable_after_freeze"] = (
    pre_cp_final_runtime_contract_df["pre_cp_post_freeze_class"].isin(["ordinary_only", "not_used"])
)
pre_cp_final_runtime_contract_df["target_conditional_exclusion"] = (
    pre_cp_final_runtime_contract_df["drop_when_source_metric_equals_target_metric"].astype(bool)
)

if pre_cp_final_runtime_contract_df["uses_actual_post_cp_mobility"].astype(bool).any():
    raise AssertionError("The completed runtime contract permits actual post-CP mobility.")

if WRITE_FINAL_OUTPUTS:
    pre_cp_final_runtime_contract_df.to_parquet(FINAL_RUNTIME_CONTRACT_PATH, index=False)

runtime_contract_summary_df = (
    pre_cp_final_runtime_contract_df
    .groupby(["pre_cp_post_freeze_class", "post_freeze_action"], observed=True, dropna=False)
    .agg(
        feature_definitions=("feature_name", "size"),
        target_conditional=("target_conditional_exclusion", "sum"),
        requires_cross_zone_state=("requires_cross_zone_recursive_state", "sum"),
    )
    .reset_index()
)

display(runtime_contract_summary_df)

,pre_cp_post_freeze_class,post_freeze_action,feature_definitions,target_conditional,requires_cross_zone_state
0,deterministic_future_known,materialize_from_future_known_calendar,16,0,0
1,exogenous_available,materialize_from_exogenous_weather,8,0,0
2,frozen_training_fitted,materialize_from_frozen_pre_cp_fit,2,0,0
3,not_used,not_used_in_frozen_counterfactual,2,0,0
4,ordinary_only,unavailable_after_freeze,15,0,0
5,recursive_required,recompute_from_recursive_no_cp_state,32,10,5
6,static_available,materialize_from_static_geography,11,0,0


**Findings.** Every one of the 86 feature definitions now has a clear rule for what happens after the Pre-CP model is frozen on January 4, 2025. Some features can still be known directly, such as calendar information, static geography, frozen seasonal patterns, and weather. Other mobility-based features must be rebuilt from the model's own earlier no-CP predictions, while some auxiliary mobility features simply become unavailable after the freeze. Two policy-regime fields are intentionally not used. Most importantly, none of the rules allows actual post-CP mobility to enter the no-CP forecast path.

**Normalize the post-freeze counterfactual calendar and freeze the weekly seasonal state.** Use the 4.2.1 counterfactual schedule as the application backbone, independently verify its origin/target identity against the canonical calendar, and attach the already-fitted annual and weekly seasonal state without observing future mobility.

The first legal recursive seed remains January 4 Evening at sequence 3675. The first future no-CP target is January 5 Overnight at sequence 3676.

In [31]:
# 4.2.3.7B — Normalize the counterfactual calendar and frozen seasonal state

counterfactual_calendar_df = pre_cp_counterfactual_calendar_df.copy()
if "modeling_track" not in counterfactual_calendar_df.columns:
    counterfactual_calendar_df["modeling_track"] = "pre_cp_reference"

if not counterfactual_calendar_df["modeling_track"].eq("pre_cp_reference").all():
    raise ValueError("The counterfactual calendar contains a non-Pre-CP modeling track.")

# Canonical identity is joined independently on both sides so a stale date or
# Daypart in the application schedule cannot silently propagate downstream.
calendar_base_cols = [
    "observation_sequence_id", "date", "daypart", "daypart_order",
    "temporal_bucket", "temporal_bucket_order", "pre_post_cp",
]


def attach_canonical_side(df, side, sequence_key):
    lookup = canonical_calendar_df[calendar_base_cols].copy()
    lookup = lookup.rename(columns={
        "observation_sequence_id": sequence_key,
        "date": f"{side}_date__canonical",
        "daypart": f"{side}_daypart__canonical",
        "daypart_order": f"{side}_daypart_order__canonical",
        "temporal_bucket": f"{side}_temporal_bucket__canonical",
        "temporal_bucket_order": f"{side}_temporal_bucket_order__canonical",
        "pre_post_cp": f"{side}_pre_post_cp__canonical",
    })

    result = df.merge(lookup, on=sequence_key, how="left", validate="many_to_one")

    for field in [
        "date", "daypart", "daypart_order",
        "temporal_bucket", "temporal_bucket_order", "pre_post_cp",
    ]:
        canonical_col = f"{side}_{field}__canonical"
        output_col = f"{side}_{field}"

        if output_col in result.columns:
            left = result[output_col]
            right = result[canonical_col]
            both = left.notna() & right.notna()

            if field == "date":
                mismatch = both & pd.to_datetime(left).ne(pd.to_datetime(right))
            else:
                mismatch = both & left.astype(str).ne(right.astype(str))

            if mismatch.any():
                raise ValueError(f"Counterfactual {output_col} disagrees with canonical calendar.")

            result[output_col] = result[output_col].where(result[output_col].notna(), right)
        else:
            result[output_col] = result[canonical_col]

        result = result.drop(columns=canonical_col)

    return result


counterfactual_calendar_df = attach_canonical_side(
    counterfactual_calendar_df, "origin", "origin_observation_sequence_id"
)
counterfactual_calendar_df = attach_canonical_side(
    counterfactual_calendar_df, "target", "target_observation_sequence_id"
)

for side in ["origin", "target"]:
    date_col = f"{side}_date"
    counterfactual_calendar_df[date_col] = pd.to_datetime(counterfactual_calendar_df[date_col])
    counterfactual_calendar_df[f"{side}_weekday_index"] = counterfactual_calendar_df[date_col].dt.weekday.astype(int)
    counterfactual_calendar_df[f"{side}_is_weekend"] = counterfactual_calendar_df[f"{side}_weekday_index"].ge(5)
    counterfactual_calendar_df[f"{side}_month"] = counterfactual_calendar_df[date_col].dt.month.astype(int)
    counterfactual_calendar_df[f"{side}_day_of_year"] = counterfactual_calendar_df[date_col].dt.dayofyear.astype(int)

counterfactual_calendar_df["origin_elapsed_days"] = (
    counterfactual_calendar_df["origin_date"] - STUDY_START_DATE
).dt.days.astype(int)

# Reuse the exact annual Fourier basis fitted upstream rather than reproducing
# its formula here.
if not {"annual_fourier_sin", "annual_fourier_cos"}.issubset(counterfactual_calendar_df.columns):
    annual_fourier_lookup_df = (
        pd.read_parquet(
            ANNUAL_FOURIER_421_PATH,
            columns=["date", "annual_fourier_sin", "annual_fourier_cos"],
        )
        .drop_duplicates("date")
        .rename(columns={"date": "target_date"})
    )
    annual_fourier_lookup_df["target_date"] = pd.to_datetime(
        annual_fourier_lookup_df["target_date"]
    )
    counterfactual_calendar_df = counterfactual_calendar_df.merge(
        annual_fourier_lookup_df, on="target_date", how="left", validate="many_to_one"
    )

counterfactual_key = ["modeling_track", "origin_observation_sequence_id", "horizon"]
if counterfactual_calendar_df.duplicated(counterfactual_key).any():
    raise ValueError("The post-freeze counterfactual calendar is not unique.")

if not (
    counterfactual_calendar_df["target_observation_sequence_id"]
    .sub(counterfactual_calendar_df["origin_observation_sequence_id"])
    .eq(counterfactual_calendar_df["horizon"])
    .all()
):
    raise ValueError("Counterfactual calendar violates target = origin + horizon.")

# All non-weekly target-side 4.2.1 fields must now be directly materializable.
counterfactual_target_required = [
    f for f in target_context_feature_names if f != "target_pre_post_cp"
]
missing_counterfactual_target = sorted(
    set(counterfactual_target_required) - set(counterfactual_calendar_df.columns)
)
if missing_counterfactual_target:
    raise ValueError(
        "Counterfactual calendar cannot materialize required target fields: "
        f"{missing_counterfactual_target}"
    )

counterfactual_calendar_summary_df = pd.DataFrame([{
    "rows": len(counterfactual_calendar_df),
    "origins": counterfactual_calendar_df["origin_observation_sequence_id"].nunique(),
    "targets": counterfactual_calendar_df["target_observation_sequence_id"].nunique(),
    "first_origin_id": int(counterfactual_calendar_df["origin_observation_sequence_id"].min()),
    "first_target_id": int(counterfactual_calendar_df["target_observation_sequence_id"].min()),
    "last_target_date": counterfactual_calendar_df["target_date"].max(),
    "horizons": ", ".join(map(str, sorted(counterfactual_calendar_df["horizon"].unique()))),
}])

display(counterfactual_calendar_summary_df)

,rows,origins,targets,first_origin_id,first_target_id,last_target_date,horizons
0,6760,2255,2255,3675,3676,2026-03-31,"1, 2, 5"


**Findings.** The no-congestion-pricing forecast schedule contains 6,760 origin-and-horizon combinations covering 2,255 forecast origins and the 1-, 2-, and 5-step horizons. It starts exactly where it should: the final observed Pre-CP period is January 4, 2025 Evening at sequence 3675, and the first no-CP prediction is January 5 Overnight at sequence 3676. The schedule then continues through March 31, 2026. This confirms that the counterfactual forecast begins at the correct policy boundary and follows the same canonical time sequence used everywhere else in the project.

**Resolve the frozen weekly-STL target value for counterfactual rows.** Unlike ordinary historical forecasts, the post-freeze path never advances to a newly fitted seasonal state. The legal Pre-CP weekly template remains frozen while the future target weekday selects the appropriate Monday–Sunday offset.

In [32]:
# 4.2.3.7C — Define frozen weekly-STL target resolution

required_frozen_weekly_cols = {
    "modeling_track", "forecast_sequence_id", "daypart",
    "weekly_stl_enabled", *WEEKLY_STL_TEMPLATE_OFFSET_COLUMNS,
}
missing_frozen_weekly_cols = sorted(
    required_frozen_weekly_cols - set(pre_cp_counterfactual_weekly_state_df.columns)
)
if missing_frozen_weekly_cols:
    raise ValueError(
        "Frozen weekly-state artifact is missing columns: "
        f"{missing_frozen_weekly_cols}"
    )

# Bring only the weekly-state fields required by the runtime resolver into the
# merge. Identity fields such as Taxi Zone and metric already exist on the
# application matrix and must not be duplicated into _x/_y columns.
frozen_weekly_lookup_df = (
    pre_cp_counterfactual_weekly_state_df[
        [
            "modeling_track",
            "forecast_sequence_id",
            "daypart",
            "weekly_stl_enabled",
            *WEEKLY_STL_TEMPLATE_OFFSET_COLUMNS,
        ]
    ]
    .rename(columns={"daypart": "target_daypart"})
    .copy()
)

frozen_weekly_lookup_df["_frozen_weekly_present"] = True


def attach_frozen_weekly_stl_state(rows):
    """Attach the single legal Pre-CP weekly template throughout post-freeze application."""
    result = rows.merge(
        frozen_weekly_lookup_df,
        on=["modeling_track", "forecast_sequence_id", "target_daypart"],
        how="left",
        validate="many_to_one",
    )

    if result["_frozen_weekly_present"].isna().any():
        raise ValueError("At least one counterfactual row missed the frozen weekly-STL state.")

    enabled = result["weekly_stl_enabled"].fillna(False).astype(bool)
    weekdays = result["target_weekday_index"].astype(int).to_numpy()
    offsets = result[WEEKLY_STL_TEMPLATE_OFFSET_COLUMNS].to_numpy(dtype=float)
    selected = offsets[np.arange(len(result)), weekdays]

    result["weekly_stl_seasonal_offset"] = np.where(enabled, selected, 0.0)

    if not np.isfinite(
        pd.to_numeric(result.loc[enabled, "weekly_stl_seasonal_offset"], errors="coerce")
    ).all():
        raise ValueError("An enabled frozen weekly-STL row has no finite target offset.")

    # The seven source offsets are runtime provenance, not seven additional predictors.
    return result.drop(columns=[*WEEKLY_STL_TEMPLATE_OFFSET_COLUMNS, "_frozen_weekly_present"])

**Build the post-freeze counterfactual application template.** Materialize everything that is legally known in advance while preserving recursive mobility state as runtime placeholders.

The legal sequence-3675 seed may carry observed Pre-CP mobility-derived state. For every later forecast origin, recursive-required 4.2.1 and 4.2.2 fields remain missing until the no-CP forecasting loop generates the synthetic state. Auxiliary ordinary-only context also remains unavailable after the seed.

Actual post-CP weather is allowed because it is exogenous. Actual post-CP mobility and future target labels are not attached.

As with the historical matrices, the application template is streamed by target metric to keep memory bounded.

In [33]:
# 4.2.3.7D — Materialize the frozen Pre-CP counterfactual application template

recursive_421_features = pre_cp_final_runtime_contract_df.loc[
    pre_cp_final_runtime_contract_df["source_notebook"].eq("4.2.1")
    & pre_cp_final_runtime_contract_df["pre_cp_post_freeze_class"].eq("recursive_required"),
    "feature_name",
].tolist()

recursive_422_features = pre_cp_final_runtime_contract_df.loc[
    pre_cp_final_runtime_contract_df["source_notebook"].eq("4.2.2")
    & pre_cp_final_runtime_contract_df["pre_cp_post_freeze_class"].eq("recursive_required"),
    "feature_name",
].tolist()

ordinary_only_422_features = pre_cp_final_runtime_contract_df.loc[
    pre_cp_final_runtime_contract_df["source_notebook"].eq("4.2.2")
    & pre_cp_final_runtime_contract_df["pre_cp_post_freeze_class"].eq("ordinary_only"),
    "feature_name",
].tolist()

static_422_features = pre_cp_final_runtime_contract_df.loc[
    pre_cp_final_runtime_contract_df["source_notebook"].eq("4.2.2")
    & pre_cp_final_runtime_contract_df["pre_cp_post_freeze_class"].eq("static_available"),
    "feature_name",
].tolist()

not_used_features = pre_cp_final_runtime_contract_df.loc[
    pre_cp_final_runtime_contract_df["pre_cp_post_freeze_class"].eq("not_used"),
    "feature_name",
].tolist()

materialized_421_classes = ["deterministic_future_known", "frozen_training_fitted", "training_fitted"]
materialized_421_features = pre_cp_final_runtime_contract_df.loc[
    pre_cp_final_runtime_contract_df["source_notebook"].eq("4.2.1")
    & pre_cp_final_runtime_contract_df["pre_cp_post_freeze_class"].isin(materialized_421_classes),
    "feature_name",
].tolist()

# Prepare reusable physical weather joins outside the target loop.
counterfactual_origin_weather_df = completed_weather_context_df[
    ["taxi_zone_id", "observation_sequence_id", *WEATHER_RETAINED_METRICS]
].rename(columns={
    "observation_sequence_id": "origin_observation_sequence_id",
    **{m: f"weather_origin__{m}" for m in WEATHER_RETAINED_METRICS},
})

counterfactual_target_weather_df = completed_weather_context_df[
    ["taxi_zone_id", "observation_sequence_id", *WEATHER_RETAINED_METRICS]
].rename(columns={
    "observation_sequence_id": "target_observation_sequence_id",
    **{m: f"weather_target_proxy__{m}" for m in WEATHER_RETAINED_METRICS},
})


def build_counterfactual_partition(target_metric):
    """
    Build one target-metric slice of the post-freeze Pre-CP application template.

    Only the legal 3675 seed may carry observed mobility-derived dynamic state.
    """
    origin_path = ORIGIN_FEATURE_PATHS_421["pre_cp_reference"]
    context_path = ORIGIN_CONTEXT_PATHS_422["pre_cp_reference"]

    # Recover the full native Pre-CP sequence roster without requiring every
    # sequence to have a nonmissing target outcome at the application target.
    sequence_roster = (
        pd.read_parquet(
            origin_path,
            columns=[
                "modeling_track", "forecast_sequence_id", "taxi_zone_id",
                "zone", "borough", "metric",
            ],
            filters=[("metric", "==", target_metric)],
        )
        .drop_duplicates("forecast_sequence_id")
        .reset_index(drop=True)
    )

    if not sequence_roster["modeling_track"].eq("pre_cp_reference").all():
        raise ValueError("Pre-CP sequence roster contains an unexpected modeling track.")

    application_calendar = counterfactual_calendar_df.drop(columns="modeling_track")
    matrix = sequence_roster.merge(application_calendar, how="cross")

    # Attach only the legal observed seed values for recursively generated
    # target-history fields. Future origins intentionally do not match this join.
    seed_origin_cols = list(dict.fromkeys([
        "modeling_track", "forecast_sequence_id", "observation_sequence_id",
        *recursive_421_features,
    ]))
    seed_origin = pd.read_parquet(
        origin_path,
        columns=seed_origin_cols,
        filters=[
            ("metric", "==", target_metric),
            ("observation_sequence_id", "==", PRE_CP_FREEZE_OBSERVATION_SEQUENCE_ID),
        ],
    ).rename(columns={"observation_sequence_id": "origin_observation_sequence_id"})

    matrix = matrix.merge(
        seed_origin,
        on=["modeling_track", "forecast_sequence_id", "origin_observation_sequence_id"],
        how="left",
        validate="many_to_one",
    )

    # Static 4.2.2 fields can be carried indefinitely; dynamic target-derived
    # and auxiliary context may only enter from the legal 3675 seed.
    seed_context_cols = list(dict.fromkeys([
        "modeling_track", "forecast_sequence_id", "observation_sequence_id",
        "taxi_zone_id", *recursive_422_features, *ordinary_only_422_features,
        *static_422_features,
    ]))
    seed_context = pd.read_parquet(
        context_path,
        columns=seed_context_cols,
        filters=[
            ("metric", "==", target_metric),
            ("observation_sequence_id", "==", PRE_CP_FREEZE_OBSERVATION_SEQUENCE_ID),
        ],
    ).rename(columns={"observation_sequence_id": "origin_observation_sequence_id"})

    dynamic_422 = list(dict.fromkeys([*recursive_422_features, *ordinary_only_422_features]))
    matrix = matrix.merge(
        seed_context[
            ["modeling_track", "forecast_sequence_id", "origin_observation_sequence_id",
             *dynamic_422]
        ],
        on=["modeling_track", "forecast_sequence_id", "origin_observation_sequence_id"],
        how="left",
        validate="many_to_one",
    )

    # Several static fields, especially borough, are already identity metadata.
    # Reuse them directly instead of creating duplicate _x/_y columns.
    static_to_add = [f for f in static_422_features if f not in matrix.columns]
    if static_to_add:
        static_lookup = (
            seed_context[["taxi_zone_id", *static_to_add]]
            .drop_duplicates()
            .reset_index(drop=True)
        )
        if static_lookup.duplicated("taxi_zone_id").any():
            raise ValueError("Static Pre-CP context is not unique by Taxi Zone.")
        matrix = matrix.merge(
            static_lookup, on="taxi_zone_id", how="left", validate="many_to_one"
        )

    matrix = attach_frozen_weekly_stl_state(matrix)

    # Preserve actual calendar regime labels as interpretation metadata before
    # clearing the two policy-regime predictor fields from the no-CP feature set.
    if "origin_pre_post_cp" in matrix.columns:
        matrix["origin_actual_pre_post_cp"] = matrix["origin_pre_post_cp"]
    if "target_pre_post_cp" in matrix.columns:
        matrix["target_actual_pre_post_cp"] = matrix["target_pre_post_cp"]

    for feature_name in not_used_features:
        if feature_name in matrix.columns:
            matrix[feature_name] = pd.NA

    matrix = (
        matrix
        .merge(
            counterfactual_origin_weather_df,
            on=["taxi_zone_id", "origin_observation_sequence_id"],
            how="left",
            validate="many_to_one",
        )
        .merge(
            counterfactual_target_weather_df,
            on=["taxi_zone_id", "target_observation_sequence_id"],
            how="left",
            validate="many_to_one",
        )
    )
    matrix["weather_zone_mapped"] = matrix["taxi_zone_id"].isin(weather_zone_ids)

    # The target-neutral local context schema still requires the same two-field
    # exclusion used in the ordinary historical matrices.
    own_features = own_target_feature_map[target_metric]
    for feature_name in own_features:
        if feature_name not in matrix.columns:
            matrix[feature_name] = np.nan
        else:
            matrix[feature_name] = np.nan

    # Any feature absent at this point must belong to a class that is explicitly
    # runtime-generated or unavailable. A missing deterministic/static/weather
    # feature is an assembly error and should stop the notebook.
    runtime_class_by_feature = pre_cp_final_runtime_contract_df.set_index(
        "feature_name"
    )["pre_cp_post_freeze_class"].to_dict()

    for feature_name in all_predictor_names:
        if feature_name in matrix.columns:
            continue

        feature_class = runtime_class_by_feature[feature_name]
        if feature_class in {"recursive_required", "ordinary_only", "not_used"}:
            matrix[feature_name] = np.nan
        else:
            raise ValueError(
                f"Counterfactual template failed to materialize {feature_name} "
                f"({feature_class})."
            )

    # After the legal seed, recursive fields are intentionally unresolved until
    # a model prediction updates the synthetic no-CP state.
    post_seed = matrix["origin_observation_sequence_id"].gt(PRE_CP_FREEZE_OBSERVATION_SEQUENCE_ID)
    matrix["recursive_state_pending"] = post_seed
    matrix["runtime_recursive_feature_count"] = len(recursive_421_features) + len(recursive_422_features)
    matrix["ordinary_only_context_unavailable_count"] = len(ordinary_only_422_features)
    matrix["own_target_context_exclusion_count"] = len(own_features)

    if "counterfactual_origin_source" not in matrix.columns:
        matrix["counterfactual_origin_source"] = np.where(
            post_seed, "recursive_no_cp_state", "legal_observed_seed"
        )

    if "recursive_state_update" not in matrix.columns:
        matrix["recursive_state_update"] = np.where(
            post_seed, "required_before_next_h1_origin", "seed_available"
        )

    # No future observed target value is ever attached to this artifact.
    if "target_observed_value" in matrix.columns:
        raise AssertionError("Counterfactual application template contains a future mobility label.")

    missing_predictors = sorted(set(all_predictor_names) - set(matrix.columns))
    if missing_predictors:
        raise ValueError(f"Counterfactual template is missing predictors: {missing_predictors}")

    metadata_order = [
        "modeling_track", "forecast_sequence_id", "taxi_zone_id", "zone", "borough", "metric",
        "origin_observation_sequence_id", "target_observation_sequence_id",
        "horizon", "horizon_label", "target_period",
        "origin_date", "origin_daypart", "origin_temporal_bucket",
        "target_date", "target_daypart", "target_temporal_bucket",
        "origin_actual_pre_post_cp", "target_actual_pre_post_cp",
        "counterfactual_origin_source", "recursive_state_update",
    ]
    runtime_order = [
        "weather_zone_mapped", "recursive_state_pending",
        "runtime_recursive_feature_count",
        "ordinary_only_context_unavailable_count",
        "own_target_context_exclusion_count",
    ]

    ordered_cols = []
    for col in [*metadata_order, *all_predictor_names, *runtime_order]:
        if col in matrix.columns and col not in ordered_cols:
            ordered_cols.append(col)

    extras = [c for c in matrix.columns if c not in ordered_cols]
    return matrix[ordered_cols + extras]


if PRE_CP_COUNTERFACTUAL_TEMPLATE_PATH.exists() and not REBUILD_COUNTERFACTUAL_TEMPLATE:
    counterfactual_build_audit_df = pd.read_parquet(
        PRE_CP_COUNTERFACTUAL_TEMPLATE_PATH,
        columns=[
            "metric", "forecast_sequence_id", "origin_observation_sequence_id",
            "target_observation_sequence_id", "horizon",
            "weather_zone_mapped", "recursive_state_pending",
        ],
    )
    counterfactual_build_summary_df = (
        counterfactual_build_audit_df.groupby("metric", observed=True)
        .agg(
            rows=("forecast_sequence_id", "size"),
            forecast_sequences=("forecast_sequence_id", "nunique"),
            origins=("origin_observation_sequence_id", "nunique"),
            horizons=("horizon", "nunique"),
            weather_mapped_rows=("weather_zone_mapped", "sum"),
            recursive_pending_rows=("recursive_state_pending", "sum"),
        )
        .reset_index()
    )
    del counterfactual_build_audit_df

else:
    temp_path = PRE_CP_COUNTERFACTUAL_TEMPLATE_PATH.with_suffix(".tmp.parquet")
    temp_path.unlink(missing_ok=True)

    writer = None
    writer_schema = None
    build_records = []

    try:
        for target_metric in FORECAST_TARGETS:
            partition = build_counterfactual_partition(target_metric)

            build_records.append({
                "metric": target_metric,
                "rows": len(partition),
                "forecast_sequences": partition["forecast_sequence_id"].nunique(),
                "origins": partition["origin_observation_sequence_id"].nunique(),
                "horizons": partition["horizon"].nunique(),
                "weather_mapped_rows": int(partition["weather_zone_mapped"].sum()),
                "recursive_pending_rows": int(partition["recursive_state_pending"].sum()),
            })

            writer, writer_schema = append_arrow_partition(
                writer, writer_schema, partition, temp_path
            )

            del partition
            gc.collect()

    finally:
        if writer is not None:
            writer.close()

    if not temp_path.exists() or temp_path.stat().st_size == 0:
        raise IOError("Failed to create the Pre-CP counterfactual application template.")

    temp_path.replace(PRE_CP_COUNTERFACTUAL_TEMPLATE_PATH)
    counterfactual_build_summary_df = pd.DataFrame(build_records)

counterfactual_build_summary_df["weather_mapped_share_pct"] = (
    counterfactual_build_summary_df["weather_mapped_rows"]
    / counterfactual_build_summary_df["rows"] * 100
).round(3)

display(
    counterfactual_build_summary_df[
        [
            "metric", "rows", "forecast_sequences", "origins",
            "horizons", "weather_mapped_share_pct", "recursive_pending_rows",
        ]
    ]
)

,metric,rows,forecast_sequences,origins,horizons,weather_mapped_share_pct,recursive_pending_rows
0,fhvhv_avg_trip_speed,1737320,257,2255,3,99.222,1736549
1,fhvhv_trip_count,1744080,258,2255,3,98.837,1743306
2,subway_ridership,1054560,156,2255,3,100.000,1054092
3,taxi_avg_trip_speed,743600,110,2255,3,98.182,743270
4,taxi_trip_count,1744080,258,2255,3,98.837,1743306


**Findings.** The no-CP application template contains 7,023,640 rows covering all five forecast targets, all 1,039 Pre-CP Reference sequences, and the 1-, 2-, and 5-step horizons. Information that can legitimately be known ahead of time—such as calendar features, frozen seasonal patterns, static geography, and weather—is filled in now. Mobility features that depend on what would have happened under no congestion pricing are intentionally left blank so they can later be generated from the model's own previous no-CP predictions. Actual post-CP mobility and the future observed target values are not included. This gives the later counterfactual forecasting code a clean starting template without leaking what actually happened after congestion pricing began.

**Validate the frozen counterfactual boundary.** Confirm that the post-freeze template preserves the native Pre-CP Reference population while leaving recursively generated and unavailable mobility state unresolved after the legal seed.

The QA explicitly distinguishes source missingness from leakage protection. Static geography may naturally be absent for unsupported/unknown geography, and weather may be absent for Taxi Zones outside the 260-zone mapped weather surface. Those are not filled here.

In [34]:
# 4.2.3.7E — Validate the frozen Pre-CP application template

counterfactual_qa_cols = list(dict.fromkeys([
    *FINAL_MATRIX_KEYS, "taxi_zone_id", "metric",
    "target_observation_sequence_id", "weather_zone_mapped",
    "origin_actual_pre_post_cp", "target_actual_pre_post_cp",
    *recursive_421_features, *recursive_422_features,
    *ordinary_only_422_features, *not_used_features,
    *materialized_421_features, *WEATHER_FORECAST_FEATURE_COLUMNS,
]))

counterfactual_audit_df = pd.read_parquet(
    PRE_CP_COUNTERFACTUAL_TEMPLATE_PATH,
    columns=[c for c in counterfactual_qa_cols if c in parquet_columns(PRE_CP_COUNTERFACTUAL_TEMPLATE_PATH)],
)

post_seed = counterfactual_audit_df["origin_observation_sequence_id"].gt(
    PRE_CP_FREEZE_OBSERVATION_SEQUENCE_ID
)
mapped_weather = counterfactual_audit_df["weather_zone_mapped"].fillna(False).astype(bool)

recursive_features = list(dict.fromkeys([
    *recursive_421_features, *recursive_422_features
]))
cross_zone_recursive_features = pre_cp_final_runtime_contract_df.loc[
    pre_cp_final_runtime_contract_df["requires_cross_zone_recursive_state"].astype(bool),
    "feature_name",
].tolist()

native_pre_sequences = pd.read_parquet(
    ORIGIN_FEATURE_PATHS_421["pre_cp_reference"],
    columns=["forecast_sequence_id"],
)["forecast_sequence_id"].nunique()

cf_sequence_count = counterfactual_audit_df["forecast_sequence_id"].nunique()
duplicate_rows = int(counterfactual_audit_df.duplicated(FINAL_MATRIX_KEYS).sum())

recursive_future_blank = (
    not post_seed.any()
    or counterfactual_audit_df.loc[post_seed, recursive_features].isna().all().all()
)
ordinary_future_blank = (
    not post_seed.any()
    or counterfactual_audit_df.loc[post_seed, ordinary_only_422_features].isna().all().all()
)
cross_zone_future_blank = (
    not cross_zone_recursive_features
    or counterfactual_audit_df.loc[post_seed, cross_zone_recursive_features].isna().all().all()
)
policy_features_blank = (
    not not_used_features
    or counterfactual_audit_df[not_used_features].isna().all().all()
)
mapped_weather_complete = (
    not mapped_weather.any()
    or counterfactual_audit_df.loc[
        mapped_weather, WEATHER_FORECAST_FEATURE_COLUMNS
    ].notna().all().all()
)

# Deterministic/frozen 4.2.1 state should be physically available throughout.
materialized_421_complete = (
    not materialized_421_features
    or counterfactual_audit_df[materialized_421_features].notna().all().all()
)

counterfactual_qa_df = pd.DataFrame([
    qa_row(
        "Counterfactual template is unique at Track × Forecast Sequence × Origin × Horizon",
        duplicate_rows == 0,
        f"duplicate keys = {duplicate_rows:,}",
    ),
    qa_row(
        "Counterfactual target sequence equals origin + horizon",
        counterfactual_audit_df["target_observation_sequence_id"]
        .sub(counterfactual_audit_df["origin_observation_sequence_id"])
        .eq(counterfactual_audit_df["horizon"]).all(),
        f"rows checked = {len(counterfactual_audit_df):,}",
    ),
    qa_row(
        "Counterfactual application begins from legal seed 3675 and first target 3676",
        int(counterfactual_audit_df["origin_observation_sequence_id"].min()) == PRE_CP_FREEZE_OBSERVATION_SEQUENCE_ID
        and int(counterfactual_audit_df["target_observation_sequence_id"].min()) == CP_FIRST_OBSERVATION_SEQUENCE_ID,
        (
            f"first origin = {int(counterfactual_audit_df['origin_observation_sequence_id'].min())}; "
            f"first target = {int(counterfactual_audit_df['target_observation_sequence_id'].min())}"
        ),
    ),
    qa_row(
        "Native Pre-CP Reference Forecast Sequence population is preserved",
        cf_sequence_count == native_pre_sequences,
        f"template = {cf_sequence_count:,}; native = {native_pre_sequences:,}",
    ),
    qa_row(
        "Counterfactual template is not restricted to the 1,029 comparison cohort",
        cf_sequence_count > 1_029,
        f"native sequences retained = {cf_sequence_count:,}",
    ),
    qa_row(
        "Post-seed recursive mobility features remain runtime placeholders",
        recursive_future_blank,
        f"recursive-required feature definitions = {len(recursive_features):,}",
    ),
    qa_row(
        "Post-seed auxiliary ordinary-only mobility context remains unavailable",
        ordinary_future_blank,
        f"ordinary-only feature definitions = {len(ordinary_only_422_features):,}",
    ),
    qa_row(
        "Cross-zone recursive context remains unresolved until synthetic zone states exist",
        cross_zone_future_blank,
        f"cross-zone recursive definitions = {len(cross_zone_recursive_features):,}",
    ),
    qa_row(
        "Policy-regime predictor fields are not used in the frozen no-CP path",
        policy_features_blank,
        f"not-used feature definitions = {len(not_used_features):,}",
    ),
    qa_row(
        "Deterministic and frozen training-fitted 4.2.1 state is materialized",
        materialized_421_complete,
        f"materialized definitions = {len(materialized_421_features):,}",
    ),
    qa_row(
        "Mapped-zone exogenous weather is complete throughout counterfactual application",
        mapped_weather_complete,
        f"mapped rows = {int(mapped_weather.sum()):,}",
    ),
    qa_row(
        "Counterfactual application artifact contains no future observed mobility label",
        "target_observed_value" not in parquet_columns(PRE_CP_COUNTERFACTUAL_TEMPLATE_PATH),
        "target_observed_value absent from template schema",
    ),
    qa_row(
        "Final runtime contract permits zero actual post-CP mobility",
        not pre_cp_final_runtime_contract_df["uses_actual_post_cp_mobility"].astype(bool).any(),
        "actual post-CP mobility usage = 0",
    ),
])

display(counterfactual_qa_df)

if not counterfactual_qa_df["status"].eq("PASS").all():
    raise AssertionError("Frozen Pre-CP counterfactual application QA failed.")

del counterfactual_audit_df
gc.collect()

,check,status,detail
0,Counterfactual template is unique at Track × Forecast Sequence × Origin × Horizon,PASS,duplicate keys = 0
1,Counterfactual target sequence equals origin + horizon,PASS,"rows checked = 7,023,640"
2,Counterfactual application begins from legal seed 3675 and first target 3676,PASS,first origin = 3675; first target = 3676
3,Native Pre-CP Reference Forecast Sequence population is preserved,PASS,"template = 1,039; native = 1,039"
4,"Counterfactual template is not restricted to the 1,029 comparison cohort",PASS,"native sequences retained = 1,039"
5,Post-seed recursive mobility features remain runtime placeholders,PASS,recursive-required feature definitions = 32
6,Post-seed auxiliary ordinary-only mobility context remains unavailable,PASS,ordinary-only feature definitions = 15
7,Cross-zone recursive context remains unresolved until synthetic zone states exist,PASS,cross-zone recursive definitions = 5
8,Policy-regime predictor fields are not used in the frozen no-CP path,PASS,not-used feature definitions = 2
9,Deterministic and frozen training-fitted 4.2.1 state is materialized,PASS,materialized definitions = 18


0

**Findings.** All 13 checks on the no-CP application template pass. The rows are unique, the future target periods line up correctly with each forecast horizon, and all 1,039 Pre-CP Reference sequences are preserved. Features that must later be generated from the model's own no-CP predictions are still blank, features that become unavailable after the freeze remain unavailable, and known calendar, seasonal, static, and weather information is present where expected. The template contains no future target label and no actual post-CP mobility. This confirms that the counterfactual dataset respects the January 4 information cutoff and is ready for the later recursive no-CP forecasting step.

### Section Summary

Section 4.2.3.7 packages the frozen Pre-CP Reference in the form required for a genuine post-CP no-congestion-pricing simulation rather than pretending that future mobility-derived predictors already exist.

The final runtime contract assigns every shared feature definition to an explicit post-freeze action. Deterministic future-known calendar features, frozen training-fitted seasonal state, static geography, and exogenous weather can be materialized directly. Auxiliary ordinary-only mobility context becomes unavailable, while recursive-required target-history and target-derived transportation context must be reconstructed from the evolving synthetic no-CP mobility state. Connected target-derived context preserves its additional requirement for predicted states across Taxi Zones.

The application template begins from the legal January 4 Evening seed at observation sequence 3675 and advances into targets beginning at sequence 3676. Actual post-CP weather remains available as external conditioning, but actual post-CP mobility and future target labels are absent. Policy-regime predictor fields are also excluded from the frozen no-CP path even though the artifact retains actual calendar regime metadata for interpretation.

The result is therefore not a falsely complete post-CP modeling matrix. It is an explicit **runtime application contract** that later forecasting code can advance recursively without crossing the congestion-pricing information boundary.

## 4.2.3.8 Final Cross-Family QA and Forecasting Handoff

The forecasting feature layer is now complete.

Before closing the notebook, consolidate the independent QA surfaces from 4.2.1, 4.2.2, weather construction, historical matrix assembly, the Pre-CP final-refit supplement, and the frozen Pre-CP application path into one final contract.

This final audit does not evaluate model accuracy. Its purpose is narrower and more important at this stage: prove that the exported datasets preserve the correct grain, native populations, target timing, feature availability, final-refit boundary, counterfactual information boundary, and label/predictor separation before any forecasting algorithm is trained.

**Consolidate the final leakage and handoff QA.** Combine the completed upstream checks with the historical-matrix, final-refit-supplement, and counterfactual checks, then add a small set of final cross-family assertions over the 86-definition feature catalog.

In [35]:
# 4.2.3.8A — Consolidate final forecasting-matrix QA

def tag_qa_scope(df, scope):
    """Normalize one completed QA table into the final cross-family audit."""
    result = df[["check", "status", "detail"]].copy()
    result.insert(0, "qa_scope", scope)
    return result

EXPECTED_FORECAST_TARGETS = [
    "taxi_trip_count",
    "taxi_avg_trip_speed",
    "fhvhv_trip_count",
    "fhvhv_avg_trip_speed",
    "subway_ridership",
]

final_cross_family_qa_df = pd.DataFrame([
    qa_row(
        "Complete feature catalog contains exactly 86 unique definitions",
        len(forecast_feature_catalog_complete_df) == 86
        and not forecast_feature_catalog_complete_df["feature_name"].duplicated().any(),
        f"feature definitions = {len(forecast_feature_catalog_complete_df):,}",
    ),
    qa_row(
        "Forecasting contract contains exactly the five approved target metrics",
        set(FORECAST_TARGETS) == set(EXPECTED_FORECAST_TARGETS)
        and len(FORECAST_TARGETS) == len(EXPECTED_FORECAST_TARGETS),
        f"targets = {FORECAST_TARGETS}",
    ),
    qa_row(
        "Historical target outcome remains outside the predictor universe",
        "target_observed_value" not in all_predictor_names,
        "target_observed_value is label-only",
    ),
    qa_row(
        "Frozen runtime contract covers every shared feature definition",
        set(pre_cp_final_runtime_contract_df["feature_name"]) == set(all_predictor_names),
        f"runtime definitions = {len(pre_cp_final_runtime_contract_df):,}",
    ),
    qa_row(
        "Frozen runtime contract permits no actual post-CP mobility",
        not pre_cp_final_runtime_contract_df["uses_actual_post_cp_mobility"].astype(bool).any(),
        "post-CP mobility feature usage = 0",
    ),
    qa_row(
        "Target-conditional local exclusions remain exactly two definitions per target",
        all(len(own_target_feature_map.get(metric, [])) == 2 for metric in FORECAST_TARGETS),
        "two target-local context fields excluded for each Forecast Sequence",
    ),
])

qa_parts = [
    tag_qa_scope(upstream_contract_qa_df, "Upstream 4.2.1 / 4.2.2 handoff"),
    tag_qa_scope(weather_completion_qa_df, "Weather source completion"),
    tag_qa_scope(weather_alignment_qa_df, "Weather horizon alignment"),
    tag_qa_scope(weather_runtime_qa_df, "Weather no-CP runtime"),
    tag_qa_scope(ordinary_matrix_qa_df, "Historical forecasting matrices"),
    tag_qa_scope(pre_cp_refit_supplement_qa_df,"Pre-CP final-refit supplement",),
    tag_qa_scope(counterfactual_qa_df, "Frozen Pre-CP application"),
    tag_qa_scope(final_cross_family_qa_df, "Final cross-family contract"),
]

final_forecasting_matrix_qa_df = pd.concat(qa_parts, ignore_index=True)

if WRITE_FINAL_OUTPUTS:
    final_forecasting_matrix_qa_df.to_parquet(FINAL_QA_423_PATH, index=False)

final_qa_summary_df = (
    final_forecasting_matrix_qa_df
    .groupby(["qa_scope", "status"], observed=True)
    .size()
    .rename("check_count")
    .reset_index()
)

display(final_qa_summary_df)

if not final_forecasting_matrix_qa_df["status"].eq("PASS").all():
    failures = final_forecasting_matrix_qa_df.loc[
        final_forecasting_matrix_qa_df["status"].ne("PASS")
    ]
    display(failures)
    raise AssertionError("Final 4.2.3 forecasting contract contains failed QA checks.")

,qa_scope,status,check_count
0,Final cross-family contract,PASS,6
1,Frozen Pre-CP application,PASS,13
2,Historical forecasting matrices,PASS,26
3,Pre-CP final-refit supplement,PASS,13
4,Upstream 4.2.1 / 4.2.2 handoff,PASS,10
5,Weather horizon alignment,PASS,6
6,Weather no-CP runtime,PASS,6
7,Weather source completion,PASS,7


**Findings.** The final notebook-wide audit passes all **87 checks**. These checks cover the upstream 4.2.1 and 4.2.2 handoffs, weather completion, weather timing, the three historical forecasting datasets, the Pre-CP final-refit supplement, and the frozen Pre-CP counterfactual template.

Together they confirm that forecast rows are unique, horizons point to the correct future periods, each modeling track keeps its intended population, weather is handled consistently, target values remain separate from predictors, and actual post-CP mobility cannot enter the no-CP forecast path. The Pre-CP training contract also preserves a clean boundary between the 2023/2024 model-selection matrix and the January 1–4 final-refit supplement, with the latter reaching the final legal observed state at sequence 3675.

There are no structural or leakage issues blocking downstream forecasting.

**Publish the final forecasting handoff manifest.** Inventory the production artifacts using Parquet metadata so the notebook closes with one compact record of what downstream forecasting work should consume.

The manifest distinguishes the ordinary historical modeling matrices, the separate Pre-CP final-refit training supplement, and the frozen Pre-CP application template. Those assets serve different stages of the forecasting workflow and should remain distinct through model selection, final refitting, evaluation, and later counterfactual application.

In [36]:
# 4.2.3.8B — Build the final forecasting handoff manifest

output_specs = [
    (
        "Completed Taxi Zone weather context",
        COMPLETED_WEATHER_CONTEXT_PATH,
        "Causally completed physical weather source used by all forecasting families",
    ),
    (
        "Weather imputation audit",
        WEATHER_IMPUTATION_AUDIT_PATH,
        "Cell-level source-completion provenance; not predictors",
    ),
    (
        "Weather feature catalog",
        WEATHER_FEATURE_CATALOG_PATH,
        "Eight production weather predictor definitions",
    ),
    (
        "Complete forecasting feature catalog",
        FINAL_COMBINED_FEATURE_CATALOG_PATH,
        "Authoritative 86-definition cross-family predictor contract",
    ),
]

for track in MODELING_TRACKS:
    output_specs.append((
        f"Historical forecasting matrix · {TRACK_LABELS[track]}",
        FINAL_MATRIX_PATHS[track],
        "Ordinary Track × Forecast Sequence × Origin × Horizon modeling matrix",
    ))

output_specs.extend([
    (
        "Pre-CP final-refit training supplement",
        PRE_CP_FINAL_REFIT_SUPPLEMENT_PATH,
        "Supervised Pre-CP training rows for January 1–4, 2025",
    ),
    (
        "Frozen Pre-CP final runtime contract",
        FINAL_RUNTIME_CONTRACT_PATH,
        "Post-freeze action for every shared feature definition",
    ),
    (
        "Pre-CP counterfactual application template",
        PRE_CP_COUNTERFACTUAL_TEMPLATE_PATH,
        "Runtime-aware post-freeze no-CP application handoff",
    ),
    (
        "Final 4.2.3 QA",
        FINAL_QA_423_PATH,
        "Consolidated cross-family leakage and handoff audit",
    ),
])

manifest_records = []
for name, path, role in output_specs:
    row = parquet_inventory_row(name, path, role)
    row["path"] = str(path)
    manifest_records.append(row)

forecasting_matrix_manifest_df = pd.DataFrame(manifest_records)
manifest_order = [
    "input", "role", "path", "rows", "columns", "size_mb",
]
manifest_order = [c for c in manifest_order if c in forecasting_matrix_manifest_df.columns]
forecasting_matrix_manifest_df = forecasting_matrix_manifest_df[
    manifest_order + [c for c in forecasting_matrix_manifest_df.columns if c not in manifest_order]
]

if WRITE_FINAL_OUTPUTS:
    forecasting_matrix_manifest_df.to_parquet(FINAL_MANIFEST_423_PATH, index=False)

display(forecasting_matrix_manifest_df)

,input,role,path,rows,columns,size_mb
0,Completed Taxi Zone weather context,Causally completed physical weather source used by all forecasting families,pipeline_data/4.2.3.intermediate_tables/weather_taxi_zone_context_completed.parquet,1541800,19,36.172
1,Weather imputation audit,Cell-level source-completion provenance; not predictors,pipeline_data/4.2.3.final_tables/weather_imputation_audit.parquet,19500,10,0.086
2,Weather feature catalog,Eight production weather predictor definitions,pipeline_data/4.2.3.final_tables/forecast_weather_feature_catalog.parquet,8,17,0.015
3,Complete forecasting feature catalog,Authoritative 86-definition cross-family predictor contract,pipeline_data/4.2.3.final_tables/forecast_feature_catalog_complete.parquet,86,20,0.020
4,Historical forecasting matrix · Post-CP Primary,Ordinary Track × Forecast Sequence × Origin × Horizon modeling matrix,pipeline_data/4.2.3.final_tables/forecasting_matrix__post_cp_primary.parquet,7486756,120,"1,053.658"
5,Historical forecasting matrix · Full-History Regime-Aware,Ordinary Track × Forecast Sequence × Origin × Horizon modeling matrix,pipeline_data/4.2.3.final_tables/forecasting_matrix__full_history_regime_aware.parquet,18688882,120,"2,536.633"
6,Historical forecasting matrix · Pre-CP Reference,Ordinary Track × Forecast Sequence × Origin × Horizon modeling matrix,pipeline_data/4.2.3.final_tables/forecasting_matrix__pre_cp_reference.parquet,11384323,120,"1,520.317"
7,Pre-CP final-refit training supplement,"Supervised Pre-CP training rows for January 1–4, 2025",pipeline_data/4.2.3.final_tables/pre_cp_final_refit_training_supplement.parquet,62340,120,7.215
8,Frozen Pre-CP final runtime contract,Post-freeze action for every shared feature definition,pipeline_data/4.2.3.final_tables/pre_cp_final_runtime_contract.parquet,86,24,0.023
9,Pre-CP counterfactual application template,Runtime-aware post-freeze no-CP application handoff,pipeline_data/4.2.3.final_tables/pre_cp_counterfactual_application_template.parquet,7023640,109,173.280


**Findings.** The final file inventory confirms that all major 4.2.3 outputs were created successfully. This includes the completed weather source and its 19,500-cell imputation audit, the 8-feature weather catalog, the full 86-feature forecasting catalog, the three large historical forecasting datasets, the **62,340-row Pre-CP final-refit supplement**, the frozen Pre-CP feature rules, the **7,023,640-row no-CP application template**, and the final **87-check** QA file.

The historical modeling matrices, final-refit supplement, and later no-CP application dataset remain clearly separated, giving downstream forecasting notebooks explicit inputs for model selection and evaluation, final production refitting, and recursive counterfactual prediction.

### Section Summary

Section 4.2.3.8 closes the leakage-safe forecasting feature build with a single cross-family audit and production manifest.

The ordinary historical handoff consists of three native modeling-track matrices at **Modeling Track × Forecast Sequence × Origin × Horizon** grain. Each contains the complete shared predictor schema, the correct target-specific context exclusions, exact h=1/h=2/h=5 timing, and the future observed mobility value as a separate label. For the Pre-CP Reference, a separate final-refit supplement carries the same physical modeling contract across the additional observed targets from **January 1–4, 2025**. The frozen Pre-CP application remains a distinct runtime-aware artifact rather than an ordinary post-CP matrix.

The consolidated QA carries forward the completed 4.2.1 and 4.2.2 contracts, the causal weather-completion and timing checks, the historical matrix audits, the final-refit training boundary, and the post-freeze counterfactual boundary. Together they verify that native Forecast Sequence populations are preserved, target arithmetic remains exact, the 1,029 cross-regime population has not been imposed on production modeling, target outcomes never enter the predictor catalog, and actual post-CP mobility cannot populate the no-CP recursive state.

The accompanying manifest provides the authoritative downstream entry points for ordinary model training and evaluation, the selected models' final Pre-CP refit, and the separate no-CP application path.

## Close

Notebook 4.2.3 completes the forecasting feature-engineering handoff.

The forecasting system now has one coherent feature architecture spanning own-target history and seasonal state, multimodal and transportation-connected context, static geography, and compact exogenous weather. Those definitions are preserved in an 86-row feature catalog together with their ordinary availability, frozen no-CP behavior, and structural relevance to the later ARIMA/SARIMAX, VARIMA, panel-tree, and neural-sequence forecasting approaches.

For ordinary historical evaluation, the three native modeling tracks are materialized separately and retain the exact rolling-origin experiment defined upstream. Each forecast target excludes its two redundant local 4.2.2 context fields, while the actual future mobility observation remains a label rather than a predictor. No global scaling, outcome-informed feature selection, or model fitting has been performed.

The Pre-CP Reference also carries a separate final-refit training supplement for **January 1–4, 2025**. It follows the same historical forecasting-matrix contract and supplies the final observed Pre-CP targets needed after model selection is complete.

The frozen Pre-CP Reference is packaged differently for post-freeze application. Its post-January 4 application template contains deterministic, frozen, static, and exogenous information that can legally be known in advance, but leaves mobility-derived recursive state unresolved after the legal seed. The later no-CP forecast loop must generate those values from its own prior predictions rather than borrowing actual post-CP transportation observations.

With that contract fixed, **4.2.4 can begin historical baseline benchmarking**. The next notebook should evaluate simple leakage-safe reference forecasts such as previous-step persistence, previous comparable Daypart, previous-day same-Daypart, and shifted trailing historical means across h=1, h=2, and h=5 before the more complex statistical, panel-tree, and neural model families are introduced.

### 4.2.3 Final Inventory of Files

This notebook writes the following downstream handoff assets to `pipeline_data/4.2.3.final_tables/`. Reusable intermediate/cache artifacts are not included in this final inventory.

The causally completed common weather source and the normalized Track × Taxi Zone × Origin × Horizon weather-alignment layer are retained under `pipeline_data/4.2.3.intermediate_tables/` for reuse, but they are not production handoff files listed below.

- **`weather_imputation_audit.parquet`**  

  Cell-level audit of the weather values completed in 4.2.3. Records which retained weather measure was missing, where the missing value occurred, the completed value, and the strictly-prior climatology rule used to fill it. These fields are provenance and QA information, not forecasting predictors.

- **`forecast_weather_feature_catalog.parquet`**  

  Feature-level catalog for the **8 production weather predictors**. Records the source weather measure, whether the predictor represents origin-observed weather or target-period proxy weather, ordinary forecasting availability, frozen Pre-CP availability, source-completion treatment, and the caveat attached to realized target-period weather.

- **`forecast_feature_catalog_complete.parquet`**  

  Authoritative cross-family catalog containing all **86 shared forecasting feature definitions**: 37 from 4.2.1, 41 from 4.2.2, and 8 from 4.2.3. Preserves feature family, timing role, ordinary availability, frozen Pre-CP treatment, target-specific exclusion rules, and downstream modeling-role metadata.

- **`forecasting_matrix__post_cp_primary.parquet`**  

  Historical forecasting matrix for the **Post-CP Primary** track at **Modeling Track × Forecast Sequence × Origin × Horizon** grain. Contains **7,486,756 rows × 120 physical columns**, preserves all **1,108 native Forecast Sequences**, and supports h=1, h=2, and h=5 forecasting.

- **`forecasting_matrix__full_history_regime_aware.parquet`**  

  Historical forecasting matrix for the **Full-History Regime-Aware** track at the same final forecasting grain. Contains **18,688,882 rows × 120 physical columns**, preserves all **1,051 native Forecast Sequences**, and spans the full historical forecasting period while retaining policy-regime information.

- **`forecasting_matrix__pre_cp_reference.parquet`**  

  Historical observed-data forecasting matrix for the **Pre-CP Reference** track. Contains **11,384,323 rows × 120 physical columns** and preserves all **1,039 native Forecast Sequences** across the 2023 initial-training and 2024 model-selection experiment.

- **`pre_cp_final_refit_training_supplement.parquet`**  

  Final-refit-only supervised continuation of the Pre-CP Reference training surface. Contains **62,340 rows × 120 physical columns** across all **1,039 native Forecast Sequences**, h=1/h=2/h=5, and target positions from **January 1–4, 2025**. It uses the same physical matrix contract and is consumed only for the selected models' final production refit.

- **`pre_cp_final_runtime_contract.parquet`**  

  Feature-level runtime contract covering all **86 forecasting feature definitions** after the Pre-CP Reference is frozen. Identifies which predictors can still be known directly, which must be reconstructed from recursive no-CP predictions, which auxiliary mobility fields become unavailable, and which fields are deliberately not used.

- **`pre_cp_counterfactual_application_template.parquet`**  

  Post-freeze no-congestion-pricing application template for the frozen **Pre-CP Reference**. Contains **7,023,640 rows × 109 physical columns** across all 1,039 native Pre-CP Forecast Sequences and h=1, h=2, and h=5. Known calendar, frozen seasonal, static, and weather information is materialized, while mobility-derived recursive state remains unresolved for later forecast generation. No future observed mobility target is attached.

- **`final_forecasting_matrix_qa.parquet`**  

  Consolidated structural, timing, availability, population, weather, and leakage-safety QA for the completed 4.2.3 package. Contains **87 checks, all passing**.

- **`forecasting_matrix_manifest.parquet`**  

  File-level manifest for the completed 4.2.3 forecasting handoff. Records the production and supporting artifacts, their paths, roles, row counts, column counts, and file sizes so downstream forecasting notebooks have one authoritative inventory.


### 4.2.3 Feature Columns

The completed 4.2.3 feature contract carries **86 shared candidate predictor definitions**:

**37 predictors from 4.2.1 + 41 predictors from 4.2.2 + 8 predictors from 4.2.3 = 86 shared feature definitions.**

The physical historical forecasting matrices contain **120 columns**, but those files also include identifiers, origin/target timing fields, stage and eligibility metadata, provenance fields, QA fields, and the future observed mobility label. Therefore, 120 physical matrix columns should **not** be interpreted as 120 forecasting predictors.


#### Inherited 4.2.1 Own-Target Historical State — 17 Columns

These features describe the forecast target's own mobility history available at the forecast origin.

- `origin_value`
- `seq_lag_1`
- `seq_lag_2`
- `seq_lag_3`
- `seq_lag_4`
- `seq_delta_1`
- `same_daypart_lag_1d`
- `same_daypart_lag_2d`
- `same_daypart_lag_7d`
- `same_daypart_delta_1d`
- `same_daypart_roll_mean_7d`
- `same_daypart_roll_std_7d`
- `same_daypart_roll_obs_share_7d`
- `same_daypart_ewm_mean_7d`
- `same_daypart_roll_mean_28d`
- `same_daypart_roll_std_28d`
- `same_daypart_roll_obs_share_28d`


#### Inherited 4.2.1 Origin Calendar Context — 7 Columns

These fields describe when the forecast is being made.

- `origin_daypart`
- `origin_temporal_bucket`
- `origin_weekday_index`
- `origin_is_weekend`
- `origin_month`
- `origin_day_of_year`
- `origin_elapsed_days`


#### Inherited 4.2.1 Target Calendar Context — 7 Columns

These fields describe the future time period being predicted.

- `horizon`
- `target_daypart`
- `target_temporal_bucket`
- `target_weekday_index`
- `target_is_weekend`
- `target_month`
- `target_day_of_year`


#### Inherited 4.2.1 Annual Seasonal Context — 2 Columns

- `annual_fourier_sin`
- `annual_fourier_cos`


#### Inherited 4.2.1 Weekly Seasonal State — 2 Columns

- `weekly_stl_enabled`
- `weekly_stl_seasonal_offset`

For an enabled weekly-STL series, the seasonal offset may legitimately be unavailable in the earliest historical rows before that modeling track has reached its first legal fitted weekly-seasonal state. Those rows are retained rather than filled using a seasonal fit learned later.


#### Inherited 4.2.1 Policy-Regime Context — 2 Columns

- `origin_pre_post_cp`
- `target_pre_post_cp`

These fields support the Full-History Regime-Aware forecasting track. They are deliberately not used when the frozen Pre-CP Reference is advanced into the hypothetical no-CP post-policy period.


**4.2.1 total: 37 shared feature definitions.**


#### Inherited 4.2.2 Multimodal Origin Context — 20 Columns

Each of the 10 retained transportation measures contributes its value at the forecast origin and its change from the immediately preceding canonical Daypart.

- `mm_taxi_trip_count_origin_value`
- `mm_taxi_trip_count_seq_delta_1`
- `mm_taxi_avg_trip_speed_origin_value`
- `mm_taxi_avg_trip_speed_seq_delta_1`
- `mm_fhvhv_trip_count_origin_value`
- `mm_fhvhv_trip_count_seq_delta_1`
- `mm_fhvhv_avg_trip_speed_origin_value`
- `mm_fhvhv_avg_trip_speed_seq_delta_1`
- `mm_subway_ridership_origin_value`
- `mm_subway_ridership_seq_delta_1`
- `mm_taxi_avg_trip_duration_origin_value`
- `mm_taxi_avg_trip_duration_seq_delta_1`
- `mm_fhvhv_avg_trip_duration_origin_value`
- `mm_fhvhv_avg_trip_duration_seq_delta_1`
- `mm_subway_transfers_origin_value`
- `mm_subway_transfers_seq_delta_1`
- `mm_bus_trip_count_origin_value`
- `mm_bus_trip_count_seq_delta_1`
- `mm_avg_bus_speed_origin_value`
- `mm_avg_bus_speed_seq_delta_1`


#### Inherited 4.2.2 Transportation-Connected Spatial Context — 10 Columns

Each retained transportation measure also contributes the same-origin state of the Taxi Zone's transportation-connected geographic environment.

- `spatial_connected__taxi_trip_count`
- `spatial_connected__taxi_avg_trip_speed`
- `spatial_connected__fhvhv_trip_count`
- `spatial_connected__fhvhv_avg_trip_speed`
- `spatial_connected__subway_ridership`
- `spatial_connected__taxi_avg_trip_duration`
- `spatial_connected__fhvhv_avg_trip_duration`
- `spatial_connected__subway_transfers`
- `spatial_connected__bus_trip_count`
- `spatial_connected__avg_bus_speed`


#### Inherited 4.2.2 Static Geographic and Policy Context — 11 Columns

These predictors were engineered upstream and are carried into the final forecasting matrices as stable Taxi Zone context.

- `borough`
- `is_cbd_zone`
- `is_cbd_gateway_zone`
- `is_cbd_adjacent_zone`
- `cbd_spatial_category`
- `distance_to_cbd_miles`
- `distance_to_gateway_miles`
- `connected_neighbor_count`
- `network_betweenness_centrality`
- `network_closeness_centrality`
- `network_clustering_coefficient`


**4.2.2 total: 41 shared feature definitions.**


#### Newly Finalized Weather Context — 8 Columns

The four retained weather measures each contribute two timing representations.


##### Origin-Observed Weather — 4 Columns

These describe weather already observed at the forecast origin.

- `weather_origin__temperature`
- `weather_origin__precipitation`
- `weather_origin__visibility`
- `weather_origin__wind_speed`


##### Realized Target-Period Weather Proxies — 4 Columns

These describe weather at the exact future period being predicted.

- `weather_target_proxy__temperature`
- `weather_target_proxy__precipitation`
- `weather_target_proxy__visibility`
- `weather_target_proxy__wind_speed`

The target-period weather fields are explicit proxies for archived operational weather forecasts that were not available for this project. Because they use the weather that actually occurred at the target period, models using them may receive somewhat better weather information in historical evaluation than would have been available in a real-time forecasting setting.

Weather remains external to congestion pricing, so all eight weather predictors may also be used by the frozen no-CP counterfactual without introducing actual post-CP mobility.


**4.2.3 total: 8 shared weather feature definitions.**


#### Target-Specific Predictor Count

The common feature catalog carries all **86 definitions** so every modeling track and target uses one shared schema.

For each of the five forecast targets, however, the target's own local 4.2.2 multimodal origin value and one-step change are redundant with the richer own-target history already supplied by 4.2.1. Those two fields are therefore structurally excluded for that target.

Therefore:

**86 shared feature definitions − 2 redundant same-target multimodal definitions = 84 structurally eligible candidate predictors per forecast target before model-specific feature selection.**

For example, when forecasting `taxi_trip_count`, the following two 4.2.2 fields are excluded:

- `mm_taxi_trip_count_origin_value`
- `mm_taxi_trip_count_seq_delta_1`

The same rule is applied to the corresponding pair for each of the other four targets.


#### Frozen Pre-CP Runtime Treatment

After the Pre-CP Reference is frozen, the 86 shared feature definitions divide into the following runtime classes:

- **32 `recursive_required`** — mobility-derived predictors that must be rebuilt from the synthetic no-CP mobility history.
- **15 `ordinary_only`** — auxiliary dynamic mobility predictors that are available for ordinary historical forecasting but are not available after the counterfactual freeze unless separately forecast.
- **16 `deterministic_future_known`** — calendar and timing information that remains directly known.
- **2 `frozen_training_fitted`** — fitted weekly-seasonal state carried forward from the legal Pre-CP fit.
- **11 `static_available`** — geographic and network context that does not change with the forecast origin.
- **8 `exogenous_available`** — weather predictors that remain available because weather is external to the congestion-pricing treatment.
- **2 `not_used`** — policy-regime fields intentionally excluded from the synthetic no-CP path.

Five of the recursive-required context definitions depend on mobility state from multiple Taxi Zones and therefore require the later recursive forecast loop to maintain a synchronized cross-zone synthetic state.


#### Non-Predictor and Missingness Boundaries

The following types of fields may appear in the physical forecasting matrices but are **not counted among the 86 forecasting feature definitions**:

- Forecast Sequence, Taxi Zone, origin, and target identifiers
- Dates and other duplicated timing/provenance fields used for joins and QA
- Modeling-track and stage metadata
- Native-track and comparison-cohort eligibility flags
- Weekly-STL fit/interval provenance fields
- Weather imputation flags and audit fields
- Same-target exclusion counts
- Connected-zone support and other QA/provenance fields
- `target_label_available`
- `target_observed_value`

`target_observed_value` is the future mobility outcome the models will attempt to predict. It is deliberately kept separate from all predictor definitions.

The historical forecasting matrices also preserve legitimate missing predictor values rather than globally filling them in 4.2.3. Examples include early lagged-history fields, weekly seasonal offsets before the first legal fitted state, and performance-based transportation context where the underlying observation is unavailable.

Any model-specific numerical imputation, categorical encoding, scaling, or other preprocessing must therefore be fitted only on the appropriate legal training window in the downstream modeling notebooks.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>